# IVF Digital Twin — Аналитика прогнозов vs реальные данные

**Цель:** Систематическое сравнение предсказаний Digital Twin с реальными данными цикла ЭКО.

### Что анализируем:
| Блок | Описание |
|---|---|
| **1. Загрузка и мёрж** | Объединение CSV-выгрузки DT и Excel-таблицы по `patient_id` = `ID` |
| **2. Входные параметры** | Возраст, AFC, фолликулы ТВП — что подавали в модель |
| **3. Воронка: прогноз vs реальность** | MC-медианы (OCC, MII, бласты, хор.кач.) против фактических |
| **4. Предсказания беременности** | MC pipeline · KAT · NVSA · CSDI (DT) · PRAI — сравнение всех моделей |
| **5. Калибровка и метрики качества** | Brier Score, AUC-ROC, calibration curve (по пациентам с исходом) |
| **6. Согласованность моделей** | Корреляции между слоями пайплайна + KAT vs PRAI |
| **7. Кластерный анализ** | Predicted cluster vs реальные исходы |
| **8. Риски и дополнительные метрики** | OHSS-риск, p_cancel, banking |
| **9. Итоговый дашборд** | Сводная таблица с флагами для выбросов |

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.metrics import (
        roc_auc_score, brier_score_loss, roc_curve,
        confusion_matrix, classification_report
    )
from sklearn.calibration import calibration_curve
from scipy import stats  

# ── Единая палитра ────────────────────────────────────────────
C = {
    'blue':   '#1B4F72',
    'red':    '#C0392B',
    'green':  '#1E8449',
    'orange': '#D68910',
    'purple': '#7D3C98',
    'gray':   '#717D7E',
    'teal':   '#148F77',
}
LAYOUT = dict(
    font=dict(family='Inter, Arial', size=12, color='#2c3e50'),
    paper_bgcolor='white',
    plot_bgcolor='#F7F9FC',
    margin=dict(t=50, b=40, l=50, r=30),
    hoverlabel=dict(bgcolor='white', font_size=12),
)
print('✅ Импорты OK')
# ── Конвертер цвета hex → rgba (для Plotly fillcolor) ────────
def hex_rgba(hex_color: str, alpha: float = 1.0) -> str:
    h = hex_color.lstrip('#')
    if len(h) == 6:
        r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
        return f'rgba({r},{g},{b},{alpha})'
    return hex_color  # fallback: вернуть как есть

✅ Импорты OK


## 1. Загрузка данных и объединение

In [2]:
PATH_CSV   = 'dt_predictions.csv'
PATH_EXCEL = 'OPU table.xlsx'

dt  = pd.read_csv(PATH_CSV)
try:
    opu = pd.read_excel(PATH_EXCEL, engine='calamine')
except Exception:
    opu = pd.read_excel(PATH_EXCEL, engine='openpyxl')

dt['_key']  = dt['patient_id'].astype(str).str.strip().str.upper()
opu['_key'] = opu['ID'].astype(str).str.strip().str.upper()

df = pd.merge(dt, opu, on='_key', how='inner', suffixes=('_dt', '_opu'))

# Дедуплицированная версия (последний расчёт на пациента)
df_u = (df.sort_values('timestamp', ascending=True)
          .drop_duplicates(subset=['_key'], keep='last')
          .copy()) if 'timestamp' in df.columns else df.copy()

df_preg = df_u[df_u['Preg'].notna()].copy()

print(f'DT записей:              {len(dt)}')
print(f'OPU строк:               {len(opu)}')
print(f'Совпало по ID (все):     {len(df)}')
print(f'Уникальных пациентов:    {len(df_u)}')
print(f'С исходом Preg:          {len(df_preg)}  '
      f'(+{int(df_preg["Preg"].sum())} / -{int((df_preg["Preg"]==0).sum())})')

DT записей:              55
OPU строк:               59
Совпало по ID (все):     60
Уникальных пациентов:    51
С исходом Preg:          12  (+6 / -6)


In [3]:
# ── Предварительный просмотр объединённой таблицы ─────────────
view_cols = [
    'patient_name', '_key',
    'age_dt', 'Age',          # возраст из обоих источников
    'afc', 'Antral follicule count',
    'p_per_transfer', 'p_kat_raw', 'p_nvsa', 'p_csdi',
    'PRAI', 'DIGITAL TWIN',
    'Preg', 'dominant_cluster'
]
# Оставляем только существующие колонки
view_cols = [c for c in view_cols if c in df.columns]
df[view_cols].head(10)

,patient_name,_key,Age,afc,Antral follicule count,p_per_transfer,p_kat_raw,p_nvsa,p_csdi,PRAI,DIGITAL TWIN,Preg,dominant_cluster
0,NAZARALIYEVA RAYXONA,AD3595191,40,5,NaN,0.3212,0.1452,0.1558,0.2693,0.128,0.026,NaN,1
1,NAZARALIYEVA RAYXONA,AD3595191,40,5,NaN,0.3212,0.1452,0.1558,0.2693,0.126,0.269,NaN,1
2,KARIMJANOVA SHOKHISTAKHON,AC2986596,30,12,NaN,0.6816,0.4268,0.5861,0.4957,0.235,0.496,NaN,1
3,KUCHKOROVA MAFTUNAKHON,AB7871088,25,60,NaN,0.7626,0.7337,0.6821,0.5236,0.517,0.524,NaN,2
4,UMIRZAKOVA MALIKA,AD7420759,27,6,NaN,0.5717,0.1991,0.2701,0.3313,NaN,0.331,NaN,1
5,UMIRZAKOVA MALIKA,AD7420759,27,6,NaN,0.5717,0.1991,0.2701,0.3313,0.512,0.645,NaN,1
6,TURGUNOVA MATLUBA,AD5571994,37,22,NaN,0.5781,0.3648,0.5143,0.4823,0.235,0.482,NaN,0
7,HOMIDOVA KURMATOY,404378919,26,16,NaN,0.7350,0.5933,0.6371,0.5438,0.509,0.544,NaN,0
8,ODILOVA MUXLISA,AD0497909,27,16,NaN,0.6700,0.4765,0.5095,0.3580,0.284,0.358,NaN,1
9,TURGUNBOEVA NILUFAR,AD1118005,25,60,NaN,0.7563,0.7257,0.6680,0.4841,0.342,0.484,NaN,2


## 2. Входные параметры: что подавали в модель

In [4]:
fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Возраст пациенток', 'Фолликулов ТВП', 'MII ооцитов'])

colors_half = {
    'Age': 'rgba(41, 128, 185, 0.65)',
    'N folicules OPU': 'rgba(39, 174, 96, 0.65)',
    'MII': 'rgba(243, 156, 18, 0.65)'
}
line_color = 'rgba(255,255,255,0.8)'

for i, (col, base_color) in enumerate(
    [('Age', colors_half['Age']),
     ('N folicules OPU', colors_half['N folicules OPU']),
     ('MII', colors_half['MII'])], 1
):
    if col not in df_u.columns:
        continue
    vals = df_u[col].dropna()
    mean_val = vals.mean()
    
    # Гистограмма
    fig.add_trace(go.Histogram(
        x=vals, name=col, nbinsx=15,
        marker=dict(color=base_color, line=dict(color=line_color, width=0.5)),
        opacity=0.7,
        hovertemplate='%{x}: %{y} пациентов<extra></extra>',
    ), row=1, col=i)
    
    # KDE
    kde_x = np.linspace(vals.min(), vals.max(), 200)
    kde = stats.gaussian_kde(vals)
    kde_y = kde(kde_x)
    hist_counts, _ = np.histogram(vals, bins=15)
    scaling_factor = hist_counts.max() / kde_y.max() if kde_y.max() > 0 else 1
    fig.add_trace(go.Scatter(
        x=kde_x, y=kde_y * scaling_factor,
        mode='lines', name=f'плотность ({col})',
        line=dict(color='black', width=1.5, dash='dot'),
        showlegend=False, hoverinfo='skip'
    ), row=1, col=i)
    
    # Линия среднего
    fig.add_vline(x=mean_val, line_dash='dash', line_color='black', line_width=1.2, row=1, col=i)
    
    # Аннотация среднего
    yref = 'y domain' if i == 1 else f'y{i} domain'
    fig.add_annotation(
        x=mean_val, y=1.02, yref=yref,
        text=f'μ = {mean_val:.1f}',
        showarrow=False, font=dict(size=10, color='black'),
        xanchor='center', row=1, col=i
    )

# Настройки layout без конфликта ключей
layout_demography = {**LAYOUT,
                     'height': 380,
                     'title': 'Демография когорты',
                     'showlegend': False,
                     'bargap': 0.05,
                     'plot_bgcolor': 'rgba(245,245,245,0.9)',
                     'paper_bgcolor': 'white'
                    }
fig.update_layout(**layout_demography)

fig.update_xaxes(showgrid=True, gridcolor='rgba(0,0,0,0.08)', showline=True, linecolor='rgba(0,0,0,0.2)')
fig.update_yaxes(showgrid=True, gridcolor='rgba(0,0,0,0.08)', showline=True, linecolor='rgba(0,0,0,0.2)', title_text="Кол-во пациентов")
fig.show()

## 3. Воронка: MC-медианы vs реальность

In [5]:
# ══════════════════════════════════════════════════════════════
# ЯЧЕЙКА A — Ооциты: ОКК и MII  (полупрозрачные + SD)
# ══════════════════════════════════════════════════════════════
from scipy.stats import pearsonr, spearmanr

# ── Маппинг: (col_pred, col_real, col_p025, col_p975, label) ─
# p025/p975 из MC-симуляции → SD ≈ (p975-p025)/(2*1.96)
# Для MII прямых CI нет → оцениваем через IQR MC (≈0.67*SD)
pairs_ooc = [
    ('med_okk', 'OCC', 'p025_okk',    'p975_okk',    'ОКК (ооциты)', C['blue']),
    ('med_mii', 'MII', None,           None,           'MII (зрелые)', C['teal']),
]
pairs_ooc = [(a, b, lo, hi, t, col) for a, b, lo, hi, t, col in pairs_ooc
             if a in df_u.columns and b in df_u.columns]

fig_ooc = make_subplots(
    rows=1, cols=len(pairs_ooc),
    subplot_titles=[t for *_, t, _ in pairs_ooc],
    horizontal_spacing=0.14,
)

metrics_ooc = []
for i, (col_dt, col_opu, col_lo, col_hi, label, color) in enumerate(pairs_ooc, 1):
    sub = df_u[[col_dt, col_opu] +
               ([col_lo, col_hi] if col_lo else [])].copy()
    name_col = 'patient_name' if 'patient_name' in df_u.columns else '_key'
    sub[name_col] = df_u[name_col]
    sub = sub.dropna(subset=[col_dt, col_opu]).astype(
        {col_dt: float, col_opu: float})
    if len(sub) < 2:
        continue

    x, y   = sub[col_opu], sub[col_dt]
    mae    = (y - x).abs().mean()
    bias   = (y - x).mean()
    r, _   = pearsonr(x, y)
    rho, _ = spearmanr(x, y)
    lim    = [0, max(x.max(), y.max()) * 1.14]

    # ── SD: из p025/p975 или из остаточной дисперсии ──────────
    if col_lo and col_lo in sub.columns:
        sub[col_lo] = sub[col_lo].astype(float)
        sub[col_hi] = sub[col_hi].astype(float)
        sd_y = ((sub[col_hi] - sub[col_lo]) / (2 * 1.96)).clip(lower=0)
    else:
        # Для MII: SD ≈ std остатков (MC медиана - реал)
        resid = (y - x).abs()
        sd_y  = pd.Series([resid.mean()] * len(sub), index=sub.index)

    # ── Шейдированная полоса вокруг диагонали (±mean_SD) ──────
    x_arr   = np.linspace(lim[0], lim[1], 100)
    mean_sd = sd_y.mean()
    rgba_fill = hex_rgba(color, 0.10)
    rgba_line = hex_rgba(color, 0.35)
    for mult, y_arr in [(1, x_arr + mean_sd), (-1, x_arr - mean_sd)]:
        fig_ooc.add_trace(go.Scatter(
            x=x_arr, y=np.clip(y_arr, 0, None),
            mode='lines',
            line=dict(color=rgba_line, width=0.8, dash='dot'),
            fill='tonexty' if mult == -1 else None,
            fillcolor=rgba_fill,
            showlegend=False, hoverinfo='skip',
        ), row=1, col=i)

    # ── Диагональ идеального совпадения ───────────────────────
    fig_ooc.add_trace(go.Scatter(
        x=lim, y=lim, mode='lines',
        line=dict(color='rgba(150,150,150,0.7)', dash='dash', width=1.5),
        showlegend=False, hoverinfo='skip',
    ), row=1, col=i)

    # ── Линия тренда ──────────────────────────────────────────
    if len(sub) >= 4:
        z = np.polyfit(x, y, 1)
        fig_ooc.add_trace(go.Scatter(
            x=x_arr, y=np.poly1d(z)(x_arr), mode='lines',
            line=dict(color=hex_rgba(color, 0.7), width=2, dash='dot'),
            showlegend=False, hoverinfo='skip',
        ), row=1, col=i)

    # ── Scatter с error bars (SD) и полупрозрачностью ─────────
    fig_ooc.add_trace(go.Scatter(
        x=x, y=y,
        mode='markers',
        text=sub[name_col],
        error_y=dict(
            type='data',
            array=sd_y.values,
            color=hex_rgba(color, 0.45),
            thickness=1.5,
            width=4,
        ),
        hovertemplate=(
            '<b>%{text}</b><br>'
            'Реально: %{x:.0f}<br>'
            'MC-медиана: %{y:.0f}<br>'
            'SD: ±%{error_y.array:.1f}<extra></extra>'
        ),
        marker=dict(
            size=11,
            color=hex_rgba(color, 0.70),
            line=dict(width=1.5, color=hex_rgba(color, 0.95)),
        ),
        showlegend=False,
    ), row=1, col=i)

    fig_ooc.update_xaxes(title_text='Реально', range=lim, row=1, col=i,
                         zeroline=False)
    fig_ooc.update_yaxes(title_text='MC-медиана', range=lim, row=1, col=i,
                         zeroline=False)
    fig_ooc.add_annotation(
        text=(
            f'r = {r:.2f} | ρ = {rho:.2f}<br>'
            f'MAE = {mae:.1f} | Bias = {bias:+.1f}<br>'
            f'Среднее SD = {mean_sd:.1f}'
        ),
        xref=f'x{i if i > 1 else ""}',
        yref=f'y{i if i > 1 else ""}',
        x=lim[0] + (lim[1] - lim[0]) * 0.04,
        y=lim[1] * 0.95,
        showarrow=False,
        font=dict(size=10, color='#333'),
        align='left',
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='#ccc', borderwidth=1, borderpad=5,
    )
    metrics_ooc.append({
        'Стадия': label, 'N': len(sub),
        'MAE': round(mae, 2), 'Bias': round(bias, 2),
        'Pearson r': round(r, 3), 'Spearman ρ': round(rho, 3),
        'Среднее SD': round(mean_sd, 2),
    })

fig_ooc.update_layout(
    **LAYOUT, height=500,
    title=dict(text='Воронка — Ооциты: MC-прогноз vs реальность (±SD)',
               font=dict(size=15)),
)
fig_ooc.show()
pd.DataFrame(metrics_ooc)


,Стадия,N,MAE,Bias,Pearson r,Spearman ρ,Среднее SD
0,ОКК (ооциты),51,0.51,0.39,0.942,0.917,0.00
1,MII (зрелые),51,0.37,0.33,0.955,0.931,0.37


In [6]:

# ══════════════════════════════════════════════════════════════
# ЯЧЕЙКА B — Эмбриология: 2PN, бластоцисты, хор.кач. (+ SD)
# ══════════════════════════════════════════════════════════════

pairs_emb = [
    ('med_pn2',    '2pN',     None,          None,           '2PN (зиготы)',         C['purple']),
    ('med_blasts', 'Bl',      'p025_blasts', 'p975_blasts',  'Бластоцисты всего',    C['orange']),
    ('med_good',   'Good Bl', 'p025_good',   'p975_good',    'Бластоцисты хор.кач.', C['red']),
]
pairs_emb = [(a, b, lo, hi, t, col) for a, b, lo, hi, t, col in pairs_emb
             if a in df_u.columns and b in df_u.columns]

fig_emb = make_subplots(
    rows=1, cols=len(pairs_emb),
    subplot_titles=[t for *_, t, _ in pairs_emb],
    horizontal_spacing=0.10,
)

metrics_emb = []
for i, (col_dt, col_opu, col_lo, col_hi, label, color) in enumerate(pairs_emb, 1):
    sub = df_u[[col_dt, col_opu] +
               ([col_lo, col_hi] if col_lo else [])].copy()
    name_col = 'patient_name' if 'patient_name' in df_u.columns else '_key'
    sub[name_col] = df_u[name_col]
    sub = sub.dropna(subset=[col_dt, col_opu]).astype(
        {col_dt: float, col_opu: float})
    if len(sub) < 2:
        continue

    x, y   = sub[col_opu], sub[col_dt]
    mae    = (y - x).abs().mean()
    bias   = (y - x).mean()
    r, _   = pearsonr(x, y)
    rho, _ = spearmanr(x, y)
    lim    = [0, max(x.max(), y.max()) * 1.14]

    # ── SD ────────────────────────────────────────────────────
    if col_lo and col_lo in sub.columns:
        sub[col_lo] = sub[col_lo].astype(float)
        sub[col_hi] = sub[col_hi].astype(float)
        sd_y = ((sub[col_hi] - sub[col_lo]) / (2 * 1.96)).clip(lower=0)
    else:
        resid = (y - x).abs()
        sd_y  = pd.Series([resid.mean()] * len(sub), index=sub.index)

    mean_sd  = sd_y.mean()
    x_arr    = np.linspace(lim[0], lim[1], 100)
    rgba_fill = hex_rgba(color, 0.08)
    rgba_line = hex_rgba(color, 0.30)

    # ── ±30% коридор (полупрозрачный) ─────────────────────────
    for y_bnd in [x_arr * 1.30, x_arr * 0.70]:
        fig_emb.add_trace(go.Scatter(
            x=x_arr, y=np.clip(y_bnd, 0, None),
            mode='lines',
            line=dict(color='rgba(180,180,180,0.5)', width=0.8, dash='dot'),
            showlegend=False, hoverinfo='skip',
        ), row=1, col=i)

    # ── ±mean_SD шейдированная полоса ─────────────────────────
    fig_emb.add_trace(go.Scatter(
        x=x_arr, y=np.clip(x_arr + mean_sd, 0, None),
        mode='lines',
        line=dict(color=rgba_line, width=0),
        showlegend=False, hoverinfo='skip',
    ), row=1, col=i)
    fig_emb.add_trace(go.Scatter(
        x=x_arr, y=np.clip(x_arr - mean_sd, 0, None),
        mode='lines',
        fill='tonexty',
        fillcolor=rgba_fill,
        line=dict(color=rgba_line, width=0),
        showlegend=False, hoverinfo='skip',
    ), row=1, col=i)

    # ── Диагональ ─────────────────────────────────────────────
    fig_emb.add_trace(go.Scatter(
        x=lim, y=lim, mode='lines',
        line=dict(color='rgba(120,120,120,0.6)', dash='dash', width=1.5),
        showlegend=False, hoverinfo='skip',
    ), row=1, col=i)

    # ── Линия тренда ──────────────────────────────────────────
    if len(sub) >= 4:
        z = np.polyfit(x, y, 1)
        fig_emb.add_trace(go.Scatter(
            x=x_arr, y=np.poly1d(z)(x_arr), mode='lines',
            line=dict(color=hex_rgba(color, 0.65), width=2.2, dash='dot'),
            showlegend=False, hoverinfo='skip',
        ), row=1, col=i)

    # ── Цвет точек по точности прогноза ───────────────────────
    dot_colors = []
    for _, row_ in sub.iterrows():
        real_v = float(row_[col_opu])
        pred_v = float(row_[col_dt])
        if real_v == 0:
            dot_colors.append('rgba(160,160,160,0.55)')
        elif abs(pred_v - real_v) / real_v <= 0.30:
            dot_colors.append(hex_rgba(C['green'], 0.75))
        else:
            dot_colors.append(hex_rgba(C['red'], 0.75))

    # ── Scatter с error bars ───────────────────────────────────
    fig_emb.add_trace(go.Scatter(
        x=x, y=y,
        mode='markers',
        text=sub[name_col],
        error_y=dict(
            type='data',
            array=sd_y.values,
            color=hex_rgba(color, 0.40),
            thickness=1.5,
            width=4,
        ),
        hovertemplate=(
            '<b>%{text}</b><br>'
            'Реально: %{x:.0f}<br>'
            'MC-медиана: %{y:.0f}<br>'
            'SD: ±%{error_y.array:.1f}<extra></extra>'
        ),
        marker=dict(
            size=11,
            color=dot_colors,
            line=dict(width=1.5, color='rgba(255,255,255,0.9)'),
        ),
        showlegend=False,
    ), row=1, col=i)

    fig_emb.update_xaxes(title_text='Реально', range=lim, row=1, col=i,
                         zeroline=False)
    fig_emb.update_yaxes(title_text='MC-медиана', range=lim, row=1, col=i,
                         zeroline=False)

    n_ok = sum(
        1 for _, r2 in sub.iterrows()
        if float(r2[col_opu]) > 0
        and abs(float(r2[col_dt]) - float(r2[col_opu])) / float(r2[col_opu]) <= 0.30
    )
    fig_emb.add_annotation(
        text=(
            f'r = {r:.2f} | ρ = {rho:.2f}<br>'
            f'MAE = {mae:.1f} | Bias = {bias:+.1f}<br>'
            f'Ср. SD = {mean_sd:.1f} | ✅ ±30%: {n_ok}/{len(sub)}'
        ),
        xref=f'x{i if i > 1 else ""}',
        yref=f'y{i if i > 1 else ""}',
        x=lim[0] + (lim[1] - lim[0]) * 0.04,
        y=lim[1] * 0.95,
        showarrow=False,
        font=dict(size=10, color='#333'),
        align='left',
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='#ccc', borderwidth=1, borderpad=5,
    )
    metrics_emb.append({
        'Стадия': label, 'N': len(sub),
        'MAE': round(mae, 2), 'Bias': round(bias, 2),
        'Pearson r': round(r, 3), 'Spearman ρ': round(rho, 3),
        'Среднее SD': round(mean_sd, 2),
        'В ±30%': f'{n_ok}/{len(sub)}',
    })

fig_emb.update_layout(
    **LAYOUT, height=500,
    title=dict(text='Воронка — Эмбриология: MC-прогноз vs реальность (±SD)',
               font=dict(size=15)),
)
fig_emb.show()

# ── Легенда цветов (ячейка B) ─────────────────────────────────
from IPython.display import display, HTML
display(HTML(
    '<div style="font-family:Inter,sans-serif;font-size:12px;'
    'padding:8px 12px;background:#f9f9f9;border-radius:6px;'
    'border-left:3px solid #ccc;display:inline-block">'
    '<span style="color:#1E8449;font-weight:bold">● Зелёный</span> — '
    'прогноз в пределах ±30%&nbsp;&nbsp;&nbsp;'
    '<span style="color:#C0392B;font-weight:bold">● Красный</span> — '
    'отклонение >30%&nbsp;&nbsp;&nbsp;'
    '<span style="color:#888;font-weight:bold">● Серый</span> — '
    'реальное = 0&nbsp;&nbsp;&nbsp;'
    '<span style="color:#555">Полоса = ±среднее SD симуляции&nbsp;&nbsp;'
    'Пунктир-серый = ±30% коридор</span>'
    '</div>'
))

pd.DataFrame(metrics_emb)

● Зелёный — прогноз в пределах ±30%    ● Красный — отклонение >30%    ● Серый — реальное = 0    Полоса = ±среднее SD симуляции  Пунктир-серый = ±30% коридор

,Стадия,N,MAE,Bias,Pearson r,Spearman ρ,Среднее SD,В ±30%
0,2PN (зиготы),49,1.69,-0.10,0.894,0.872,1.69,32/49
1,Бластоцисты всего,47,1.87,1.28,0.749,0.693,1.36,18/47
2,Бластоцисты хор.кач.,47,1.57,0.64,0.738,0.726,1.46,18/47


## 4. Предсказания беременности: все модели

In [7]:
model_cols = {
    'MC (p_per_transfer)': 'p_per_transfer',
    'Bayes':               'bayes_mean',
    'KAT':                 'p_kat_raw',
    'NVSA':                'p_nvsa',
    'CSDI':                'p_csdi',
    'DT (таблица)':        'DIGITAL TWIN',
    'PRAI':                'PRAI',
}
model_cols = {k: v for k, v in model_cols.items() if v in df_u.columns}

# ── Box + strip (jitter) ─────────────────────────────────────
fig = go.Figure()
palette = list(C.values())
for i, (name, col) in enumerate(model_cols.items()):
    vals = df_u[col].dropna()
    color = palette[i % len(palette)]
    fig.add_trace(go.Box(
        y=vals, name=name,
        boxmean='sd',
        marker_color=color,
        marker=dict(size=5, opacity=0.6),
        line_color=color,
        fillcolor=hex_rgba(color, 0.20),
        hovertemplate=f'<b>{name}</b><br>%{{y:.3f}}<extra></extra>',
    ))

fig.add_hline(y=0.5, line_dash='dot', line_color=C['gray'],
              annotation_text='P=50%', annotation_position='right')
fig.update_layout(**LAYOUT, height=420,
                  title='Распределение прогнозов беременности по всем моделям',
                  yaxis_title='P(беременность)', yaxis_range=[0, 1])
fig.show()

# Описательная статистика
pd.DataFrame({k: df_u[v].describe().round(3) for k,v in model_cols.items()}).T[    ['count','mean','std','min','25%','50%','75%','max']]

,count,mean,std,min,25%,50%,75%,max
MC (p_per_transfer),51.0,0.613,0.131,0.282,0.521,0.650,0.723,0.790
Bayes,51.0,0.434,0.037,0.357,0.409,0.440,0.467,0.475
KAT,51.0,0.455,0.158,0.098,0.369,0.491,0.565,0.734
NVSA,51.0,0.497,0.173,0.146,0.360,0.537,0.642,0.682
CSDI,51.0,0.406,0.093,0.224,0.336,0.404,0.487,0.560
DT (таблица),51.0,0.427,0.133,0.106,0.326,0.413,0.505,0.796
PRAI,50.0,0.356,0.169,0.080,0.230,0.342,0.499,0.700


## 5. KAT (симулированный) vs PRAI (реальный полный цикл)

In [8]:
# ══════════════════════════════════════════════════════════════
# KAT (симулированный) vs PRAI (реальный полный цикл)
# Полупрозрачные маркеры + CI bands + SD на гистограмме
# ══════════════════════════════════════════════════════════════

from scipy.stats import pearsonr, spearmanr, wilcoxon, norm as _norm

sub = df_u[[
    'p_kat_raw', 'PRAI',
    'ci_kat_low', 'ci_kat_high',   # 95% CI KAT из CSV
    '_key', 'Age', 'dominant_cluster',
] + (['patient_name'] if 'patient_name' in df_u.columns else [])
].dropna(subset=['p_kat_raw', 'PRAI']).copy()

if len(sub) < 2:
    print('Недостаточно данных')
else:
    name_col      = 'patient_name' if 'patient_name' in sub.columns else '_key'
    cluster_names = {0: 'C0 Standard', 1: 'C1 Poor', 2: 'C2 High'}
    cluster_color = {0: '#1976D2',     1: '#C62828',  2: '#2E7D32'}

    diff   = sub['p_kat_raw'] - sub['PRAI']
    r, _   = pearsonr(sub['PRAI'], sub['p_kat_raw'])
    rho, _ = spearmanr(sub['PRAI'], sub['p_kat_raw'])
    mae    = diff.abs().mean()
    bias   = diff.mean()
    sd_diff = diff.std()

    # ── SD KAT из CI: SD ≈ (ci_high − ci_low) / (2×1.96) ────
    has_ci = ('ci_kat_low' in sub.columns and
              sub['ci_kat_low'].notna().sum() > 0)
    if has_ci:
        sub['sd_kat'] = ((sub['ci_kat_high'] - sub['ci_kat_low'])
                         / (2 * 1.96)).clip(lower=0.005)
    else:
        sub['sd_kat'] = pd.Series(
            [sub['p_kat_raw'].std() * 0.5] * len(sub), index=sub.index)

    # ════════════════════════════════════════════════════════════
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[
            f'KAT vs PRAI  (r={r:.3f}, ρ={rho:.3f}, MAE={mae:.3f})',
            f'Распределение разниц KAT−PRAI  (bias={bias:+.3f})',
        ],
        horizontal_spacing=0.13,
    )

    # ── ЛЕВЫЙ ГРАФИК: scatter по кластерам + CI bands ─────────

    # 1. Диагональ идеального совпадения
    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1], mode='lines',
        line=dict(color='rgba(150,150,150,0.55)', dash='dash', width=1.5),
        showlegend=False, hoverinfo='skip',
    ), row=1, col=1)

    # 2. Шейдированная полоса ±mean(SD_KAT) вдоль диагонали
    x_arr    = np.linspace(0, 1, 120)
    mean_sd  = sub['sd_kat'].mean()
    fig.add_trace(go.Scatter(
        x=x_arr, y=np.clip(x_arr + mean_sd, 0, 1),
        mode='lines', line=dict(color='rgba(100,100,100,0)', width=0),
        showlegend=False, hoverinfo='skip',
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=x_arr, y=np.clip(x_arr - mean_sd, 0, 1),
        mode='lines',
        fill='tonexty', fillcolor='rgba(150,150,150,0.10)',
        line=dict(color='rgba(100,100,100,0.18)', dash='dot', width=0.8),
        name=f'±SD KAT (μ={mean_sd:.3f})',
        showlegend=True,
        hoverinfo='skip',
    ), row=1, col=1)

    # 3. Линия тренда
    if len(sub) >= 4:
        z = np.polyfit(sub['PRAI'], sub['p_kat_raw'], 1)
        y_fit = np.poly1d(z)(x_arr)
        fig.add_trace(go.Scatter(
            x=x_arr, y=np.clip(y_fit, 0, 1), mode='lines',
            line=dict(color='rgba(80,80,80,0.45)', dash='dot', width=1.8),
            name='Линия тренда', showlegend=True, hoverinfo='skip',
        ), row=1, col=1)

    # 4. Scatter по кластерам с error bars (CI KAT) и полупрозрачностью
    for c in [0, 1, 2]:
        s = sub[sub['dominant_cluster'] == c] if 'dominant_cluster' in sub.columns else sub
        if s.empty:
            continue
        cname  = cluster_names.get(c, str(c))
        chex   = cluster_color.get(c, '#888')

        fig.add_trace(go.Scatter(
            x=s['PRAI'],
            y=s['p_kat_raw'],
            mode='markers',
            name=cname,
            text=s[name_col],
            error_y=dict(
                type='data',
                array=s['sd_kat'].values,
                color=hex_rgba(chex, 0.35),
                thickness=1.5,
                width=5,
            ),
            hovertemplate=(
                '<b>%{text}</b><br>'
                'PRAI: %{x:.3f}<br>'
                'KAT: %{y:.3f}<br>'
                'KAT SD: ±%{error_y.array:.3f}<extra></extra>'
            ),
            marker=dict(
                size=12,
                color=hex_rgba(chex, 0.68),
                line=dict(width=1.8, color=hex_rgba(chex, 0.95)),
                symbol='circle',
            ),
        ), row=1, col=1)

    # ── ПРАВЫЙ ГРАФИК: гистограмма разниц + KDE + SD ──────────

    # 1. Гистограмма
    fig.add_trace(go.Histogram(
        x=diff,
        nbinsx=14,
        marker=dict(
            color=hex_rgba(C['blue'], 0.55),
            line=dict(color='white', width=0.8),
        ),
        name='KAT−PRAI',
        showlegend=False,
        hovertemplate='%{x:.3f}: %{y} пациентов<extra></extra>',
    ), row=1, col=2)

    # 2. KDE поверх гистограммы (нормальное приближение)
    x_kde  = np.linspace(diff.min() - 0.05, diff.max() + 0.05, 200)
    n_bins = 14
    bin_w  = (diff.max() - diff.min()) / n_bins
    scale  = len(diff) * bin_w   # масштаб плотности → счётчики
    y_kde  = _norm.pdf(x_kde, loc=bias, scale=sd_diff) * scale
    fig.add_trace(go.Scatter(
        x=x_kde, y=y_kde, mode='lines',
        line=dict(color=hex_rgba(C['blue'], 0.90), width=2.5),
        name='KDE (норм. прибл.)',
        hovertemplate='%{x:.3f}: плотность=%{y:.2f}<extra></extra>',
    ), row=1, col=2)

    # 3. Вертикальные линии: 0, bias, ±SD
    for x_v, color_v, dash_v, ann, pos in [
        (0,           C['red'],    'dash', 'Нет разницы',       'top right'),
        (bias,        C['orange'], 'dot',  f'bias={bias:+.3f}', 'top left'),
        (bias+sd_diff, '#888',     'dot',  f'+SD={bias+sd_diff:+.3f}', 'bottom right'),
        (bias-sd_diff, '#888',     'dot',  f'−SD={bias-sd_diff:+.3f}', 'bottom left'),
    ]:
        fig.add_vline(
            x=x_v,
            line_dash=dash_v,
            line_color=color_v,
            line_width=1.8,
            annotation_text=ann,
            annotation_position=pos,
            annotation_font_size=10,
            row=1, col=2,
        )

    # 4. Шейдированная зона ±SD вокруг bias
    fig.add_vrect(
        x0=bias - sd_diff, x1=bias + sd_diff,
        fillcolor=hex_rgba(C['orange'], 0.08),
        line_width=0,
        annotation_text=f'±1SD ({sd_diff:.3f})',
        annotation_position='top left',
        annotation_font_size=9,
        row=1, col=2,
    )

    # ── Оси и layout ──────────────────────────────────────────
    fig.update_xaxes(title_text='PRAI (реальный цикл)', range=[0, 1],
                     row=1, col=1, zeroline=False)
    fig.update_yaxes(title_text='KAT (симулированный)', range=[0, 1],
                     row=1, col=1, zeroline=False)
    fig.update_xaxes(title_text='KAT − PRAI', row=1, col=2, zeroline=False)
    fig.update_yaxes(title_text='N пациентов', row=1, col=2, zeroline=False)

    fig.update_layout(
        **LAYOUT,
        height=490,
        title=dict(
            text='KAT (симулированный) vs PRAI (реальный полный цикл)',
            font=dict(size=15),
        ),
        legend=dict(
            x=0.01, y=0.99,
            bgcolor='rgba(255,255,255,0.80)',
            bordercolor='#ddd', borderwidth=1,
            font=dict(size=11),
        ),
    )
    fig.show()

    # ── Аннотация метрик под графиком ─────────────────────────
    from IPython.display import display, HTML
    display(HTML(
        f'<div style="font-family:Inter,sans-serif;font-size:12px;'
        f'padding:10px 14px;border-radius:6px;'
        f'border-left:3px solid #1B4F72;display:inline-block;margin-top:6px">'
        f'<b>Pearson r</b> = {r:.3f} &nbsp;|&nbsp; '
        f'<b>Spearman ρ</b> = {rho:.3f} &nbsp;|&nbsp; '
        f'<b>MAE</b> = {mae:.3f} &nbsp;|&nbsp; '
        f'<b>Bias</b> = {bias:+.3f} &nbsp;|&nbsp; '
        f'<b>SD разниц</b> = {sd_diff:.3f} &nbsp;|&nbsp; '
        f'<b>N</b> = {len(sub)}'
        f'</div>'
    ))

    # ── Wilcoxon ──────────────────────────────────────────────
    if len(sub) >= 10:
        w_stat, w_p = wilcoxon(sub['p_kat_raw'], sub['PRAI'])
        verdict = (
            'Статистически значимое систематическое расхождение ⚠️'
            if w_p < 0.05 else 'Значимых расхождений нет ✅'
        )
        display(HTML(
            f'<div style="font-family:Inter,sans-serif;font-size:12px;'
            f'padding:8px 14px;background:border-radius:6px;'
            f'border-left:3px solid #F9A825;display:inline-block;margin-top:4px">'
            f'<b>Wilcoxon signed-rank:</b> stat={w_stat:.1f}, p={w_p:.4f} '
            f'→ {verdict}'
            f'</div>'
        ))

Pearson r = 0.681  |  Spearman ρ = 0.679  |  MAE = 0.125  |  Bias = +0.096  |  SD разниц = 0.131  |  N = 50

Wilcoxon signed-rank: stat=210.0, p=0.0000 → Статистически значимое систематическое расхождение ⚠️

## 6. Калибровка и метрики качества

In [9]:
df_p = df_u[df_u['Preg'].notna()].copy()
y_true = df_p['Preg'].astype(int)

eval_models = {
    'MC': 'p_per_transfer',
    'KAT': 'p_kat_raw',
    'NVSA': 'p_nvsa',
    'CSDI': 'p_csdi',
    'DT': 'DIGITAL TWIN',
    'PRAI': 'PRAI',
}
eval_models = {k: v for k, v in eval_models.items() if v in df_p.columns}

if  len(df_p) >= 6:
    results = []
    for name, col in eval_models.items():
        sub = df_p[[col,'Preg']].dropna()
        if len(sub) < 4:
            continue
        y, p = sub['Preg'].astype(int), sub[col]
        brier = brier_score_loss(y, p)
        try:
            auc = roc_auc_score(y, p)
        except Exception:
            auc = np.nan
        acc = ((p >= 0.5).astype(int) == y).mean()
        results.append({'Модель': name, 'N': len(sub),
                        'Brier ↓': round(brier,4),
                        'AUC ↑': round(auc,4) if not np.isnan(auc) else None,
                        'Acc@0.5': round(acc,3),
                        'Mean pred': round(p.mean(),3),
                        'Real rate': round(y.mean(),3)})
    res_df = pd.DataFrame(results)
    print('── Метрики качества ──')
    display(res_df)
else:
    print(f'Пациентов с исходом: {len(df_p)} — метрики рассчитаются при N≥6')

── Метрики качества ──


,Модель,N,Brier ↓,AUC ↑,Acc@0.5,Mean pred,Real rate
0,MC,12,0.2840,0.4444,0.500,0.648,0.5
1,KAT,12,0.2569,0.5000,0.417,0.506,0.5
2,NVSA,12,0.2522,0.4167,0.500,0.560,0.5
3,CSDI,12,0.2842,0.3889,0.417,0.389,0.5
4,DT,12,0.1720,0.9167,0.667,0.505,0.5
5,PRAI,12,0.2107,0.7222,0.583,0.484,0.5


In [10]:
# ── Violin по исходу ─────────────────────────────────────────
if len(df_p) >= 4:
    show_models = {k: v for k, v in list(eval_models.items())[:4]}
    fig = make_subplots(rows=1, cols=len(show_models),
                        subplot_titles=list(show_models.keys()))

    for i, (name, col) in enumerate(show_models.items(), 1):
        sub = df_p[[col,'Preg']].dropna()
        if sub.empty:
            continue
        for val, label, color in [(0,'Нет беременности',C['red']),
                                   (1,'Беременность',   C['green'])]:
            g = sub[sub['Preg']==val][col]
            if g.empty:
                continue
            fig.add_trace(go.Violin(
                y=g, name=label,
                box_visible=True, meanline_visible=True,
                fillcolor=hex_rgba(color, 0.27), line_color=color,
                opacity=0.8, showlegend=(i==1),
            ), row=1, col=i)

    fig.update_yaxes(range=[0,1], title_text='P(беременность)')
    fig.update_layout(**LAYOUT, height=400,
                      title='Предсказанная вероятность по группам исхода',
                      violinmode='overlay')
    fig.show()

## 7. Согласованность моделей пайплайна

In [11]:
corr_map = {
    'MC': 'p_per_transfer', 'Bayes': 'bayes_mean',
    'KAT': 'p_kat_raw', 'NVSA': 'p_nvsa',
    'CSDI': 'p_csdi', 'DT': 'DIGITAL TWIN', 'PRAI': 'PRAI',
}
corr_map = {k: v for k, v in corr_map.items() if v in df_u.columns}
corr_df  = df_u[[v for v in corr_map.values()]].dropna()
corr_df.columns = list(corr_map.keys())

pearson_m  = corr_df.corr(method='pearson').round(2)
spearman_m = corr_df.corr(method='spearman').round(2)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Pearson r', 'Spearman ρ'])

for col_idx, (mat, title) in enumerate(
    [(pearson_m, 'Pearson'), (spearman_m, 'Spearman')], 1
):
    labels = list(mat.columns)
    z = mat.values
    text = [[f'{v:.2f}' for v in row] for row in z]
    fig.add_trace(go.Heatmap(
        z=z, x=labels, y=labels,
        text=text, texttemplate='%{text}',
        colorscale='RdYlGn', zmid=0, zmin=-1, zmax=1,
        hovertemplate='%{y} vs %{x}: %{z:.3f}<extra></extra>',
        colorbar=dict(title='r', len=0.5,
                      x=0.46 if col_idx==1 else 1.01),
        showscale=True,
    ), row=1, col=col_idx)

fig.update_layout(**LAYOUT, height=420,
                  title='Матрица корреляций между моделями пайплайна')
fig.show()

print('\n── Пары с низкой согласованностью (|r| < 0.7) ──')
for i in range(len(pearson_m.columns)):
    for j in range(i+1, len(pearson_m.columns)):
        v = pearson_m.iloc[i,j]
        if abs(v) < 0.7:
            print(f'  {pearson_m.columns[i]} vs {pearson_m.columns[j]}: r={v:.3f}')


── Пары с низкой согласованностью (|r| < 0.7) ──
  MC vs CSDI: r=0.540
  MC vs DT: r=0.570
  MC vs PRAI: r=0.650
  Bayes vs CSDI: r=0.600
  Bayes vs DT: r=0.580
  KAT vs CSDI: r=0.550
  KAT vs DT: r=0.450
  KAT vs PRAI: r=0.680
  NVSA vs CSDI: r=0.600
  NVSA vs DT: r=0.580
  CSDI vs DT: r=0.350
  CSDI vs PRAI: r=0.330
  DT vs PRAI: r=0.660


In [12]:
# ══════════════════════════════════════════════════════════════
# АНАЛИЗ НЕОПРЕДЕЛЁННОСТИ МЕЖДУ МОДЕЛЯМИ — расширенная версия
#
# Панель 1: Scatter uncertainty map
#   X     = среднее предсказание (%)
#   Y     = SD между моделями (пп)
#   Цвет  = кластер (C0/C1/C2)
#   Размер = возраст
#   Форма = зона неопределённости
#   Обводка = исход
#
# Панель 2: Декомпозиция — откуда берётся расхождение
#   Box + strip для каждой модели, отсортированы по mean
# ══════════════════════════════════════════════════════════════

from IPython.display import display, HTML

name_col = 'patient_name' if 'patient_name' in df_u.columns else '_key'

# ── Подготовка ────────────────────────────────────────────────
pred_list = [c for c in
    ['p_per_transfer','p_kat_raw','p_nvsa','p_csdi','DIGITAL TWIN','PRAI']
    if c in df_u.columns]

MODEL_LABELS = {
    'p_per_transfer': 'MC pipeline',
    'p_kat_raw':      'KAT',
    'p_nvsa':         'NVSA',
    'p_csdi':         'CSDI',
    'DIGITAL TWIN':   'DT (таблица)',
    'PRAI':           'PRAI',
}

df_u_c = df_u.copy()
df_u_c['pred_mean']  = df_u_c[pred_list].mean(axis=1)
df_u_c['pred_std']   = df_u_c[pred_list].std(axis=1)
df_u_c['pred_range'] = df_u_c[pred_list].max(axis=1) - df_u_c[pred_list].min(axis=1)
df_u_c['n_models']   = df_u_c[pred_list].notna().sum(axis=1)

# Зоны неопределённости
ZONE_BINS   = [-1, 0.08, 0.18, 1]
ZONE_LABELS = ['Low', 'Moderate', 'High']
ZONE_SHAPES = {'Low': 'circle', 'Moderate': 'diamond', 'High': 'star'}
ZONE_COLORS_BG = {
    'Low':      'rgba(46,125,50,0.07)',
    'Moderate': 'rgba(245,166,35,0.07)',
    'High':     'rgba(198,40,40,0.10)',
}
df_u_c['uncertainty_group'] = pd.cut(
    df_u_c['pred_std'], bins=ZONE_BINS, labels=ZONE_LABELS)

CLUSTER_COLOR = {0: C['blue'], 1: C['red'], 2: C['green']}
CLUSTER_NAME  = {0: 'C0 Standard', 1: 'C1 Poor', 2: 'C2 High'}

# Размер по возрасту
if 'Age' in df_u_c.columns:
    age_min, age_max = df_u_c['Age'].min(), df_u_c['Age'].max()
    df_u_c['marker_sz'] = 9 + (df_u_c['Age'] - age_min) / (
                               age_max - age_min + 1e-9) * 12
else:
    df_u_c['marker_sz'] = 11

y_max = df_u_c['pred_std'].max() * 1.25 * 100
x_max = 100

# ── Список предсказаний для hover (все модели) ───────────────
def _hover_models(row):
    parts = []
    for col in pred_list:
        v = row.get(col, np.nan)
        if pd.notna(v):
            parts.append(f'{MODEL_LABELS.get(col, col)}: {v*100:.1f}%')
    return '<br>'.join(parts)

df_u_c['hover_models'] = df_u_c.apply(_hover_models, axis=1)

# ════════════════════════════════════════════════════════════════
# Создаём figure: 2 панели
# ════════════════════════════════════════════════════════════════
fig_unc = make_subplots(
    rows=2, cols=1,
    row_heights=[0.58, 0.42],
    subplot_titles=[
        'Карта неопределённости: среднее vs SD',
        'Декомпозиция: распределение предсказаний по моделям',
    ],
    vertical_spacing=0.14,
)

# ────────────────────────────────────────────────────────────
# ПАНЕЛЬ 1 — Scatter
# ────────────────────────────────────────────────────────────

# 1a. Фоновые зоны
for y0_v, y1_v, fc, label in [
    (0,   8,  ZONE_COLORS_BG['Low'],      'Low SD (<8пп)'),
    (8,   18, ZONE_COLORS_BG['Moderate'], 'Moderate (8–18пп)'),
    (18,  y_max * 1.05, ZONE_COLORS_BG['High'], 'High (>18пп)'),
]:
    fig_unc.add_shape(type='rect',
        x0=0, x1=x_max, y0=y0_v, y1=min(y1_v, y_max * 1.05),
        fillcolor=fc, line_width=0,
        row=1, col=1,
    )
    fig_unc.add_annotation(
        x=1, y=min((y0_v + y1_v) / 2, y_max * 0.95),
        text=label, xanchor='left',
        showarrow=False,
        font=dict(size=9, color='rgba(180,180,180,0.80)'),
        row=1, col=1,
    )

# 1b. Горизонтальные пороги
for y_v, lbl, col_line, ann_pos in [
    (8,  'SD=8пп',  'rgba(46,125,50,0.55)',  'top right'),
    (18, 'SD=18пп', 'rgba(198,40,40,0.55)',  'bottom right'),
]:
    fig_unc.add_hline(
        y=y_v, line_dash='dot',
        line_color=col_line, line_width=1.4,
        annotation_text=lbl,
        annotation_position=ann_pos,
        annotation_font_size=9,
        row=1, col=1,
    )

# 1c. Scatter по кластерам × зонам
clusters = (df_u_c['dominant_cluster'].dropna().unique().astype(int).tolist()
            if 'dominant_cluster' in df_u_c.columns else [-1])

for cluster_id in sorted(clusters):
    if 'dominant_cluster' in df_u_c.columns:
        d = df_u_c[df_u_c['dominant_cluster'] == cluster_id].copy()
    else:
        d = df_u_c.copy()
    if d.empty:
        continue

    chex   = CLUSTER_COLOR.get(cluster_id, '#888')
    c_name = CLUSTER_NAME.get(cluster_id, str(cluster_id))

    # Форма по зоне неопределённости
    symbols = d['uncertainty_group'].map(ZONE_SHAPES).fillna('circle')

    # Обводка по исходу
    if 'Preg' in d.columns:
        border_c = ['rgba(0,200,0,1.0)'  if v == 1 else
                    'rgba(220,0,0,1.0)'  if v == 0 else
                    'rgba(255,255,255,0.8)' for v in d['Preg']]
        border_w = [2.5 if v in [0,1] else 1.2 for v in d['Preg']]
    else:
        border_c = ['rgba(255,255,255,0.8)'] * len(d)
        border_w = [1.2] * len(d)

    # Hover со всеми моделями
    hover_txt = (
        '<b>%{text}</b><br>'
        'Ср. прогноз: %{x:.1f}%<br>'
        'SD между моделями: %{y:.1f}пп<br>'
        'Возраст: %{customdata[0]:.0f}<br>'
        '─────────────────<br>'
        '%{customdata[1]}'
        '<extra></extra>'
    )
    customdata = np.column_stack([
        d.get('Age', pd.Series(0, index=d.index)).fillna(0),
        d['hover_models'],
    ])

    fig_unc.add_trace(go.Scatter(
        x=d['pred_mean'] * 100,
        y=d['pred_std']  * 100,
        mode='markers',
        name=c_name,
        legendgroup=f'cluster_{cluster_id}',
        text=d[name_col],
        customdata=customdata,
        hovertemplate=hover_txt,
        marker=dict(
            symbol=symbols.values,
            size=d['marker_sz'].values,
            color=hex_rgba(chex, 0.70),
            line=dict(color=border_c, width=border_w),
        ),
    ), row=1, col=1)

# 1d. Подписи только High-зоны
high_unc = df_u_c[df_u_c['uncertainty_group'] == 'High']
for _, row in high_unc.iterrows():
    fig_unc.add_annotation(
        x=row['pred_mean'] * 100,
        y=row['pred_std']  * 100,
        text=str(row[name_col])[:16],
        showarrow=True, arrowhead=2,
        arrowsize=0.8, arrowwidth=1.2,
        arrowcolor='rgba(198,40,40,0.50)',
        ax=20, ay=-20,
        font=dict(size=9, color='rgba(198,40,40,0.90)'),
        row=1, col=1,
    )

# ────────────────────────────────────────────────────────────
# ПАНЕЛЬ 2 — Декомпозиция по моделям
# ────────────────────────────────────────────────────────────

# Сортируем модели по убыванию среднего — сразу видна иерархия
model_means = {c: df_u_c[c].mean() for c in pred_list if c in df_u_c.columns}
sorted_models = sorted(model_means, key=lambda c: model_means[c], reverse=True)

# Цвет модели по позиции
model_palette = [C['blue'], C['teal'], C['green'],
                 C['orange'], C['red'], C['purple']]

for mi, col in enumerate(sorted_models):
    vals = df_u_c[col].dropna() * 100
    mlabel = MODEL_LABELS.get(col, col)
    mcolor = model_palette[mi % len(model_palette)]

    # Box
    fig_unc.add_trace(go.Box(
        x=vals,
        y=[mlabel] * len(vals),
        orientation='h',
        name=mlabel,
        legendgroup=f'model_{col}',
        showlegend=False,
        boxmean=True,
        fillcolor=hex_rgba(mcolor, 0.15),
        line=dict(color=hex_rgba(mcolor, 0.60), width=1.5),
        marker=dict(size=4, opacity=0.0),
        width=0.5,
        hovertemplate=(
            f'<b>{mlabel}</b><br>'
            'Предсказание: %{x:.1f}%<extra></extra>'
        ),
    ), row=2, col=1)

    # Strip (jitter)
    np.random.seed(mi + 42)
    jitter = np.random.uniform(-0.18, 0.18, len(vals))
    # Цвет точек по зоне неопределённости пациента
    sub_d = df_u_c[[col, 'uncertainty_group']].dropna(subset=[col])
    dot_colors = [
        hex_rgba(C['green'],  0.65) if g == 'Low' else
        hex_rgba(C['orange'], 0.65) if g == 'Moderate' else
        hex_rgba(C['red'],    0.80)
        for g in sub_d['uncertainty_group']
    ]

    fig_unc.add_trace(go.Scatter(
        x=sub_d[col] * 100,
        y=[mi + 0.5 + j * 0.2 for j in jitter[:len(sub_d)]],
        mode='markers',
        name=f'{mlabel} (dots)',
        legendgroup=f'model_{col}',
        showlegend=False,
        text=df_u_c.loc[sub_d.index, name_col],
        hovertemplate=(
            f'<b>%{{text}}</b><br>'
            f'{mlabel}: %{{x:.1f}}%<extra></extra>'
        ),
        marker=dict(
            size=7,
            color=dot_colors,
            line=dict(color='rgba(255,255,255,0.6)', width=0.8),
        ),
    ), row=2, col=1)

# Медиана по когорте — вертикальная линия
cohort_median = df_u_c['pred_mean'].median() * 100
fig_unc.add_vline(
    x=cohort_median, line_dash='dot',
    line_color='rgba(150,150,150,0.55)', line_width=1.5,
    annotation_text=f'Ср. {cohort_median:.0f}%',
    annotation_position='top right',
    annotation_font_size=9,
    row=2, col=1,
)

# Y-тики правой панели
fig_unc.update_yaxes(
    tickvals=list(range(len(sorted_models))),
    ticktext=[MODEL_LABELS.get(c, c) for c in sorted_models],
    row=2, col=1,
)

# ────────────────────────────────────────────────────────────
# Layout
# ────────────────────────────────────────────────────────────
fig_unc.update_xaxes(
    title_text='Среднее предсказание (%)', range=[0, 105],
    zeroline=False, row=1, col=1,
)
fig_unc.update_yaxes(
    title_text='SD между моделями (пп)', range=[0, y_max],
    zeroline=False, row=1, col=1,
)
fig_unc.update_xaxes(
    title_text='Предсказанная вероятность (%)',
    range=[0, 105], zeroline=False, row=2, col=1,
)
fig_unc.update_yaxes(zeroline=False, row=2, col=1)

fig_unc.update_layout(
    **{k: v for k, v in LAYOUT.items() if k != 'margin'},
    height=860,
    title=dict(
        text='Анализ неопределённости между моделями',
        font=dict(size=14),
    ),
    legend=dict(
        x=0.0, y=-0.15,
        orientation='h',
        bgcolor='rgba(0,0,0,0)',
        bordercolor='rgba(150,150,150,0.35)',
        borderwidth=1,
        font=dict(size=10),
        tracegroupgap=4,
    ),
    margin=dict(t=70, b=120, l=80, r=30),
)
fig_unc.show()

# ────────────────────────────────────────────────────────────
# Сводная таблица неопределённости
# ────────────────────────────────────────────────────────────
zone_stat = (df_u_c.groupby('uncertainty_group', observed=True)
             .agg(
                 N          =('pred_std','count'),
                 SD_median  =('pred_std', lambda x: round(x.median()*100,1)),
                 Mean_median=('pred_mean', lambda x: round(x.median()*100,1)),
                 Age_median =('Age','median'),
             ))
if 'Preg' in df_u_c.columns:
    zone_stat['Preg_rate'] = (df_u_c.groupby('uncertainty_group', observed=True)
                              ['Preg'].mean().round(3))
zone_stat.columns = ['N', 'SD медиана (пп)', 'Ср.прогноз (%)', 'Возраст', 'Preg rate']
zone_stat = zone_stat.reindex(['Low','Moderate','High']).dropna(how='all')

display(HTML(
    '<div style="font-family:Inter,sans-serif;font-size:11px;'
    'padding:9px 14px;border-radius:6px;'
    'border-left:3px solid #888;line-height:2;display:inline-block">'
    '<b>Панель 1:</b> Форма = зона SD (● Low · ◆ Moderate · ★ High) · '
    'Цвет = кластер · Размер = возраст · Обводка = исход<br>'
    '<b>Панель 2:</b> Точки окрашены по зоне неопределённости · '
    'Модели отсортированы по среднему предсказанию (сверху = выше прогноз)'
    '</div>'
))
display(zone_stat)

Панель 1: Форма = зона SD (● Low · ◆ Moderate · ★ High) · Цвет = кластер · Размер = возраст · Обводка = исход Панель 2: Точки окрашены по зоне неопределённости · Модели отсортированы по среднему предсказанию (сверху = выше прогноз)

,N,SD медиана (пп),Ср.прогноз (%),Возраст,Preg rate
uncertainty_group,,,,,
Low,11,6.3,46.0,36.0,0.667
Moderate,38,13.1,47.6,29.5,0.375
High,2,18.8,53.7,30.5,1.000


In [13]:
# ════════════════════════════════════════════════════════════════
#  FIGURE 2 — Top disagreement patients
# ════════════════════════════════════════════════════════════════

top15 = (
    df_u_c
    .nlargest(15, 'pred_std')
    .sort_values('pred_std')
)

fig2 = go.Figure()

fig2.add_trace(go.Bar(
    y=top15[name_col].str[:24],
    x=top15['pred_std'] * 100,

    orientation='h',

    marker=dict(
        color=top15['pred_std'],
        colorscale='Reds',
        line=dict(
            color='white',
            width=1
        )
    ),

    text=[f'{v*100:.1f}' for v in top15['pred_std']],
    textposition='outside',

    hovertemplate=
        '<b>%{y}</b><br>' +
        'Disagreement SD: %{x:.1f} pp' +
        '<extra></extra>',
))

# Critical threshold
fig2.add_vline(
    x=18,
    line_dash='dash',
    line_color='red',
    annotation_text='Critical disagreement'
)

fig2.update_layout(
    **LAYOUT,

    title=dict(
        text='Patients with highest model disagreement',
        x=0.5
    ),

    height=700,
    width=900,

    xaxis=dict(
        title='Prediction disagreement (SD, percentage points)'
    ),

    yaxis=dict(
        title=''
    ),

    template='plotly_white'
)

fig2.show()

## 8. Кластерный анализ

In [14]:
# ══════════════════════════════════════════════════════════════
# КЛАСТЕРНЫЙ ДАШБОРД — Donut + Профиль + Реальные исходы
# (без Box-графика беременности — он вынесен отдельно)
# ══════════════════════════════════════════════════════════════

from IPython.display import display, HTML
import scipy.stats as _scipy_stats

name_col = 'patient_name' if 'patient_name' in df_u.columns else '_key'

if 'dominant_cluster' not in df_u.columns:
    print('⚠️  Нет колонки dominant_cluster')
else:
    CLUSTER_NAMES = {0: 'C0 Standard', 1: 'C1 Poor', 2: 'C2 High'}
    CLUSTER_HEX   = {0: '#1976D2',     1: '#C62828', 2: '#2E7D32'}
    PATTERNS      = {0: '/',           1: 'x',        2: ''}

    # Локальный df_preg
    _df_preg = df_u[df_u['Preg'].notna()].copy()
    _df_preg['dominant_cluster'] = (
        _df_preg['dominant_cluster'].dropna().astype(int))
    df_preg = _df_preg

    counts  = df_u['dominant_cluster'].value_counts().sort_index()
    n_total = len(df_u)

    def binom_ci(k, n, alpha=0.05):
        if n == 0: return 0.0, 0.0
        lo = _scipy_stats.beta.ppf(alpha/2,   k,   n-k+1) if k > 0 else 0.0
        hi = _scipy_stats.beta.ppf(1-alpha/2, k+1, n-k)   if k < n else 1.0
        return lo * 100, hi * 100

    # ─────────────────────────────────────────────────────────
    # ФИГУРА 1: Donut (отдельно — Pie несовместим с xy)
    # ─────────────────────────────────────────────────────────
    fig_donut = go.Figure(go.Pie(
        labels=[CLUSTER_NAMES.get(c, c) for c in counts.index],
        values=counts.values,
        marker=dict(
            colors=[hex_rgba(CLUSTER_HEX.get(c, '#888'), 0.45)
                    for c in counts.index],
            line=dict(
                color=[CLUSTER_HEX.get(c, '#888') for c in counts.index],
                width=2,
            ),
        ),
        textinfo='label+percent',
        textfont=dict(size=11),
        hole=0.45,
        pull=[0.04] * len(counts),
        sort=False,
        hovertemplate='%{label}<br>%{value} пациентов · %{percent}<extra></extra>',
    ))
    fig_donut.add_annotation(
        text=f'<b>{n_total}</b><br>пациентов',
        x=0.5, y=0.5,
        xref='paper', yref='paper',
        showarrow=False,
        font=dict(size=13),
        align='center',
    )
    fig_donut.update_layout(
        **{k: v for k, v in LAYOUT.items() if k != 'margin'},
        height=310,
        title=dict(text='Состав когорты по кластерам', font=dict(size=13)),
        margin=dict(t=50, b=20, l=30, r=30),
        legend=dict(
            orientation='h', x=0.5, xanchor='center', y=-0.05,
            bgcolor='rgba(0,0,0,0)', font=dict(size=10),
        ),
    )
    fig_donut.show()

    # ─────────────────────────────────────────────────────────
    # ФИГУРА 2: Профиль + Реальные исходы (1×2, только xy)
    # ─────────────────────────────────────────────────────────
    fig_cl = make_subplots(
        rows=1, cols=2,
        subplot_titles=[
            'Профиль кластеров (медианы, нормированные)',
            'Реальные исходы + 95% CI',
        ],
        column_widths=[0.58, 0.42],
        horizontal_spacing=0.12,
    )

    # ── (1,1) Профиль признаков ───────────────────────────────
    profile_map = {col: lbl for col, lbl in [
        ('N folicules OPU', 'Фолл.ТВП'),
        ('MII',             'MII'),
        ('Bl',              'Бластоцисты'),
        ('Good Bl',         'Хор.кач.'),
        ('Age',             'Возраст'),
    ] if col in df_u.columns}

    feat_labels = list(profile_map.values())
    norm_data, raw_data = {}, {}
    for col, lbl in profile_map.items():
        by_c    = df_u.groupby('dominant_cluster')[col].median()
        col_max = by_c.max()
        norm_data[lbl] = by_c / col_max if col_max > 0 else by_c
        raw_data[lbl]  = by_c

    for c in [0, 1, 2]:
        chex     = CLUSTER_HEX[c]
        cname    = CLUSTER_NAMES[c]
        y_vals   = [norm_data[lbl].get(c, 0) for lbl in feat_labels]
        raw_vals = [raw_data[lbl].get(c, 0)  for lbl in feat_labels]

        fig_cl.add_trace(go.Bar(
            name=cname,
            x=feat_labels,
            y=y_vals,
            legendgroup=f'c{c}',
            showlegend=True,
            marker=dict(
                color=hex_rgba(chex, 0.40),
                line=dict(color=hex_rgba(chex, 0.70), width=1.5),
                pattern=dict(
                    shape=PATTERNS[c],
                    size=4,
                    fgcolor=hex_rgba(chex, 0.35),
                ),
            ),
            text=[f'{v:.0f}' for v in raw_vals],
            textposition='outside',
            textfont=dict(size=9),
            hovertemplate=(
                f'<b>{cname}</b><br>'
                '%{x}: %{text} (норм. %{y:.2f})<extra></extra>'
            ),
        ), row=1, col=1)

    fig_cl.update_yaxes(
        title_text='Нормированная медиана',
        range=[0, 1.30],
        zeroline=False,
        row=1, col=1,
    )

    # ── (1,2) Реальные исходы + биномиальный CI ───────────────
    if len(df_preg) >= 3:
        preg_by_c = (df_preg.groupby('dominant_cluster')['Preg']
                     .agg(['sum', 'count'])
                     .rename(columns={'sum': 'k', 'count': 'n'}))

        x_labels = [CLUSTER_NAMES.get(int(c), c) for c in preg_by_c.index]
        y_vals   = (preg_by_c['k'] / preg_by_c['n'] * 100).values
        ci_lo, ci_hi = [], []
        for i, (_, r) in enumerate(preg_by_c.iterrows()):
            lo, hi = binom_ci(int(r['k']), int(r['n']))
            ci_lo.append(y_vals[i] - lo)
            ci_hi.append(hi - y_vals[i])

        fig_cl.add_trace(go.Bar(
            x=x_labels,
            y=y_vals,
            marker=dict(
                color=[hex_rgba(CLUSTER_HEX.get(int(c), '#888'), 0.40)
                       for c in preg_by_c.index],
                line=dict(
                    color=[CLUSTER_HEX.get(int(c), '#888')
                           for c in preg_by_c.index],
                    width=1.8,
                ),
                pattern=dict(
                    shape='/',
                    size=5,
                    fgcolor=[hex_rgba(CLUSTER_HEX.get(int(c), '#888'), 0.30)
                             for c in preg_by_c.index],
                ),
            ),
            error_y=dict(
                type='data', symmetric=False,
                array=ci_hi, arrayminus=ci_lo,
                color='rgba(180,180,180,0.70)',
                thickness=2.0, width=9,
            ),
            text=[f'{int(r["k"])}/{int(r["n"])}' for _, r in preg_by_c.iterrows()],
            textposition='outside',
            textfont=dict(size=11),
            showlegend=False,
            hovertemplate='<b>%{x}</b><br>%{y:.1f}%<br>%{text}<extra></extra>',
        ), row=1, col=2)

        overall = df_preg['Preg'].mean() * 100
        fig_cl.add_hline(
            y=overall,
            line_dash='dot',
            line_color='rgba(150,150,150,0.55)',
            line_width=1.5,
            annotation_text=f'Когорта {overall:.0f}%',
            annotation_position='top right',
            annotation_font_size=9,
            row=1, col=2,
        )
    else:
        fig_cl.add_annotation(
            text='Недостаточно данных об исходах',
            x=0.5, y=0.5, xref='x2 domain', yref='y2 domain',
            showarrow=False, font=dict(size=11, color='gray'),
        )

    fig_cl.update_yaxes(
        title_text='% беременности',
        range=[0, 115],
        zeroline=False,
        row=1, col=2,
    )

    fig_cl.update_layout(
        **{k: v for k, v in LAYOUT.items() if k != 'margin'},
        height=430,
        barmode='group',
        title=dict(text='Профиль и исходы по кластерам', font=dict(size=14)),
        legend=dict(
            orientation='h',
            x=0.5, xanchor='center',
            y=-0.12, yanchor='top',
            bgcolor='rgba(0,0,0,0)',
            bordercolor='rgba(150,150,150,0.35)',
            borderwidth=1,
            font=dict(size=10),
            tracegroupgap=4,
        ),
        margin=dict(t=60, b=90, l=60, r=30),
    )
    fig_cl.show()

    # ── Сводная таблица ───────────────────────────────────────
    rows_tbl = []
    for c in sorted(df_u['dominant_cluster'].dropna().unique().astype(int)):
        sub   = df_u[df_u['dominant_cluster'] == c]
        sub_p = df_preg[df_preg['dominant_cluster'] == c]
        k = int(sub_p['Preg'].sum()) if len(sub_p) > 0 else 0
        n = len(sub_p)
        lo, hi = binom_ci(k, n)
        rows_tbl.append({
            'Кластер':      CLUSTER_NAMES[c],
            'N':            len(sub),
            'Мед. возраст': f'{sub["Age"].median():.0f}' if 'Age' in sub else '—',
            'Мед. OCC':     f'{sub["OCC"].median():.0f}' if 'OCC' in sub else '—',
            'Мед. KAT':     f'{sub["p_kat_raw"].median():.2f}' if 'p_kat_raw' in sub else '—',
            'Мед. PRAI':    f'{sub["PRAI"].median():.2f}' if 'PRAI' in sub else '—',
            'Беременность': f'{k}/{n} ({k/n*100:.0f}%)' if n > 0 else '—',
            '95% CI':       f'{lo:.0f}–{hi:.0f}%' if n > 0 else '—',
        })

    display(HTML(
        '<div style="font-family:Inter,sans-serif;font-size:11px;'
        'padding:9px 14px;border-radius:6px;'
        'border-left:3px solid #888;line-height:2;display:inline-block">'
        '<b>Профиль:</b> значения = реальные медианы · высота = доля от максимума · '
        'штриховка различает кластеры<br>'
        '<b>Исходы:</b> error bars = биномиальный 95% CI (Clopper–Pearson)'
        '</div>'
    ))
    display(pd.DataFrame(rows_tbl).set_index('Кластер'))

Профиль: значения = реальные медианы · высота = доля от максимума · штриховка различает кластеры Исходы: error bars = биномиальный 95% CI (Clopper–Pearson)

,N,Мед. возраст,Мед. OCC,Мед. KAT,Мед. PRAI,Беременность,95% CI
Кластер,,,,,,,
C0 Standard,21,30,14,0.55,0.45,4/6 (67%),22–96%
C1 Poor,23,32,4,0.34,0.23,1/3 (33%),1–91%
C2 High,7,27,31,0.63,0.51,1/3 (33%),1–91%


In [15]:
# ══════════════════════════════════════════════════════════════
# P(БЕРЕМЕННОСТЬ) ПО КЛАСТЕРАМ — отдельный график
# Box + Strip: размер=возраст, обводка=исход, цвет=кластер
# ══════════════════════════════════════════════════════════════

name_col = 'patient_name' if 'patient_name' in df_u.columns else '_key'

CLUSTER_NAMES = {0: 'C0 Standard', 1: 'C1 Poor', 2: 'C2 High'}
CLUSTER_HEX   = {0: '#1976D2',     1: '#C62828', 2: '#2E7D32'}

fig_preg_cl = go.Figure()
np.random.seed(42)

for ci, c in enumerate([0, 1, 2]):
    sub_c = df_u[df_u['dominant_cluster'] == c].copy().reset_index(drop=True)
    vals  = sub_c['p_per_transfer'].dropna()
    if vals.empty:
        continue

    chex  = CLUSTER_HEX[c]
    cname = CLUSTER_NAMES[c]

    # ── Box ───────────────────────────────────────────────────
    fig_preg_cl.add_trace(go.Box(
        x=[ci] * len(vals),
        y=vals.values,
        name=cname,
        boxmean=True,
        fillcolor=hex_rgba(chex, 0.15),
        line=dict(color=hex_rgba(chex, 0.55), width=2),
        marker=dict(size=3, opacity=0),
        width=0.55,
        legendgroup=f'c{c}',
        showlegend=True,
        hoverinfo='skip',
    ))

    # ── Обводка по исходу ────────────────────────────────────
    if 'Preg' in sub_c.columns:
        border_c = ['rgba(0,200,0,1.0)'     if v == 1 else
                    'rgba(220,0,0,1.0)'     if v == 0 else
                    'rgba(255,255,255,0.75)' for v in sub_c['Preg']]
        border_w = [3.0 if v in [0, 1] else 1.2 for v in sub_c['Preg']]
    else:
        border_c = ['rgba(255,255,255,0.75)'] * len(sub_c)
        border_w = [1.2] * len(sub_c)

    # ── Размер по возрасту ────────────────────────────────────
    if 'Age' in sub_c.columns:
        age_min, age_max = sub_c['Age'].min(), sub_c['Age'].max()
        sizes = (8 + (sub_c['Age'] - age_min) /
                 (age_max - age_min + 1e-9) * 12).values
    else:
        sizes = np.full(len(sub_c), 10)

    # ── Hover: все модели ─────────────────────────────────────
    pred_cols_h = [c_ for c_ in
        ['p_per_transfer','p_kat_raw','p_nvsa','p_csdi','DIGITAL TWIN','PRAI']
        if c_ in sub_c.columns]
    MODEL_LBL = {
        'p_per_transfer': 'MC',
        'p_kat_raw':      'KAT',
        'p_nvsa':         'NVSA',
        'p_csdi':         'CSDI',
        'DIGITAL TWIN':   'DT',
        'PRAI':           'PRAI',
    }
    def _build_hover(row_):
        parts = [f'{MODEL_LBL.get(c_,c_)}: {row_[c_]*100:.1f}%'
                 for c_ in pred_cols_h if pd.notna(row_.get(c_))]
        return '<br>'.join(parts)
    sub_c['_hover_models'] = sub_c.apply(_build_hover, axis=1)

    age_txt = sub_c['Age'].map(lambda v: f'{v:.0f}').values if 'Age' in sub_c.columns else ['—']*len(sub_c)

    idx_p  = vals.index
    jitter = np.random.uniform(-0.17, 0.17, len(idx_p))

    customdata = np.column_stack([
        age_txt[idx_p],
        sub_c.loc[idx_p, '_hover_models'].values,
    ])

    fig_preg_cl.add_trace(go.Scatter(
        x=ci + jitter,
        y=vals.values,
        mode='markers',
        name=cname,
        showlegend=False,
        legendgroup=f'c{c}',
        text=sub_c.loc[idx_p, name_col],
        customdata=customdata,
        hovertemplate=(
            '<b>%{text}</b><br>'
            f'Кластер: {cname}<br>'
            'Возраст: %{customdata[0]}<br>'
            '─────────────────<br>'
            '%{customdata[1]}'
            '<extra></extra>'
        ),
        marker=dict(
            size=sizes[idx_p],
            color=hex_rgba(chex, 0.55),
            line=dict(
                color=[border_c[i] for i in idx_p],
                width=[border_w[i] for i in idx_p],
            ),
        ),
    ))

# ── Горизонтальная линия среднего по когорте ─────────────────
cohort_mean = df_u['p_per_transfer'].mean() * 100
fig_preg_cl.add_hline(
    y=cohort_mean / 100,
    line_dash='dot',
    line_color='rgba(150,150,150,0.50)',
    line_width=1.5,
    annotation_text=f'Когорта {cohort_mean:.0f}%',
    annotation_position='top right',
    annotation_font_size=9,
)

# ── Layout ────────────────────────────────────────────────────
fig_preg_cl.update_xaxes(
    tickvals=[0, 1, 2],
    ticktext=[CLUSTER_NAMES[c] for c in [0, 1, 2]],
    zeroline=False,
)
fig_preg_cl.update_yaxes(
    title_text='P(беременность) — MC прогноз',
    range=[0, 1.08],
    zeroline=False,
    gridcolor='rgba(200,200,200,0.20)',
)
fig_preg_cl.update_layout(
    **{k: v for k, v in LAYOUT.items() if k != 'margin'},
    height=500,
    title=dict(
        text='P(беременность) по кластерам',
        font=dict(size=14),
    ),
    legend=dict(
        orientation='h',
        x=0.5, xanchor='center',
        y=-0.10, yanchor='top',
        bgcolor='rgba(0,0,0,0)',
        bordercolor='rgba(150,150,150,0.35)',
        borderwidth=1,
        font=dict(size=11),
    ),
    margin=dict(t=60, b=80, l=70, r=30),
)
fig_preg_cl.show()

from IPython.display import display, HTML
display(HTML(
    '<div style="font-family:Inter,sans-serif;font-size:11px;'
    'padding:8px 12px;border-radius:6px;'
    'border-left:3px solid #888;line-height:1.9;display:inline-block">'
    '<b>Размер</b> = возраст (больше → старше) &nbsp;|&nbsp; '
    '<b>Обводка:</b> '
    '<span style="color:#00C800">зелёная</span> = беременность ✅ · '
    '<span style="color:#E60000">красная</span> = нет ❌ · '
    'белая = нет данных &nbsp;|&nbsp; '
    '<b>Hover</b> = все модели пайплайна'
    '</div>'
))

Размер = возраст (больше → старше)  |  Обводка: зелёная = беременность ✅ · красная = нет ❌ · белая = нет данных  |  Hover = все модели пайплайна

In [16]:
# ══════════════════════════════════════════════════════════════
# ВЕРОЯТНОСТИ ПРИНАДЛЕЖНОСТИ К КЛАСТЕРАМ
#
# Фигура 1: Тернарный scatter — каждый пациент как точка
#           в симплексе вероятностей (C0, C1, C2)
# Фигура 2: Горизонтальные stacked bars — уверенность модели
#           по каждому пациенту, отсортированные по кластерам
# ══════════════════════════════════════════════════════════════

from IPython.display import display, HTML

name_col = 'patient_name' if 'patient_name' in df_u.columns else '_key'

CP = ['cluster_c0_prob', 'cluster_c1_prob', 'cluster_c2_prob']
if not all(c in df_u.columns for c in CP):
    print(f'⚠️  Отсутствуют колонки: {[c for c in CP if c not in df_u.columns]}')
else:
    CLUSTER_NAMES = {0: 'C0 Standard', 1: 'C1 Poor', 2: 'C2 High'}
    CLUSTER_HEX   = {0: '#1976D2',     1: '#C62828', 2: '#2E7D32'}

    sub = df_u[CP + ['dominant_cluster', name_col]
               + (['Age']  if 'Age'  in df_u.columns else [])
               + (['Preg'] if 'Preg' in df_u.columns else [])
               ].dropna(subset=CP).copy()
    sub['dominant_cluster'] = sub['dominant_cluster'].astype(int)

    # Энтропия как мера неуверенности
    probs = sub[CP].values.clip(1e-9, 1)
    sub['entropy']    = -np.sum(probs * np.log(probs), axis=1)
    sub['confidence'] = sub[CP].max(axis=1)

    # Размер маркера
    if 'Age' in sub.columns:
        age_min, age_max = sub['Age'].min(), sub['Age'].max()
        sub['msize'] = 9 + (sub['Age'] - age_min) / (age_max - age_min + 1e-9) * 12
    else:
        sub['msize'] = 11

    # Обводка по исходу
    if 'Preg' in sub.columns:
        sub['border_c'] = sub['Preg'].map(
            lambda v: 'rgba(0,200,0,1.0)'     if v == 1 else
                      'rgba(220,0,0,1.0)'     if v == 0 else
                      'rgba(255,255,255,0.75)')
        sub['border_w'] = sub['Preg'].map(
            lambda v: 3.0 if v in [0, 1] else 1.2)
    else:
        sub['border_c'] = 'rgba(255,255,255,0.75)'
        sub['border_w'] = 1.2

    # ─────────────────────────────────────────────────────────
    # ФИГУРА 1: Тернарный scatter
    # ─────────────────────────────────────────────────────────
    fig_tern = go.Figure()

    for c in [0, 1, 2]:
        d = sub[sub['dominant_cluster'] == c]
        if d.empty:
            continue
        chex  = CLUSTER_HEX[c]
        cname = CLUSTER_NAMES[c]

        # Hover
        hover_lines = [
            '<b>%{text}</b>',
            'C0 Standard: %{a:.2f}',
            'C1 Poor:     %{b:.2f}',
            'C2 High:     %{c:.2f}',
            'Уверенность: %{customdata[0]:.2f}',
            'Энтропия:    %{customdata[1]:.3f}',
        ]
        if 'Age' in d.columns:
            hover_lines.append('Возраст: %{customdata[2]:.0f}')
        hover_lines.append('<extra></extra>')

        customdata = np.column_stack([
            d['confidence'],
            d['entropy'],
            d.get('Age', pd.Series(0, index=d.index)).fillna(0),
        ])

        fig_tern.add_trace(go.Scatterternary(
            a=d['cluster_c0_prob'],
            b=d['cluster_c1_prob'],
            c=d['cluster_c2_prob'],
            mode='markers',
            name=cname,
            text=d[name_col],
            customdata=customdata,
            hovertemplate='<br>'.join(hover_lines),
            marker=dict(
                size=d['msize'].values,
                color=hex_rgba(chex, 0.62),
                line=dict(
                    color=d['border_c'].values,
                    width=d['border_w'].values,
                ),
                symbol='circle',
            ),
        ))

    # Подписи пациентов с высокой энтропией (на границе кластеров)
    ENTROPY_THRESH = sub['entropy'].quantile(0.80)
    for _, row in sub[sub['entropy'] >= ENTROPY_THRESH].iterrows():
        fig_tern.add_trace(go.Scatterternary(
            a=[row['cluster_c0_prob']],
            b=[row['cluster_c1_prob']],
            c=[row['cluster_c2_prob']],
            mode='text',
            text=[str(row[name_col])[:12]],
            textfont=dict(size=8, color='rgba(180,180,180,0.85)'),
            showlegend=False,
            hoverinfo='skip',
        ))

    # ── Разметка вершин ───────────────────────────────────────
    TERN_LAYOUT = dict(
        aaxis=dict(
            title=dict(text='C0 Standard', font=dict(size=12)),
            min=0, linecolor='rgba(150,150,150,0.4)',
            gridcolor='rgba(150,150,150,0.15)',
            tickfont=dict(size=9),
        ),
        baxis=dict(
            title=dict(text='C1 Poor', font=dict(size=12)),
            min=0, linecolor='rgba(150,150,150,0.4)',
            gridcolor='rgba(150,150,150,0.15)',
            tickfont=dict(size=9),
        ),
        caxis=dict(
            title=dict(text='C2 High', font=dict(size=12)),
            min=0, linecolor='rgba(150,150,150,0.4)',
            gridcolor='rgba(150,150,150,0.15)',
            tickfont=dict(size=9),
        ),
        bgcolor='rgba(0,0,0,0)',
    )

    fig_tern.update_layout(
        **{k: v for k, v in LAYOUT.items() if k not in ('margin',)},
        ternary=TERN_LAYOUT,
        height=520,
        title=dict(
            text='Вероятности принадлежности к кластерам — тернарный симплекс',
            font=dict(size=14),
        ),
        legend=dict(
            orientation='h', x=0.5, xanchor='center',
            y=-0.06, yanchor='top',
            bgcolor='rgba(0,0,0,0)',
            bordercolor='rgba(150,150,150,0.35)', borderwidth=1,
            font=dict(size=11),
        ),
        margin=dict(t=60, b=80, l=60, r=60),
    )
    fig_tern.show()

    # ─────────────────────────────────────────────────────────
    # ФИГУРА 2: Горизонтальные stacked bars — уверенность
    # Сортировка: сначала по кластеру, внутри — по уверенности ↓
    # ─────────────────────────────────────────────────────────
    sub_sorted = sub.sort_values(
        ['dominant_cluster', 'confidence'],
        ascending=[True, False],
    ).reset_index(drop=True)

    y_labels = sub_sorted[name_col].str[:20].tolist()
    n_pat    = len(sub_sorted)

    fig_conf = go.Figure()

    # Три сегмента: C0, C1, C2
    for c, cp_col in [(0, 'cluster_c0_prob'),
                      (1, 'cluster_c1_prob'),
                      (2, 'cluster_c2_prob')]:
        chex  = CLUSTER_HEX[c]
        cname = CLUSTER_NAMES[c]
        vals  = sub_sorted[cp_col].values * 100

        # Доминирующий сегмент — насыщеннее
        is_dominant = (sub_sorted['dominant_cluster'] == c)
        bar_colors = [
            hex_rgba(chex, 0.65) if dom else hex_rgba(chex, 0.30)
            for dom in is_dominant
        ]
        bar_patterns = [
            '' if dom else '/'
            for dom in is_dominant
        ]

        fig_conf.add_trace(go.Bar(
            y=y_labels,
            x=vals,
            orientation='h',
            name=cname,
            legendgroup=f'c{c}',
            marker=dict(
                color=bar_colors,
                line=dict(color=hex_rgba(chex, 0.70), width=0.5),
                pattern=dict(
                    shape=bar_patterns,
                    size=4,
                    fgcolor=[hex_rgba(chex, 0.25) if not dom else 'rgba(0,0,0,0)'
                             for dom in is_dominant],
                ),
            ),
            hovertemplate=(
                f'<b>%{{y}}</b><br>'
                f'{cname}: %{{x:.1f}}%<extra></extra>'
            ),
        ))

    # Вертикальная линия: порог уверенности 50%
    fig_conf.add_vline(
        x=50, line_dash='dot',
        line_color='rgba(150,150,150,0.55)', line_width=1.2,
        annotation_text='50%', annotation_position='top',
        annotation_font_size=9,
    )

    # Разделители между кластерами
    cluster_sizes = sub_sorted.groupby('dominant_cluster').size().sort_index()
    sep_pos = 0
    for c in sorted(cluster_sizes.index)[:-1]:
        sep_pos += cluster_sizes[c]
        # Горизонтальная линия-разделитель
        fig_conf.add_hline(
            y=n_pat - sep_pos - 0.5,
            line_dash='dot',
            line_color=hex_rgba(CLUSTER_HEX[c], 0.40),
            line_width=1.5,
        )
        # Метка кластера
        cluster_label_y = n_pat - sep_pos + cluster_sizes[c] / 2 - 1
        fig_conf.add_annotation(
            x=103, y=n_pat - sep_pos + cluster_sizes[c] / 2 - 1,
            text=f'← {CLUSTER_NAMES[c]}',
            showarrow=False,
            font=dict(size=8, color=hex_rgba(CLUSTER_HEX[c], 0.70)),
            xanchor='left',
        )

    fig_conf.update_layout(
        **{k: v for k, v in LAYOUT.items() if k != 'margin'},
        barmode='stack',
        height=max(420, n_pat * 14),
        title=dict(
            text='Уверенность в кластерной принадлежности (%, сортировка по кластеру→уверенности)',
            font=dict(size=13),
        ),
        legend=dict(
            orientation='h', x=0.5, xanchor='center',
            y=-0.05, yanchor='top',
            bgcolor='rgba(0,0,0,0)',
            bordercolor='rgba(150,150,150,0.35)', borderwidth=1,
            font=dict(size=10),
        ),
        xaxis=dict(
            title='Вероятность (%)', range=[0, 115],
            zeroline=False, gridcolor='rgba(200,200,200,0.20)',
        ),
        yaxis=dict(zeroline=False, tickfont=dict(size=9)),
        margin=dict(t=60, b=70, l=160, r=60),
    )
    fig_conf.show()

    # ── Сводка по уверенности ────────────────────────────────
    display(HTML(
        '<div style="font-family:Inter,sans-serif;font-size:11px;'
        'padding:9px 14px;border-radius:6px;'
        'border-left:3px solid #888;line-height:2;display:inline-block">'
        '<b>Тернарный симплекс:</b> каждая точка = пациент в пространстве '
        '(P(C0), P(C1), P(C2)) · вершины = чистые кластеры · '
        'центр = равная неопределённость · '
        '<b>размер</b>=возраст · <b>обводка</b>='
        '<span style="color:#00C800">беременность</span>/'
        '<span style="color:#E60000">нет</span><br>'
        '<b>Stacked bars:</b> насыщенный цвет = доминирующий кластер · '
        'штриховка = второстепенный · подписи сверху = %-я уверенность'
        '</div>'
    ))

    conf_stats = sub.groupby('dominant_cluster').agg(
        N=('confidence','count'),
        Уверенность_медиана=('confidence', lambda x: f'{x.median():.2f}'),
        Уверенность_мин=('confidence', lambda x: f'{x.min():.2f}'),
        Энтропия_медиана=('entropy', lambda x: f'{x.median():.3f}'),
    )
    conf_stats.index = [CLUSTER_NAMES.get(i, i) for i in conf_stats.index]
    display(conf_stats)

Тернарный симплекс: каждая точка = пациент в пространстве (P(C0), P(C1), P(C2)) · вершины = чистые кластеры · центр = равная неопределённость · размер =возраст · обводка = беременность / нет Stacked bars: насыщенный цвет = доминирующий кластер · штриховка = второстепенный · подписи сверху = %-я уверенность

,N,Уверенность_медиана,Уверенность_мин,Энтропия_медиана
C0 Standard,21,0.90,0.52,0.324
C1 Poor,23,1.00,0.62,0.000
C2 High,7,0.99,0.88,0.031


## 9. Риски, OHSS, banking

In [17]:
# ══════════════════════════════════════════════════════════════
# Cohort Risk Profile — расширенная версия
#
# Два типа рисков требуют разной визуализации:
#   • OHSS-риски        → бинарные (0/100%) → доля пациентов
#   • Blast/Cancel риски → непрерывные     → distribution + dots
# ══════════════════════════════════════════════════════════════

from IPython.display import display, HTML
import plotly.graph_objects as go
from plotly.subplots import make_subplots

name_col = 'patient_name' if 'patient_name' in df_u.columns else '_key'

# ── Конфиг рисков ─────────────────────────────────────────────
OHSS_RISKS = {
    'OHSS любой':      'ohss_any',
    'OHSS умеренный':  'ohss_moderate',
    'OHSS тяжёлый':    'ohss_severe',
}
CONT_RISKS = {
    'Нет бластоцист':      'p_no_blast',
    'Нет хор.бластоцист':  'p_no_good_blast',
}
# Фильтруем по наличию колонок
OHSS_RISKS = {k: v for k, v in OHSS_RISKS.items() if v in df_u.columns}
CONT_RISKS = {k: v for k, v in CONT_RISKS.items() if v in df_u.columns}

CLUSTER_NAME  = {0: 'C0 Standard', 1: 'C1 Poor', 2: 'C2 High'}
CLUSTER_COLOR = {0: C['blue'], 1: C['red'], 2: C['green']}

RISK_THRESH_LOW  = 10   # %
RISK_THRESH_HIGH = 25   # %

if not OHSS_RISKS and not CONT_RISKS:
    print('⚠️  Нет доступных колонок рисков')
else:
    n_panels = (1 if OHSS_RISKS else 0) + (1 if CONT_RISKS else 0)
    col_widths = []
    if OHSS_RISKS: col_widths.append(0.42)
    if CONT_RISKS: col_widths.append(0.58)

    fig_a = make_subplots(
        rows=1, cols=n_panels,
        column_widths=col_widths,
        subplot_titles=(
            (['Доля пациентов с OHSS-риском']
             if OHSS_RISKS else []) +
            (['Непрерывные риски по пациентам']
             if CONT_RISKS else [])
        ),
        horizontal_spacing=0.14,
    )

    col_ohss = 1
    col_cont = 2 if OHSS_RISKS else 1

    # ─────────────────────────────────────────────────────────
    # ПАНЕЛЬ A: OHSS — доля пациентов в зоне риска
    # ─────────────────────────────────────────────────────────
    if OHSS_RISKS:
        ohss_rows = []
        for label, col in OHSS_RISKS.items():
            vals = df_u[col].dropna() * 100
            n_total   = len(vals)
            # OHSS-колонки бинарны (0 или ≈100%)
            n_at_risk = (vals > 50).sum()
            n_safe    = n_total - n_at_risk
            ohss_rows.append({
                'label':      label,
                'pct_risk':   n_at_risk / n_total * 100,
                'pct_safe':   n_safe    / n_total * 100,
                'n_risk':     n_at_risk,
                'n_total':    n_total,
                'mean_val':   vals.mean(),
            })

        labels_ohss   = [r['label'] for r in ohss_rows]
        pct_risk_vals = [r['pct_risk'] for r in ohss_rows]
        pct_safe_vals = [r['pct_safe'] for r in ohss_rows]

        # Стрипы «в зоне риска»
        bar_color = [
            hex_rgba(C['red'],    0.75) if p > 25 else
            hex_rgba(C['orange'], 0.75) if p > 10 else
            hex_rgba(C['green'],  0.75)
            for p in pct_risk_vals
        ]

        # Сегмент «в зоне риска»
        fig_a.add_trace(go.Bar(
            y=labels_ohss,
            x=pct_risk_vals,
            orientation='h',
            name='В зоне риска',
            legendgroup='ohss',
            marker=dict(
                color=bar_color,
                line=dict(color='rgba(255,255,255,0.6)', width=1),
                pattern=dict(shape='/', size=4,
                             fgcolor='rgba(255,255,255,0.25)'),
            ),
            text=[f'{p:.0f}% (n={r["n_risk"]})' for p, r in
                  zip(pct_risk_vals, ohss_rows)],
            textposition='auto',
            textfont=dict(size=10),
            insidetextanchor='middle',
            hovertemplate=(
                '<b>%{y}</b><br>'
                'В зоне риска: %{x:.1f}%<br>'
                'N пациентов: %{customdata}<extra></extra>'
            ),
            customdata=[r['n_risk'] for r in ohss_rows],
            showlegend=True,
        ), row=1, col=col_ohss)

        # Сегмент «без риска»
        fig_a.add_trace(go.Bar(
            y=labels_ohss,
            x=pct_safe_vals,
            orientation='h',
            name='Без риска',
            legendgroup='ohss',
            marker=dict(
                color='rgba(150,150,150,0.18)',
                line=dict(color='rgba(150,150,150,0.35)', width=1),
            ),
            hovertemplate='<b>%{y}</b><br>Без риска: %{x:.1f}%<extra></extra>',
            showlegend=True,
        ), row=1, col=col_ohss)

        # Пороговые линии
        for xv, lbl, col_line, ann_pos in [
            (25, '25%', 'rgba(220,120,0,0.55)', 'top left'),
            (50, '50%', 'rgba(180,0,0,0.55)',   'bottom left'),
        ]:
            fig_a.add_vline(
                x=xv, line_dash='dot',
                line_color=col_line, line_width=1.4,
                annotation_text=lbl,
                annotation_position=ann_pos,
                annotation_font_size=9,
                col=col_ohss,
            )

        fig_a.update_xaxes(
            title_text='% пациентов когорты',
            range=[0, 115],
            row=1, col=col_ohss,
            zeroline=False,
        )

    # ─────────────────────────────────────────────────────────
    # ПАНЕЛЬ B: Непрерывные риски — Box + Strip (по кластерам)
    # ─────────────────────────────────────────────────────────
    if CONT_RISKS:
        risk_labels_cont = list(CONT_RISKS.keys())
        risk_cols_cont   = list(CONT_RISKS.values())

        # Коробки (box) с квартилями
        for label, col in CONT_RISKS.items():
            vals = df_u[col].dropna() * 100
            fig_a.add_trace(go.Box(
                x=vals,
                y=[label] * len(vals),
                orientation='h',
                name=label,
                legendgroup='box',
                showlegend=False,
                boxmean=True,
                fillcolor=hex_rgba(C['gray'], 0.12),
                line=dict(color=hex_rgba(C['gray'], 0.50), width=1.5),
                marker=dict(size=3, opacity=0),
                hoverinfo='skip',
                width=0.35,
            ), row=1, col=col_cont)

        # Dots по кластерам (jitter по Y)
        clusters = (df_u['dominant_cluster'].dropna().unique().astype(int).tolist()
                    if 'dominant_cluster' in df_u.columns else [-1])

        np.random.seed(42)
        for cluster_id in sorted(clusters):
            if 'dominant_cluster' in df_u.columns:
                d = df_u[df_u['dominant_cluster'] == cluster_id].copy()
            else:
                d = df_u.copy()
            if d.empty:
                continue

            chex   = CLUSTER_COLOR.get(cluster_id, '#888')
            c_name = CLUSTER_NAME.get(cluster_id, str(cluster_id))

            for ki, (label, col) in enumerate(CONT_RISKS.items()):
                sub_d = d[[col, name_col]].dropna().copy()
                if sub_d.empty:
                    continue

                jitter = np.random.uniform(-0.14, 0.14, len(sub_d))
                y_jit  = [label] * len(sub_d)  # Plotly handles category

                # Обводка по исходу
                if 'Preg' in d.columns:
                    border_c = [
                        'rgba(0,200,0,1.0)'   if d.loc[idx, 'Preg'] == 1 else
                        'rgba(220,0,0,1.0)'   if d.loc[idx, 'Preg'] == 0 else
                        'rgba(255,255,255,0.8)'
                        for idx in sub_d.index
                    ]
                    border_w = [
                        2.5 if d.loc[idx, 'Preg'] in [0, 1] else 1.0
                        for idx in sub_d.index
                    ]
                else:
                    border_c = ['rgba(255,255,255,0.8)'] * len(sub_d)
                    border_w = [1.0] * len(sub_d)

                fig_a.add_trace(go.Scatter(
                    x=sub_d[col] * 100,
                    y=[ki + 0.5 + j * 0.18 for j in jitter],
                    mode='markers',
                    name=c_name,
                    legendgroup=f'cluster_{cluster_id}',
                    showlegend=(ki == 0),
                    text=sub_d[name_col],
                    hovertemplate=(
                        f'<b>%{{text}}</b><br>'
                        f'{label}: %{{x:.1f}}%<br>'
                        f'Кластер: {c_name}<extra></extra>'
                    ),
                    marker=dict(
                        size=9,
                        color=hex_rgba(chex, 0.68),
                        line=dict(color=border_c, width=border_w),
                    ),
                ), row=1, col=col_cont)

        # Пороговые линии
        for xv, lbl, col_line, ann_pos in [
            (RISK_THRESH_LOW,  f'{RISK_THRESH_LOW}%',  'rgba(200,160,0,0.55)', 'top right'),
            (RISK_THRESH_HIGH, f'{RISK_THRESH_HIGH}%', 'rgba(180,40,40,0.55)', 'bottom right'),
        ]:
            fig_a.add_vline(
                x=xv, line_dash='dot',
                line_color=col_line, line_width=1.4,
                annotation_text=lbl,
                annotation_position=ann_pos,
                annotation_font_size=9,
                col=col_cont,
            )

        fig_a.update_xaxes(
            title_text='Риск (%)',
            range=[0, max(
                df_u[c].max() * 105
                for c in CONT_RISKS.values() if c in df_u.columns
            )],
            row=1, col=col_cont,
            zeroline=False,
        )
        fig_a.update_yaxes(
            tickvals=list(range(len(CONT_RISKS))),
            ticktext=list(CONT_RISKS.keys()),
            row=1, col=col_cont,
        )

    # ─────────────────────────────────────────────────────────
    # Layout
    # ─────────────────────────────────────────────────────────
    fig_a.update_layout(
        **{k: v for k, v in LAYOUT.items() if k != 'margin'},
        height=430,
        barmode='stack',
        title=dict(
            text='Cohort Risk Profile',
            font=dict(size=14),
        ),
        legend=dict(
            orientation='h',
            x=0.5, xanchor='center',
            y=-0.22, yanchor='top',
            bgcolor='rgba(0,0,0,0)',
            bordercolor='rgba(150,150,150,0.35)',
            borderwidth=1,
            font=dict(size=10),
        ),
        margin=dict(t=70, b=100, l=140, r=30),
    )
    fig_a.update_yaxes(zeroline=False)
    fig_a.show()

    # ─────────────────────────────────────────────────────────
    # Сводная таблица рисков
    # ─────────────────────────────────────────────────────────
    rows_stat = []
    for label, col in {**OHSS_RISKS, **CONT_RISKS}.items():
        if col not in df_u.columns:
            continue
        vals = df_u[col].dropna() * 100
        n_hi = (vals > RISK_THRESH_HIGH).sum()
        rows_stat.append({
            'Риск':          label,
            'Среднее':       f'{vals.mean():.1f}%',
            'SD':            f'±{vals.std():.1f}%',
            'Медиана':       f'{vals.median():.1f}%',
            'P75':           f'{vals.quantile(.75):.1f}%',
            f'N >{RISK_THRESH_HIGH}%': n_hi,
        })

    display(HTML(
        '<div style="font-family:Inter,sans-serif;font-size:11px;'
        'padding:9px 14px;border-radius:6px;'
        'border-left:3px solid #888;line-height:1.8;display:inline-block">'
        '<b>Левая панель:</b> доля пациентов с бинарным OHSS-риском · '
        '<b>Правая панель:</b> непрерывные риски — Box=квартили+среднее, '
        'точки = пациенты по кластерам (● C0 · ◆ C1 · ■ C2), '
        'обводка: <span style="color:#00C800">зелёная</span>=беременность · '
        '<span style="color:#E60000">красная</span>=нет'
        '</div>'
    ))
    display(pd.DataFrame(rows_stat).set_index('Риск'))

Левая панель: доля пациентов с бинарным OHSS-риском · Правая панель: непрерывные риски — Box=квартили+среднее, точки = пациенты по кластерам (● C0 · ◆ C1 · ■ C2), обводка: зелёная =беременность · красная =нет

,Среднее,SD,Медиана,P75,N >25%
Риск,,,,,
OHSS любой,29.4%,±46.0%,0.0%,100.0%,15
OHSS умеренный,9.8%,±30.0%,0.0%,0.0%,5
OHSS тяжёлый,19.6%,±40.1%,0.0%,0.0%,10
Нет бластоцист,9.4%,±15.6%,0.4%,13.6%,7
Нет хор.бластоцист,14.7%,±20.3%,2.8%,24.4%,10


In [18]:
# ══════════════════════════════════════════════════════════════
# Blast Failure Risk Decomposition — расширенная версия
#
# Кодирование измерений:
#   X     = MC P(беременность) %
#   Y     = Риск нет хор.бластоцист %
#   Цвет  = MC − PRAI (RdBu: синий=MC занижает, красный=MC завышает)
#   Размер = возраст
#   Форма = кластер (C0 / C1 / C2)
#   Обводка = исход (зелёная / красная / белая)
#   Стрелки = вектор коррекции MC → PRAI
# ══════════════════════════════════════════════════════════════

from IPython.display import display, HTML

name_col = 'patient_name' if 'patient_name' in df_u.columns else '_key'

needed = ['p_per_transfer', 'PRAI', 'p_no_good_blast']
if not all(c in df_u.columns for c in needed):
    print(f'⚠️  Отсутствуют колонки: {[c for c in needed if c not in df_u.columns]}')
else:
    # ── Данные ───────────────────────────────────────────────
    extra_cols = [c for c in
        ['Age', 'dominant_cluster', 'Preg',
         'p_no_blast', 'p_cancel_risk', 'pred_std', 'ohss_any']
        if c in df_u.columns]

    sub = df_u[needed + extra_cols + [name_col]].dropna(subset=needed).copy()
    sub['x_mc']       = sub['p_per_transfer'] * 100
    sub['x_prai']     = sub['PRAI']           * 100
    sub['y_val']      = sub['p_no_good_blast']* 100
    sub['diff']       = sub['p_per_transfer'] - sub['PRAI']

    # Размер маркера по возрасту
    if 'Age' in sub.columns:
        age_min, age_max = sub['Age'].min(), sub['Age'].max()
        sub['marker_sz'] = 9 + (sub['Age'] - age_min) / (age_max - age_min + 1e-9) * 13
    else:
        sub['marker_sz'] = 12

    CLUSTER_SHAPE = {0: 'circle', 1: 'diamond', 2: 'square'}
    CLUSTER_NAME  = {0: 'C0 Standard', 1: 'C1 Poor', 2: 'C2 High'}

    x_max = sub['x_mc'].max() * 1.12
    y_max = sub['y_val'].max() * 1.22

    fig_b = go.Figure()

    # ─────────────────────────────────────────────────────────
    # 1. Фоновые квадранты
    # ─────────────────────────────────────────────────────────
    quadrants = [
        (50, 100, 0,  20, 'rgba(46,125,50,0.07)',
         '✅ Высокий прогноз<br>Низкий blast risk', 52, 5),
        (0,  50,  0,  20, 'rgba(100,150,200,0.07)',
         '⚠️ Низкий прогноз<br>Низкий blast risk', 2, 5),
        (50, 100, 20, 65, 'rgba(245,166,35,0.08)',
         '⚠️ Высокий прогноз<br>Умеренный blast risk', 52, 25),
        (0,  50,  20, 65, 'rgba(198,40,40,0.10)',
         '🔴 Низкий прогноз<br>Высокий blast risk', 2, 25),
    ]
    for x0, x1, y0, y1, fc, ann, tx, ty in quadrants:
        fig_b.add_shape(
            type='rect',
            x0=x0, x1=min(x1, x_max * 1.05),
            y0=y0, y1=min(y1, y_max * 1.05),
            fillcolor=fc, line_width=0,
        )
        fig_b.add_annotation(
            x=min(tx, x_max * 0.9), y=min(ty, y_max * 0.88),
            text=ann, showarrow=False,
            font=dict(size=9, color='rgba(180,180,180,0.80)'),
            align='left',
        )

    # ─────────────────────────────────────────────────────────
    # 2. Опорные линии
    # ─────────────────────────────────────────────────────────
    for x_v, label, pos in [
        (50, 'MC=50%', 'top right'),
    ]:
        fig_b.add_vline(
            x=x_v, line_dash='dot',
            line_color='rgba(150,150,150,0.50)', line_width=1.5,
            annotation_text=label, annotation_position=pos,
            annotation_font_size=9,
        )
    for y_v, color_v, label, pos in [
        (20, 'rgba(200,160,0,0.60)',  'Умеренный риск 20%',  'bottom right'),
        (40, 'rgba(180,40,40,0.60)',  'Высокий риск 40%',    'bottom right'),
    ]:
        fig_b.add_hline(
            y=y_v, line_dash='dot',
            line_color=color_v, line_width=1.5,
            annotation_text=label, annotation_position=pos,
            annotation_font_size=9,
        )

    # ─────────────────────────────────────────────────────────
    # 3. Векторы коррекции MC → PRAI (стрелки)
    # ─────────────────────────────────────────────────────────
    for _, row in sub.iterrows():
        if abs(row['diff']) < 0.05:  # мелкие расхождения не рисуем
            continue
        arrow_color = ('rgba(198,40,40,0.30)' if row['diff'] > 0
                       else 'rgba(46,125,50,0.30)')
        fig_b.add_annotation(
            x=row['x_prai'], y=row['y_val'],
            ax=row['x_mc'],  ay=row['y_val'],
            xref='x', yref='y', axref='x', ayref='y',
            showarrow=True,
            arrowhead=2, arrowsize=0.9,
            arrowwidth=1.2,
            arrowcolor=arrow_color,
            text='',
        )

    # ─────────────────────────────────────────────────────────
    # 4. Scatter по кластерам
    # ─────────────────────────────────────────────────────────
    clusters = (sub['dominant_cluster'].dropna().unique().astype(int).tolist()
                if 'dominant_cluster' in sub.columns else [-1])

    for cluster_id in sorted(clusters):
        if 'dominant_cluster' in sub.columns:
            d = sub[sub['dominant_cluster'] == cluster_id].copy()
        else:
            d = sub.copy()
            cluster_id = -1

        if d.empty:
            continue

        shape  = CLUSTER_SHAPE.get(cluster_id, 'circle')
        c_name = CLUSTER_NAME.get(cluster_id, 'Unknown')

        # Обводка по исходу
        border_colors, border_widths = [], []
        for _, row in d.iterrows():
            preg = row.get('Preg', np.nan)
            if preg == 1:
                border_colors.append('rgba(0,200,0,1.0)')
                border_widths.append(3)
            elif preg == 0:
                border_colors.append('rgba(230,0,0,1.0)')
                border_widths.append(3)
            else:
                border_colors.append('rgba(255,255,255,0.85)')
                border_widths.append(1.2)

        # Hover
        hover_lines = [
            '<b>%{text}</b>',
            'MC P(беременность): %{x:.1f}%',
            'Риск нет хор.бластоцист: %{y:.1f}%',
            'MC − PRAI: %{customdata[0]:+.3f}',
        ]
        if 'Age'          in d.columns: hover_lines.append('Возраст: %{customdata[1]:.0f}')
        if 'pred_std'     in d.columns: hover_lines.append('SD моделей: %{customdata[2]:.3f}')
        if 'p_cancel_risk'in d.columns: hover_lines.append('Риск отмены: %{customdata[3]:.1%}')
        if 'p_no_blast'   in d.columns: hover_lines.append('Риск 0 бластоцист: %{customdata[4]:.1%}')
        hover_lines.append(f'Кластер: {c_name}<extra></extra>')

        customdata = np.column_stack([
            d['diff'].fillna(0),
            d.get('Age',          pd.Series(0, index=d.index)).fillna(0),
            d.get('pred_std',     pd.Series(0, index=d.index)).fillna(0),
            d.get('p_cancel_risk',pd.Series(0, index=d.index)).fillna(0),
            d.get('p_no_blast',   pd.Series(0, index=d.index)).fillna(0),
        ])

        fig_b.add_trace(go.Scatter(
            x=d['x_mc'],
            y=d['y_val'],
            mode='markers',
            name=c_name,
            text=d[name_col],
            customdata=customdata,
            hovertemplate='<br>'.join(hover_lines),
            marker=dict(
                symbol=shape,
                size=d['marker_sz'].values,
                color=d['diff'].values,
                colorscale='RdBu_r',
                cmin=-0.25, cmax=0.25,
                colorbar=dict(
                    title=dict(text='MC−PRAI', side='right'),
                    thickness=14, len=0.55,
                    x=1.02, y=0.5,
                    tickformat='+.2f',
                    tickfont=dict(size=9),
                ),
                opacity=0.82,
                line=dict(color=border_colors, width=border_widths),
                showscale=(cluster_id == clusters[0]),
            ),
        ))

    # ─────────────────────────────────────────────────────────
    # 5. Layout
    # ─────────────────────────────────────────────────────────
    fig_b.update_layout(
        **{k: v for k, v in LAYOUT.items() if k != 'margin'},
        height=560,
        title=dict(
            text='Blast Failure Risk Decomposition  ·  MC vs PRAI',
            font=dict(size=14),
        ),
        legend=dict(
            x=1.10, y=0.20,
            bgcolor='rgba(0,0,0,0)',
            bordercolor='rgba(150,150,150,0.4)',
            borderwidth=1,
            font=dict(size=10),
        ),
        xaxis=dict(
            title='MC P(беременность) %',
            range=[0, x_max],
            zeroline=False,
            gridcolor='rgba(200,200,200,0.20)',
        ),
        yaxis=dict(
            title='Риск нет хор. бластоцист (%)',
            range=[0, y_max],
            zeroline=False,
            gridcolor='rgba(200,200,200,0.20)',
        ),
        margin=dict(t=55, b=50, l=60, r=160),
    )
    fig_b.show()

    # ─────────────────────────────────────────────────────────
    # 6. Легенда кодирования
    # ─────────────────────────────────────────────────────────
    display(HTML(
        '<div style="font-family:Inter,sans-serif;font-size:11px;'
        'padding:10px 14px;border-radius:6px;'
        'border-left:3px solid #888;line-height:2;display:inline-block">'
        '<b>Кодирование:</b>&nbsp;&nbsp;'
        '<b>X</b> = MC P(беременность) &nbsp;|&nbsp;'
        '<b>Y</b> = Риск нет хор.бластоцист &nbsp;|&nbsp;'
        '<b>Цвет</b> = MC−PRAI '
        '(<span style="color:#C62828">красный</span>=MC завышает · '
        '<span style="color:#1565C0">синий</span>=MC занижает) &nbsp;|&nbsp;'
        '<b>Размер</b> = возраст &nbsp;|&nbsp;'
        '<b>Форма</b>: ● C0 · ◆ C1 · ■ C2 &nbsp;|&nbsp;'
        '<b>Обводка</b>: '
        '<span style="color:#00C800">━ беременность</span> · '
        '<span style="color:#E60000">━ нет</span> · '
        'белая = нет данных &nbsp;|&nbsp;'
        '<b>Стрелки</b> = вектор коррекции MC→PRAI'
        '</div>'
    ))

Кодирование:    X = MC P(беременность)  |  Y = Риск нет хор.бластоцист  |  Цвет = MC−PRAI ( красный =MC завышает · синий =MC занижает)  |  Размер = возраст  |  Форма : ● C0 · ◆ C1 · ■ C2  |  Обводка : ━ беременность · ━ нет · белая = нет данных  |  Стрелки = вектор коррекции MC→PRAI

In [19]:
# ══════════════════════════════════════════════════════════════
# Conflict Zones & Decision Surface  —  расширенная версия
#
# Кодирование измерений:
#   X     = разброс между моделями (pred_std, пп) → неопределённость
#   Y     = риск отсутствия хор.бластоцист (p_no_good_blast, %)
#   Цвет  = зона расхождения MC vs PRAI (Aligned / Moderate / Severe)
#   Размер = возраст пациентки (старше → больше маркер)
#   Форма = доминирующий кластер (C0 / C1 / C2)
#   Обводка = известный исход (✅ беременность / ❌ нет / — нет данных)
# ══════════════════════════════════════════════════════════════

from IPython.display import display, HTML

# ── Проверка и подготовка df_tmp ──────────────────────────────
df_tmp = df_u.copy()

pred_cols_all = [c for c in
    ['p_per_transfer','p_kat_raw','p_nvsa','p_csdi','DIGITAL TWIN','PRAI']
    if c in df_tmp.columns]

if 'pred_std' not in df_tmp.columns:
    df_tmp['pred_std']  = df_tmp[pred_cols_all].std(axis=1)
if 'pred_mean' not in df_tmp.columns:
    df_tmp['pred_mean'] = df_tmp[pred_cols_all].mean(axis=1)

df_tmp['mc_prai_diff'] = (
    df_tmp['p_per_transfer'] - df_tmp['PRAI']
    if ('p_per_transfer' in df_tmp.columns and 'PRAI' in df_tmp.columns)
    else pd.Series(0, index=df_tmp.index)
)

# ── Конфигурация ──────────────────────────────────────────────
name_col = 'patient_name' if 'patient_name' in df_tmp.columns else '_key'

ZONE_CFG = {
    'Aligned':  dict(color='#2E7D32', label='Aligned  |MC−PRAI|<5пп',  thresh=0.05),
    'Moderate': dict(color='#F9A825', label='Moderate  5–15пп',         thresh=0.15),
    'Severe':   dict(color='#C62828', label='Severe  >15пп',            thresh=99),
}

CLUSTER_SHAPE = {0: 'circle', 1: 'diamond', 2: 'square'}
CLUSTER_NAME  = {0: 'C0 Standard', 1: 'C1 Poor', 2: 'C2 High'}

def zone_label(x):
    ax = abs(x)
    if ax < 0.05:  return 'Aligned'
    if ax < 0.15:  return 'Moderate'
    return 'Severe'

# ── Подготовка данных ─────────────────────────────────────────
needed = ['pred_std', 'p_no_good_blast', 'mc_prai_diff',
          'Age', 'dominant_cluster']
sub = df_tmp[needed + [name_col]
             + (['Preg'] if 'Preg' in df_tmp.columns else [])
             + (['p_cancel_risk'] if 'p_cancel_risk' in df_tmp.columns else [])
             + (['pred_mean'] if 'pred_mean' in df_tmp.columns else [])
             ].dropna(subset=needed).copy()

sub['zone']      = sub['mc_prai_diff'].apply(zone_label)
sub['x_val']     = sub['pred_std']         * 100   # пп
sub['y_val']     = sub['p_no_good_blast']  * 100   # %
sub['marker_sz'] = 8 + (sub['Age'] - sub['Age'].min()) / (
                        sub['Age'].max() - sub['Age'].min() + 1e-9) * 14

if not sub.empty:
    # ── Границы осей ─────────────────────────────────────────
    x_max = sub['x_val'].max() * 1.18
    y_max = sub['y_val'].max() * 1.20

    fig_c = go.Figure()

    # ─────────────────────────────────────────────────────────
    # 1. Decision surface — фоновые зоны
    # ─────────────────────────────────────────────────────────
    zones_bg = [
        # (x0, x1, y0, y1, color, label, text_x, text_y)
        (0,    15, 0,   20, 'rgba(46,125,50,0.07)',
         '✅ Низкий риск<br>Высокая согласованность', 4, 5),
        (0,    15, 20,  60, 'rgba(245,166,35,0.08)',
         '⚠️ Умеренный риск<br>Хорошая согласованность', 4, 40),
        (15,   35, 0,   35, 'rgba(245,166,35,0.09)',
         '⚠️ Умеренный риск<br>Низкая согласованность', 22, 8),
        (15,   35, 35,  75, 'rgba(198,40,40,0.11)',
         '🔴 Высокий риск<br>Низкая согласованность', 22, 52),
    ]
    for x0, x1, y0, y1, fc, ann_text, tx, ty in zones_bg:
        if x1 > x_max * 0.3 or y1 > y_max * 0.3:  # показываем только релевантные
            fig_c.add_shape(type='rect',
                x0=x0, x1=min(x1, x_max * 1.1),
                y0=y0, y1=min(y1, y_max * 1.1),
                fillcolor=fc, line_width=0)
            fig_c.add_annotation(
                x=min(tx, x_max * 0.85),
                y=min(ty, y_max * 0.90),
                text=ann_text,
                showarrow=False,
                font=dict(size=9, color='rgba(180,180,180,0.85)'),
                align='left',
            )

    # ─────────────────────────────────────────────────────────
    # 2. Опорные линии — пороги принятия решений
    # ─────────────────────────────────────────────────────────
    fig_c.add_vline(x=15, line_dash='dot', line_color='rgba(120,120,120,0.50)',
                    line_width=1.5,
                    annotation_text='SD=15пп',
                    annotation_position='top right',
                    annotation_font_size=9)
    fig_c.add_hline(y=20, line_dash='dot', line_color='rgba(120,120,120,0.50)',
                    line_width=1.5,
                    annotation_text='Blast failure=20%',
                    annotation_position='bottom right',
                    annotation_font_size=9)

    # ─────────────────────────────────────────────────────────
    # 3. Scatter по зонам × кластерам × исходам
    # ─────────────────────────────────────────────────────────
    for zone_name, zcfg in ZONE_CFG.items():
        for cluster_id, shape in CLUSTER_SHAPE.items():
            mask = (sub['zone'] == zone_name) & (sub['dominant_cluster'] == cluster_id)
            d = sub[mask]
            if d.empty:
                continue

            # Обводка маркера по исходу
            border_colors, border_widths = [], []
            for _, row in d.iterrows():
                preg = row.get('Preg', np.nan)
                if preg == 1:
                    border_colors.append('rgba(0,180,0,1.0)')
                    border_widths.append(3)
                elif preg == 0:
                    border_colors.append('rgba(220,0,0,1.0)')
                    border_widths.append(3)
                else:
                    border_colors.append('rgba(255,255,255,0.90)')
                    border_widths.append(1.5)

            # Hover-текст
            hover_parts = ['<b>%{text}</b>']
            hover_parts.append('SD моделей: %{x:.1f}пп')
            hover_parts.append('Риск нет бластоцист: %{y:.1f}%')
            if 'pred_mean' in d.columns:
                hover_parts.append('Ср. прогноз: %{customdata[0]:.1%}')
            if 'p_cancel_risk' in d.columns:
                hover_parts.append('Риск отмены: %{customdata[1]:.1%}')
            hover_parts.append('Возраст: %{customdata[2]:.0f}')
            hover_parts.append(f'Зона: {zone_name}')
            hover_parts.append(f'Кластер: {CLUSTER_NAME[cluster_id]}')

            customdata = np.column_stack([
                d.get('pred_mean',   pd.Series(np.nan, index=d.index)).fillna(0),
                d.get('p_cancel_risk', pd.Series(np.nan, index=d.index)).fillna(0),
                d['Age'],
            ])

            fig_c.add_trace(go.Scatter(
                x=d['x_val'],
                y=d['y_val'],
                mode='markers',
                name=f'{zone_name} · {CLUSTER_NAME[cluster_id]}',
                legendgroup=zone_name,
                legendgrouptitle=dict(text=zone_name) if cluster_id == 0 else None,
                text=d[name_col],
                customdata=customdata,
                hovertemplate='<br>'.join(hover_parts) + '<extra></extra>',
                marker=dict(
                    symbol=shape,
                    size=d['marker_sz'].values,
                    color=hex_rgba(zcfg['color'], 0.68),
                    line=dict(
                        color=border_colors,
                        width=border_widths,
                    ),
                ),
            ))

    # ─────────────────────────────────────────────────────────
    # 4. Опорные линии — пороги принятия решений
    # ─────────────────────────────────────────────────────────
    fig_c.update_layout(
        **{k: v for k, v in LAYOUT.items() if k != 'margin'},
        height=580,
        title=dict(
            text='Conflict Zones & Decision Surface — неопределённость vs эмбриологический риск',
            font=dict(size=14),
        ),
        legend=dict(
            groupclick='toggleitem',
            x=1.01, y=1,
            bgcolor='rgba(0,0,0,0)',
            bordercolor='#ddd', borderwidth=1,
            font=dict(size=10),
            tracegroupgap=6,
        ),
        xaxis=dict(
            title='Разброс между моделями (SD, пп)',
            range=[0, x_max],
            zeroline=False,
            gridcolor='rgba(200,200,200,0.25)',
        ),
        yaxis=dict(
            title='Риск отсутствия хор. бластоцист (%)',
            range=[0, y_max],
            zeroline=False,
            gridcolor='rgba(200,200,200,0.25)',
        ),
        margin=dict(t=55, b=50, l=60, r=200),
    )
    fig_c.show()

    # ─────────────────────────────────────────────────────────
    # 6. Легенда кодирования + сводная статистика по зонам
    # ─────────────────────────────────────────────────────────
    display(HTML(
        '<div style="font-family:Inter,sans-serif;font-size:11px;'
        'padding:10px 14px;border-radius:6px;'
        'border-left:3px solid #888;line-height:2;display:inline-block">'
        '<b>Кодирование измерений:</b><br>'
        '&nbsp;<b>X</b> = SD между моделями пайплайна (пп) — чем правее, тем выше неопределённость<br>'
        '&nbsp;<b>Y</b> = Риск нет хор.бластоцист (%) — чем выше, тем хуже эмбриология<br>'
        '&nbsp;<b>Цвет</b> = Зона расхождения |MC−PRAI|: '
        '<span style="color:#2E7D32"><b>Aligned</b></span> &lt;5пп · '
        '<span style="color:#F9A825"><b>Moderate</b></span> 5–15пп · '
        '<span style="color:#C62828"><b>Severe</b></span> &gt;15пп<br>'
        '&nbsp;<b>Размер</b> = Возраст (больше маркер → старше пациентка)<br>'
        '&nbsp;<b>Форма</b> = Кластер: ● C0 Standard · ◆ C1 Poor · ■ C2 High<br>'
        '&nbsp;<b>Обводка</b>: '
        '<span style="color:#00B400"><b>─── зелёная</b></span> = беременность ✅ · '
        '<span style="color:#DC0000"><b>─── красная</b></span> = нет беременности ❌ · '
        'белая = нет данных'
        '</div>'
    ))

    # Сводка по зонам
    zone_stat = (sub.groupby('zone')
                   .agg(N=('x_val','count'),
                        SD_median=('x_val','median'),
                        BlastRisk_median=('y_val','median'),
                        Age_median=('Age','median'))
                   .round(1))
    if 'Preg' in sub.columns:
        zone_preg = sub.groupby('zone')['Preg'].mean().round(3).rename('Preg_rate')
        zone_stat = zone_stat.join(zone_preg)
    zone_stat = zone_stat.reindex(['Aligned','Moderate','Severe']).dropna(how='all')
    print('\n── Статистика по зонам ──')
    display(zone_stat)

else:
    print('⚠️  Недостаточно данных для построения графика')


── Статистика по зонам ──


Кодирование измерений:   X = SD между моделями пайплайна (пп) — чем правее, тем выше неопределённость   Y = Риск нет хор.бластоцист (%) — чем выше, тем хуже эмбриология   Цвет = Зона расхождения |MC−PRAI|: Aligned <5пп · Moderate 5–15пп · Severe >15пп   Размер = Возраст (больше маркер → старше пациентка)   Форма = Кластер: ● C0 Standard · ◆ C1 Poor · ■ C2 High   Обводка : ─── зелёная = беременность ✅ · ─── красная = нет беременности ❌ · белая = нет данных

,N,SD_median,BlastRisk_median,Age_median,Preg_rate
zone,,,,,
Aligned,1,6.3,0.0,41.0,1.000
Moderate,9,11.7,1.0,33.0,0.600
Severe,40,12.4,5.9,31.0,0.333


In [20]:
# ── Banking: ожидаемые эуплоиды (улучшенный визуал) ─────────────────────────────
if 'banking_expected_euploid' in df_u.columns and 'Good Bl' in df_u.columns:
    # Определяем имя колонки пациента
    name_col = 'patient_name' if 'patient_name' in df_u.columns else '_key'
    
    # Возрастные группы
    df_u['age_group'] = pd.cut(df_u['Age'],
                               bins=[0, 30, 35, 38, 41, 100],
                               labels=['<30', '30-35', '35-38', '38-41', '>41'])
    
    # Подготовка данных
    sub_b2 = df_u[['banking_expected_euploid', 'Good Bl', 'Age', name_col]].dropna(
        subset=['banking_expected_euploid', 'Good Bl'])
    
    banking_by_age = df_u.groupby('age_group', observed=True)['banking_expected_euploid'].median()
    
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=['Ожидаемые эуплоиды vs Хор.кач.бластоцисты',
                                        'Ожидаемые эуплоиды по возрасту'],
                        horizontal_spacing=0.12)
    
    # --- Левый график: scatter с цветом по возрасту ---
    fig.add_trace(go.Scatter(
        x=sub_b2['Good Bl'],
        y=sub_b2['banking_expected_euploid'],
        mode='markers',
        text=sub_b2[name_col],
        marker=dict(
            size=10,
            color=sub_b2['Age'],
            colorscale='RdYlGn_r',
            showscale=True,
            colorbar=dict(
                title='Возраст',
                x=0.45,
                y=0.5,
                len=0.6,
                thickness=15,
                bgcolor='rgba(255,255,255,0.8)'
            ),
            opacity=0.75,
            line=dict(width=0.5, color='white')
        ),
        hovertemplate=(
            '<b>%{text}</b><br>'
            'Хор.бласты: %{x}<br>'
            'Ожид.эуплоиды: %{y:.1f}<br>'
            'Возраст: %{marker.color:.0f}<extra></extra>'
        ),
        showlegend=False,
    ), row=1, col=1)
    
    # Добавляем линию тренда (линейную регрессию)
    from scipy import stats
    import numpy as np
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        sub_b2['Good Bl'], sub_b2['banking_expected_euploid']
    )
    x_line = np.array([sub_b2['Good Bl'].min(), sub_b2['Good Bl'].max()])
    y_line = slope * x_line + intercept
    fig.add_trace(go.Scatter(
        x=x_line, y=y_line,
        mode='lines',
        line=dict(color='black', width=1.5, dash='dash'),
        name=f'тренд (R²={r_value**2:.2f})',
        showlegend=True,
        legendgroup='trend',
        hovertemplate='Линия тренда<extra></extra>'
    ), row=1, col=1)
    
    # --- Правый график: bar plot с медианами (полупрозрачные цвета) ---
    age_colors = [
        'rgba(46,125,50,0.75)',   # <30 зелёный
        'rgba(0,128,128,0.75)',   # 30-35 бирюзовый
        'rgba(249,168,37,0.75)',  # 35-38 оранжевый
        'rgba(198,40,40,0.75)',   # 38-41 красный
        'rgba(183,28,28,0.75)'    # >41 тёмно-красный
    ]
    fig.add_trace(go.Bar(
        x=banking_by_age.index.astype(str),
        y=banking_by_age.values,
        marker_color=age_colors[:len(banking_by_age)],
        marker_line_color='white',
        marker_line_width=1,
        text=[f'{v:.1f}' for v in banking_by_age.values],
        textposition='outside',
        textfont=dict(size=11, color='black'),
        showlegend=False,
        hovertemplate='Возраст %{x}: медиана=%{y:.2f}<extra></extra>',
    ), row=1, col=2)
    
    # Настройка осей
    fig.update_xaxes(title_text='Хор.кач. бластоцист (реально)', row=1, col=1, gridcolor='lightgray')
    fig.update_yaxes(title_text='Ожидаемых эуплоидных', row=1, col=1, gridcolor='lightgray')
    fig.update_xaxes(title_text='Возрастная группа', row=1, col=2, gridcolor='lightgray')
    fig.update_yaxes(title_text='Медиана ожид. эуплоидных', row=1, col=2, gridcolor='lightgray')
    
    # Объединяем LAYOUT с дополнительными настройками, чтобы избежать конфликта
    layout_banking = {**LAYOUT,
                      'height': 450,
                      'title': 'Модуль банкинга',
                      'plot_bgcolor': 'rgba(245,245,245,0.6)',
                      'paper_bgcolor': 'white',
                      'margin': dict(l=60, r=100, t=80, b=50),
                      'legend': dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
                     }
    fig.update_layout(**layout_banking)
    
    fig.show()
else:
    print("Не хватает колонок 'banking_expected_euploid' или 'Good Bl'")

## 10. Точность по возрастным группам + коэффициенты конверсии

In [21]:
# ══════════════════════════════════════════════════════════════
# Прогноз vs Реальность по возрастным группам — с SD
# ══════════════════════════════════════════════════════════════

df_u['age_group'] = pd.cut(
    df_u['Age'],
    bins=[0, 30, 35, 38, 41, 100],
    labels=['<30', '30–35', '35–38', '38–41', '>41'],
)

age_pairs = [
    ('med_okk',    'OCC',     'p025_okk',    'p975_okk',    'ОКК (ооциты)'),
    ('med_mii',    'MII',     None,           None,           'MII (зрелые)'),
    ('med_blasts', 'Bl',      'p025_blasts', 'p975_blasts',  'Бластоцисты'),
    ('med_good',   'Good Bl', 'p025_good',   'p975_good',    'Бластоцисты хор.кач.'),
]
age_pairs = [
    (a, b, lo, hi, t) for a, b, lo, hi, t in age_pairs
    if a in df_u.columns and b in df_u.columns
]

AGE_COLORS = {
    '<30':    '#1565C0',
    '30–35':  '#2E7D32',
    '35–38':  '#F57F17',
    '38–41':  '#E65100',
    '>41':    '#B71C1C',
}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[t for *_, t in age_pairs],
    vertical_spacing=0.16,
    horizontal_spacing=0.10,
)

for idx, (col_dt, col_opu, col_lo, col_hi, title) in enumerate(age_pairs):
    row_i = idx // 2 + 1
    col_i = idx  % 2 + 1

    # ── Агрегация ──────────────────────────────────────────────
    cols_needed = ['age_group', col_dt, col_opu]
    if col_lo and col_lo in df_u.columns:
        cols_needed += [col_lo, col_hi]
    sub = df_u[cols_needed].dropna(subset=['age_group', col_dt, col_opu]).copy()
    sub = sub.astype({col_dt: float, col_opu: float})

    grp = sub.groupby('age_group', observed=True)

    # медианы
    pred_med  = grp[col_dt].median()
    real_med  = grp[col_opu].median()
    # SD реальных значений
    real_sd   = grp[col_opu].std().fillna(0)
    # n на группу
    n_per_grp = grp[col_opu].count()

    # SD прогноза: из p025/p975 если есть, иначе std MC
    if col_lo and col_lo in sub.columns:
        sub['_mc_sd'] = ((sub[col_hi].astype(float) - sub[col_lo].astype(float))
                         / (2 * 1.96)).clip(lower=0)
        pred_sd = grp['_mc_sd'].median().fillna(0)
    else:
        pred_sd = grp[col_dt].std().fillna(0)

    # Дельта: медиана(pred) − медиана(real)
    delta = pred_med - real_med

    age_labels = pred_med.index.astype(str)
    x_pos      = np.arange(len(age_labels))

    # ── Scatter-lines: соединяем прогноз и реальность ─────────
    for xi, ag in enumerate(age_labels):
        y_pred_v = pred_med.get(ag, np.nan)
        y_real_v = real_med.get(ag, np.nan)
        if np.isnan(y_pred_v) or np.isnan(y_real_v):
            continue
        age_color = AGE_COLORS.get(ag, '#888')
        fig.add_trace(go.Scatter(
            x=[ag, ag], y=[y_pred_v, y_real_v],
            mode='lines',
            line=dict(color=hex_rgba(age_color, 0.30), width=2),
            showlegend=False, hoverinfo='skip',
        ), row=row_i, col=col_i)

    # ── Бары: MC прогноз с SD ─────────────────────────────────
    fig.add_trace(go.Bar(
        x=age_labels,
        y=pred_med.values,
        name='MC прогноз' if idx == 0 else 'MC прогноз',
        legendgroup='pred',
        showlegend=(idx == 0),
        marker=dict(
            color=[hex_rgba(AGE_COLORS.get(ag, '#888'), 0.40)
                   for ag in age_labels],
            line=dict(
                color=[hex_rgba(AGE_COLORS.get(ag, '#888'), 0.85)
                       for ag in age_labels],
                width=1.8,
            ),
            pattern=dict(shape='/', size=4,
                         fgcolor=[hex_rgba(AGE_COLORS.get(ag, '#888'), 0.55)
                                  for ag in age_labels]),
        ),
        error_y=dict(
            type='data',
            array=pred_sd.reindex(age_labels).fillna(0).values,
            color='rgba(80,80,80,0.50)',
            thickness=1.5, width=6,
        ),
        hovertemplate=(
            '<b>%{x}</b><br>'
            'MC медиана: %{y:.2f}<br>'
            'MC SD: ±%{error_y.array:.2f}<extra></extra>'
        ),
        offsetgroup='pred',
    ), row=row_i, col=col_i)

    # ── Бары: реальные значения с SD ──────────────────────────
    fig.add_trace(go.Bar(
        x=age_labels,
        y=real_med.values,
        name='Реально' if idx == 0 else 'Реально',
        legendgroup='real',
        showlegend=(idx == 0),
        marker=dict(
            color=[hex_rgba(AGE_COLORS.get(ag, '#888'), 0.72)
                   for ag in age_labels],
            line=dict(
                color=[hex_rgba(AGE_COLORS.get(ag, '#888'), 0.95)
                       for ag in age_labels],
                width=1.8,
            ),
        ),
        error_y=dict(
            type='data',
            array=real_sd.reindex(age_labels).fillna(0).values,
            color='rgba(50,50,50,0.55)',
            thickness=2.0, width=6,
        ),
        hovertemplate=(
            '<b>%{x}</b><br>'
            'Реал. медиана: %{y:.2f}<br>'
            'Реал. SD: ±%{error_y.array:.2f}<extra></extra>'
        ),
        offsetgroup='real',
    ), row=row_i, col=col_i)

    # ── Аннотации: Δ (pred − real) над каждой группой ─────────
    y_max = max(
        (pred_med + pred_sd).max(),
        (real_med + real_sd).max()
    ) if len(pred_med) > 0 else 1

    for ag in age_labels:
        d = delta.get(ag, np.nan)
        n = int(n_per_grp.get(ag, 0))
        if np.isnan(d) or n == 0:
            continue
        color_ann = C['red'] if d > 0 else C['green']
        sign      = '+' if d > 0 else ''
        fig.add_annotation(
            x=ag,
            y=y_max * 1.10,
            text=f'<b>{sign}{d:.1f}</b><br><span style="font-size:9px">n={n}</span>',
            showarrow=False,
            font=dict(size=9, color=color_ann),
            xref=f'x{idx+1}' if idx > 0 else 'x',
            yref=f'y{idx+1}' if idx > 0 else 'y',
        )

    fig.update_yaxes(
        title_text='Медиана',
        range=[0, y_max * 1.22],
        row=row_i, col=col_i,
        zeroline=False,
    )
    fig.update_xaxes(
        tickfont=dict(size=10),
        row=row_i, col=col_i,
    )

# ── Layout ────────────────────────────────────────────────────
fig.update_layout(
    **LAYOUT,
    height=600,
    barmode='group',
    title=dict(
        text='Прогноз MC vs Реальность (медиана ± SD)',
        font=dict(size=15),
    ),
    legend=dict(
        orientation='h',
        x=0.5, xanchor='center',
        y=1.04, yanchor='bottom',
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='#ddd', borderwidth=1,
        font=dict(size=12),
    ),
    bargap=0.22,
    bargroupgap=0.06,
)
fig.show()

# ── Легенда цветов и обозначений ──────────────────────────────
from IPython.display import display, HTML
display(HTML(
    '<div style="font-family:Inter,sans-serif;font-size:11px;'
    'padding:9px 14px;border-radius:6px;'
    'border-left:3px solid #ccc;display:inline-block;margin-top:6px;'
    'line-height:1.8">'
    '<b>Штриховка</b> = MC прогноз &nbsp;|&nbsp; '
    '<b>Сплошная заливка</b> = Реальные данные &nbsp;|&nbsp; '
    '<b>Error bars</b> = ±SD &nbsp;|&nbsp; '
    '<b>Δ над барами</b> = MC медиана − Реальная медиана '
    '(<span style="color:#C0392B">красный</span> = переоценка, '
    '<span style="color:#1E8449">зелёный</span> = недооценка)'
    '</div>'
))

Штриховка = MC прогноз  |  Сплошная заливка = Реальные данные  |  Error bars = ±SD  |  Δ над барами = MC медиана − Реальная медиана ( красный = переоценка, зелёный = недооценка)

In [22]:
# ══════════════════════════════════════════════════════════════
# Коэффициенты конверсии: реальные vs MC-прогноз (медиана ± SD)
# ══════════════════════════════════════════════════════════════

# ── Зависимости (проверка) ────────────────────────────────────
# Требуются из предыдущих ячеек:
#   df_u, C, LAYOUT, hex_rgba, np, pd, go, make_subplots

# ── Данные ───────────────────────────────────────────────────
_real_cols = ['OCC', 'MII', '2pN', 'Bl', 'Good Bl']
_mc_cols   = ['med_okk', 'med_mii', 'med_pn2', 'med_blasts', 'med_good']
_ci_cols   = ['p025_okk', 'p975_okk',
              'p025_blasts', 'p975_blasts',
              'p025_good', 'p975_good']

_avail_real = [c for c in _real_cols if c in df_u.columns]
_avail_mc   = [c for c in _mc_cols   if c in df_u.columns]
_avail_ci   = [c for c in _ci_cols   if c in df_u.columns]

df_rates = (df_u[_avail_real]
            .dropna()
            .astype(float)
            .pipe(lambda d: d[d['OCC'] > 0]))

df_mc = (df_u[_avail_mc + _avail_ci]
         .dropna(subset=_avail_mc)
         .astype({c: float for c in _avail_mc})
         .pipe(lambda d: d[d['med_okk'] > 0]))

if len(df_rates) < 2 or len(df_mc) < 2:
    print(f'⚠️  Недостаточно данных: real={len(df_rates)}, mc={len(df_mc)}')
else:
    # ── Вычисляем распределения коэффициентов конверсии ──────
    # Реальные: посчитываем ratio для каждого пациента → median + SD
    _real_ratios = {
        'OCC→MII':   df_rates['MII']    / df_rates['OCC'],
        'MII→2PN':   df_rates['2pN']    / df_rates['MII'].replace(0, np.nan),
        'OCC→Blast': df_rates['Bl']     / df_rates['OCC'],
        'Blast→HQ':  df_rates['Good Bl']/ df_rates['Bl'].replace(0, np.nan),
    }
    # MC: аналогично по пациентам
    _mc_ratios = {
        'OCC→MII':   df_mc['med_mii']    / df_mc['med_okk'],
        'MII→2PN':   df_mc['med_pn2']    / df_mc['med_mii'].replace(0, np.nan),
        'OCC→Blast': df_mc['med_blasts'] / df_mc['med_okk'],
        'Blast→HQ':  df_mc['med_good']   / df_mc['med_blasts'].replace(0, np.nan),
    }

    rate_labels = list(_real_ratios.keys())

    real_med = [_real_ratios[k].median() * 100 for k in rate_labels]
    real_sd  = [_real_ratios[k].std()    * 100 for k in rate_labels]
    mc_med   = [_mc_ratios[k].median()   * 100 for k in rate_labels]
    mc_sd    = [_mc_ratios[k].std()      * 100 for k in rate_labels]
    n_real   = [_real_ratios[k].dropna().count() for k in rate_labels]

    delta    = [m - r for m, r in zip(mc_med, real_med)]

    # Цвет стадии
    stage_colors = {
        'OCC→MII':   C['blue'],
        'MII→2PN':   C['teal'],
        'OCC→Blast': C['orange'],
        'Blast→HQ':  C['red'],
    }

    # ── Фигура ───────────────────────────────────────────────
    fig = go.Figure()

    # 1. Связующие линии (прогноз → реальность)
    for xi, label in enumerate(rate_labels):
        chex = stage_colors.get(label, '#888')
        fig.add_shape(
            type='line',
            x0=xi - 0.22, x1=xi - 0.22,
            y0=real_med[xi], y1=mc_med[xi],
            line=dict(color=hex_rgba(chex, 0.30), width=1.5, dash='dot'),
        )

    # 2. MC прогноз — штриховка + низкая прозрачность + SD
    fig.add_trace(go.Bar(
        name='MC прогноз',
        x=rate_labels,
        y=mc_med,
        offsetgroup='mc',
        marker=dict(
            color=[hex_rgba(stage_colors.get(l, '#888'), 0.35)
                   for l in rate_labels],
            line=dict(
                color=[hex_rgba(stage_colors.get(l, '#888'), 0.80)
                       for l in rate_labels],
                width=1.8,
            ),
            pattern=dict(
                shape='/',
                size=5,
                fgcolor=[hex_rgba(stage_colors.get(l, '#888'), 0.55)
                         for l in rate_labels],
            ),
        ),
        error_y=dict(
            type='data',
            array=mc_sd,
            color='rgba(80,80,80,0.45)',
            thickness=1.5,
            width=7,
        ),
        hovertemplate=(
            '<b>%{x}</b><br>'
            'MC медиана: %{y:.1f}%<br>'
            'MC SD: ±%{error_y.array:.1f}%<extra></extra>'
        ),
    ))

    # 3. Реальные — насыщенная заливка + SD
    fig.add_trace(go.Bar(
        name='Реально',
        x=rate_labels,
        y=real_med,
        offsetgroup='real',
        marker=dict(
            color=[hex_rgba(stage_colors.get(l, '#888'), 0.72)
                   for l in rate_labels],
            line=dict(
                color=[hex_rgba(stage_colors.get(l, '#888'), 0.95)
                       for l in rate_labels],
                width=1.8,
            ),
        ),
        error_y=dict(
            type='data',
            array=real_sd,
            color='rgba(40,40,40,0.50)',
            thickness=2.0,
            width=7,
        ),
        hovertemplate=(
            '<b>%{x}</b><br>'
            'Реал. медиана: %{y:.1f}%<br>'
            'Реал. SD: ±%{error_y.array:.1f}%<br>'
            'n = ' + '%{customdata}<extra></extra>'
        ),
        customdata=n_real,
    ))

    # 4. Аннотации Δ над каждой парой + n
    y_top = [max(m + s, r + s2) * 1.08
             for m, s, r, s2 in zip(mc_med, mc_sd, real_med, real_sd)]

    for xi, (label, d, n, yt) in enumerate(
            zip(rate_labels, delta, n_real, y_top)):
        color_ann = C['red'] if d > 0 else C['green']
        sign      = '+' if d > 0 else ''
        fig.add_annotation(
            x=label,
            y=yt,
            text=(f'<b>{sign}{d:.1f}пп</b><br>'
                  f'<span style="font-size:9px">n={n}</span>'),
            showarrow=False,
            font=dict(size=10, color=color_ann),
            align='center',
            bgcolor='rgba(255,255,255,0.80)',
            bordercolor='rgba(200,200,200,0.6)',
            borderwidth=1, borderpad=3,
        )

    # 5. Нулевая линия и сетка
    fig.update_yaxes(
        zeroline=True, zerolinecolor='rgba(150,150,150,0.4)',
        gridcolor='rgba(200,200,200,0.3)',
    )

    y_max = max(m + s for m, s in zip(mc_med + real_med,
                                       mc_sd   + real_sd)) * 1.25

    fig.update_layout(
        **LAYOUT,
        height=460,
        barmode='group',
        bargap=0.25,
        bargroupgap=0.06,
        title=dict(
            text='Коэффициенты конверсии: MC-прогноз vs Реальность',
            font=dict(size=15),
        ),
        yaxis=dict(
            title='Коэффициент конверсии (%)',
            range=[0, y_max],
            zeroline=False,
        ),
        legend=dict(
            orientation='h',
            x=0.5, xanchor='center',
            y=1.04, yanchor='bottom',
            bgcolor='rgba(255,255,255,0.85)',
            bordercolor='#ddd', borderwidth=1,
            font=dict(size=12),
        ),
    )
    fig.show()

    # ── Сводная таблица метрик ────────────────────────────────
    from IPython.display import display, HTML

    rows_tbl = []
    for label, rm, rs, mm, ms, d, n in zip(
            rate_labels, real_med, real_sd, mc_med, mc_sd, delta, n_real):
        bias_color = '#C0392B' if d > 0 else '#1E8449'
        rows_tbl.append({
            'Стадия':        label,
            'N':             n,
            'Реал. медиана': f'{rm:.1f}%',
            'Реал. SD':      f'±{rs:.1f}%',
            'MC медиана':    f'{mm:.1f}%',
            'MC SD':         f'±{ms:.1f}%',
            'Δ (MC−Реал)':   f'{"+" if d>0 else ""}{d:.1f}пп',
        })

    summary_conv = pd.DataFrame(rows_tbl)

    def _style_delta(val):
        if '+' in str(val):
            return 'color:#C0392B;font-weight:bold'
        if val.startswith('-'):
            return 'color:#1E8449;font-weight:bold'
        return ''

    display(HTML(
        '<div style="font-family:Inter,sans-serif;font-size:11px;'
        'padding:8px 12px;border-radius:6px;'
        'border-left:3px solid #ccc;margin-bottom:6px">'
        '<b>Штриховка</b> = MC прогноз &nbsp;|&nbsp; '
        '<b>Сплошная</b> = Реальность &nbsp;|&nbsp; '
        '<b>Error bars</b> = ±SD &nbsp;|&nbsp; '
        '<b>Δ</b> = MC медиана − Реальная медиана '
        '(<span style="color:#C0392B">красный</span> = MC завышает, '
        '<span style="color:#1E8449">зелёный</span> = MC занижает)'
        '</div>'
    ))

    display(summary_conv.style.applymap(
        _style_delta, subset=['Δ (MC−Реал)']
    ))

Штриховка = MC прогноз  |  Сплошная = Реальность  |  Error bars = ±SD  |  Δ = MC медиана − Реальная медиана ( красный = MC завышает, зелёный = MC занижает)

,Стадия,N,Реал. медиана,Реал. SD,MC медиана,MC SD,Δ (MC−Реал)
0,OCC→MII,47,77.4%,±23.2%,80.0%,±20.2%,+2.6пп
1,MII→2PN,46,75.0%,±31.9%,71.9%,±12.1%,-3.1пп
2,OCC→Blast,47,28.6%,±21.7%,41.7%,±18.0%,+13.1пп
3,Blast→HQ,37,88.9%,±24.8%,75.0%,±25.4%,-13.9пп


## 11. Автоматическая интерпретация + список на медразбор

In [23]:
# ══════════════════════════════════════════════════════════════════════════════
#  БЛОК: АВТОМАТИЧЕСКАЯ ИНТЕРПРЕТАЦИЯ ИТОГОВОЙ ТАБЛИЦЫ
#  Вставить как отдельную ячейку после генерации summary_df и df
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from IPython.display import display, HTML

# ── Цветовая схема для HTML-вывода ────────────────────────────────────────────
CSS = """
<style>
  .ivf-block        { font-family: 'Segoe UI', sans-serif; font-size: 13px;
                      line-height: 1.6; color: #1a1a2e; margin-bottom: 18px; }
  .ivf-section      { background: #f0f4f8; border-left: 4px solid #1B4F72;
                      padding: 12px 16px; border-radius: 6px; margin-bottom: 14px; }
  .ivf-section h3   { margin: 0 0 8px 0; color: #1B4F72; font-size: 14px; }
  .ivf-warn         { background: #fff8e1; border-left: 4px solid #f9a825; }
  .ivf-danger       { background: #ffebee; border-left: 4px solid #c62828; }
  .ivf-ok           { background: #e8f5e9; border-left: 4px solid #2e7d32; }
  .ivf-neutral      { background: #e3f2fd; border-left: 4px solid #1565c0; }
  .ivf-badge        { display: inline-block; padding: 1px 8px; border-radius: 10px;
                      font-size: 11px; font-weight: bold; margin: 1px; }
  .badge-red        { background: #ffcdd2; color: #b71c1c; }
  .badge-orange     { background: #ffe0b2; color: #e65100; }
  .badge-green      { background: #c8e6c9; color: #1b5e20; }
  .badge-blue       { background: #bbdefb; color: #0d47a1; }
  .badge-gray       { background: #e0e0e0; color: #424242; }
  .review-table     { width: 100%; border-collapse: collapse; font-size: 12px; }
  .review-table th  { background: #1B4F72; color: white; padding: 6px 10px;
                      text-align: left; }
  .review-table td  { padding: 5px 10px; border-bottom: 1px solid #e0e0e0; }
  .review-table tr:hover td { background: #f5f5f5; }
  .priority-1       { border-left: 4px solid #c62828 !important; }
  .priority-2       { border-left: 4px solid #f9a825 !important; }
  .priority-3       { border-left: 4px solid #2e7d32 !important; }
</style>
"""


# ══════════════════════════════════════════════════════════════════════════════
#  ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ══════════════════════════════════════════════════════════════════════════════

def _safe(df_row, col, default=np.nan):
    v = df_row.get(col, default)
    return default if (v is None or (isinstance(v, float) and np.isnan(v))) else v


def _badge(text, color='gray'):
    return f'<span class="ivf-badge badge-{color}">{text}</span>'


def _pct(v):
    """Форматирует float 0-1 как процент."""
    try:
        return f"{float(v)*100:.1f}%"
    except Exception:
        return "—"


# ══════════════════════════════════════════════════════════════════════════════
#  ФУНКЦИЯ: ФЛАГИ ДЛЯ ОДНОГО ПАЦИЕНТА
# ══════════════════════════════════════════════════════════════════════════════

def _patient_flags(row) -> list[dict]:
    """
    Возвращает список флагов вида:
      {'level': 'danger'|'warn'|'ok', 'code': str, 'text': str}
    """
    flags = []

    p_mc   = _safe(row, 'p_per_transfer')
    p_kat  = _safe(row, 'p_kat_raw')
    p_nvsa = _safe(row, 'p_nvsa')
    p_prai = _safe(row, 'PRAI')
    p_csdi = _safe(row, 'p_csdi')
    p_dt   = _safe(row, 'DIGITAL TWIN')
    preg   = _safe(row, 'Preg')
    cluster = _safe(row, 'dominant_cluster', -1)

    med_bl   = _safe(row, 'med_blasts', 0)
    real_bl  = _safe(row, 'Bl', np.nan)
    real_occ = _safe(row, 'OCC', np.nan)
    med_occ  = _safe(row, 'med_okk', np.nan)
    real_mii = _safe(row, 'MII', np.nan)
    med_mii  = _safe(row, 'med_mii', np.nan)
    cancel   = _safe(row, 'p_cancel_risk', 0)
    age      = _safe(row, 'Age', 0)

    # ── 1. Ключевое расхождение KAT vs PRAI ─────────────────────────────────
    if not np.isnan(p_kat) and not np.isnan(p_prai):
        diff = p_kat - p_prai
        if diff > 0.25:
            flags.append({'level': 'danger', 'code': 'KAT↑↑PRAI',
                'text': f'KAT ({_pct(p_kat)}) сильно завышает: PRAI={_pct(p_prai)}, '
                        f'разрыв +{diff*100:.1f} пп — симуляция MC слишком оптимистична'})
        elif diff > 0.12:
            flags.append({'level': 'warn', 'code': 'KAT↑PRAI',
                'text': f'KAT ({_pct(p_kat)}) выше PRAI ({_pct(p_prai)}) '
                        f'на {diff*100:.1f} пп — умеренный оптимизм симуляции'})
        elif diff < -0.15:
            flags.append({'level': 'ok', 'code': 'PRAI↑KAT',
                'text': f'PRAI ({_pct(p_prai)}) выше KAT ({_pct(p_kat)}) '
                        f'на {abs(diff)*100:.1f} пп — реальный цикл лучше прогноза, '
                        f'позитивный сигнал'})

    # ── 2. Согласованность пайплайна ─────────────────────────────────────────
    models_avail = [v for v in [p_mc, p_kat, p_nvsa, p_csdi] if not np.isnan(v)]
    if len(models_avail) >= 3:
        spread = max(models_avail) - min(models_avail)
        if spread > 0.30:
            flags.append({'level': 'warn', 'code': 'SPREAD↑',
                'text': f'Высокий разброс между моделями пайплайна: '
                        f'{spread*100:.1f} пп (MC={_pct(p_mc)}, '
                        f'KAT={_pct(p_kat)}, CSDI={_pct(p_csdi)}) — '
                        f'повышенная неопределённость прогноза'})

    # ── 3. Бластоцисты: реальные vs предсказанные ────────────────────────────
    if not np.isnan(real_bl) and not np.isnan(med_bl) and med_bl > 0:
        ratio = real_bl / med_bl
        if real_bl == 0 and med_bl >= 2:
            flags.append({'level': 'danger', 'code': 'BL=0',
                'text': f'Реально 0 бластоцист при прогнозе {int(med_bl)} — '
                        f'критический лабораторный исход, цикл без переноса'})
        elif ratio < 0.4:
            flags.append({'level': 'danger', 'code': 'BL↓↓',
                'text': f'Бластоцист значительно меньше прогноза: '
                        f'{int(real_bl)} vs {int(med_bl)} '
                        f'({ratio*100:.0f}% от ожидаемого)'})
        elif ratio < 0.65:
            flags.append({'level': 'warn', 'code': 'BL↓',
                'text': f'Бластоцист меньше прогноза: '
                        f'{int(real_bl)} vs {int(med_bl)}'})
        elif ratio > 1.4:
            flags.append({'level': 'ok', 'code': 'BL↑',
                'text': f'Бластоцист больше прогноза: '
                        f'{int(real_bl)} vs {int(med_bl)} — лабораторный ответ лучше ожиданий'})

    # ── 4. OCC: неожиданный ответ яичников ───────────────────────────────────
    if not np.isnan(real_occ) and not np.isnan(med_occ) and med_occ > 0:
        ratio_occ = real_occ / med_occ
        if real_occ == 0:
            flags.append({'level': 'danger', 'code': 'OCC=0',
                'text': 'Ооцитов не получено (отмена/пустая пункция)'})
        elif ratio_occ > 1.8:
            flags.append({'level': 'warn', 'code': 'OCC↑↑',
                'text': f'ОКК намного выше прогноза: {int(real_occ)} vs {int(med_occ)} — '
                        f'риск ССЯГ не был учтён на входе'})

    # ── 5. CSDI vs MC: расхождение диффузионной модели ───────────────────────
    if not np.isnan(p_csdi) and not np.isnan(p_mc):
        csdi_mc_diff = p_mc - p_csdi
        if csdi_mc_diff > 0.35:
            flags.append({'level': 'warn', 'code': 'CSDI↓↓MC',
                'text': f'CSDI ({_pct(p_csdi)}) значительно ниже MC ({_pct(p_mc)}) — '
                        f'диффузионная модель видит слабый эмбриологический потенциал'})

    # ── 6. Риск отмены ────────────────────────────────────────────────────────
    if not np.isnan(cancel) and cancel > 0.08:
        flags.append({'level': 'warn' if cancel < 0.18 else 'danger',
            'code': 'CANCEL↑',
            'text': f'Риск отмены цикла: {cancel*100:.1f}%'})

    # ── 7. Возраст + кластер C1 ───────────────────────────────────────────────
    if age >= 38 and cluster == 1:
        flags.append({'level': 'warn', 'code': 'AGE+C1',
            'text': f'Возраст ≥38 ({int(age)} л.) + кластер C1 Poor — '
                    f'особое внимание к тактике'})

    # ── 8. Ложноположительный исход (высокий прогноз, нет беременности) ──────
    if preg == 0 and not np.isnan(p_mc) and p_mc > 0.60:
        flags.append({'level': 'danger', 'code': 'FP',
            'text': f'Беременность НЕ наступила при MC={_pct(p_mc)} — '
                    f'ложноположительный прогноз, разбор показан'})

    # ── 9. Истинноположительный ──────────────────────────────────────────────
    if preg == 1 and not np.isnan(p_mc) and p_mc >= 0.50:
        flags.append({'level': 'ok', 'code': 'TP',
            'text': f'Беременность наступила, прогноз был обоснован (MC={_pct(p_mc)})'})

    # ── 10. Ложноотрицательный (низкий прогноз, беременность наступила) ──────
    if preg == 1 and not np.isnan(p_mc) and p_mc < 0.45:
        flags.append({'level': 'ok', 'code': 'FN+',
            'text': f'Беременность наступила вопреки низкому прогнозу MC={_pct(p_mc)} — '
                    f'интересный случай для разбора'})

    return flags


# ══════════════════════════════════════════════════════════════════════════════
#  ФУНКЦИЯ: ПРИОРИТЕТ МЕДИЦИНСКОГО РАЗБОРА
# ══════════════════════════════════════════════════════════════════════════════

def _review_priority(flags: list[dict]) -> tuple[int, str]:
    """
    Возвращает (приоритет 1-3, обоснование).
    1 = обязательный разбор, 2 = рекомендован, 3 = плановый мониторинг.
    """
    codes   = {f['code'] for f in flags}
    levels  = [f['level'] for f in flags]
    n_danger = levels.count('danger')
    n_warn   = levels.count('warn')

    if 'FP' in codes or 'BL=0' in codes or 'OCC=0' in codes:
        return 1, 'Критическое расхождение прогноза и исхода'
    if n_danger >= 2:
        return 1, 'Множественные критические флаги'
    if 'KAT↑↑PRAI' in codes and ('BL↓↓' in codes or 'BL↓' in codes):
        return 1, 'Завышенный прогноз + недостаточный выход бластоцист'
    if 'FN+' in codes:
        return 2, 'Беременность вопреки низкому прогнозу — поучительный случай'
    if n_danger >= 1 or n_warn >= 2:
        return 2, 'Один или более критических флагов'
    if n_warn >= 1:
        return 3, 'Незначительные расхождения'
    return 3, 'Прогноз соответствует реальным данным'


# ══════════════════════════════════════════════════════════════════════════════
#  ГЛАВНАЯ ФУНКЦИЯ: ПОЛНАЯ ИНТЕРПРЕТАЦИЯ
# ══════════════════════════════════════════════════════════════════════════════

def interpret_dt_summary(df: pd.DataFrame) -> None:
    """
    Принимает объединённый датафрейм df (результат merge DT + OPU).
    Выводит:
      - Общие суждения по когорте
      - Области внимания
      - Таблицу пациентов для медразбора с приоритетами
    """
    # ── Дедупликация: берём последнюю запись на пациента ─────────────────────
    # (если один пациент пересчитывался — берём финальный расчёт)
    df_u = (df.sort_values('timestamp', ascending=True)
              .drop_duplicates(subset=['_key'], keep='last')
              .copy()) if 'timestamp' in df.columns else df.copy()

    n_total    = len(df_u)
    n_outcomes = df_u['Preg'].notna().sum() if 'Preg' in df_u.columns else 0
    n_preg     = int(df_u['Preg'].sum()) if n_outcomes > 0 else 0

    # ── Вычисляем флаги по всем пациентам ────────────────────────────────────
    all_flags = {}
    all_priority = {}
    for idx, row in df_u.iterrows():
        flags = _patient_flags(row)
        priority, reason = _review_priority(flags)
        all_flags[idx]    = flags
        all_priority[idx] = (priority, reason)

    df_u['_flags']    = df_u.index.map(lambda i: all_flags.get(i, []))
    df_u['_priority'] = df_u.index.map(lambda i: all_priority.get(i, (3,''))[0])
    df_u['_p_reason'] = df_u.index.map(lambda i: all_priority.get(i, (3,''))[1])

    # ── 1. ОБЩИЕ СУЖДЕНИЯ ────────────────────────────────────────────────────
    html = [CSS, '<div class="ivf-block">']
    html.append('<div class="ivf-section ivf-neutral"><h3> Общая характеристика когорты</h3>')

    # Средние прогнозы
    mc_mean  = df_u['p_per_transfer'].mean() if 'p_per_transfer' in df_u.columns else np.nan
    kat_mean = df_u['p_kat_raw'].mean()      if 'p_kat_raw' in df_u.columns else np.nan
    prai_mean= df_u['PRAI'].mean()           if 'PRAI' in df_u.columns else np.nan
    real_rate= n_preg / n_outcomes           if n_outcomes > 0 else np.nan

    lines = [
        f'<b>N пациентов (уникальных):</b> {n_total} &nbsp;|&nbsp; '
        f'<b>С известным исходом:</b> {n_outcomes} &nbsp;|&nbsp; '
        f'<b>Реальный % беременности:</b> {"%.1f%%" % (real_rate*100) if not np.isnan(real_rate) else "—"}',

        f'<b>Средний MC-прогноз:</b> {_pct(mc_mean)} &nbsp;|&nbsp; '
        f'<b>Средний KAT:</b> {_pct(kat_mean)} &nbsp;|&nbsp; '
        f'<b>Средний PRAI:</b> {_pct(prai_mean)}',
    ]

    if not np.isnan(real_rate) and not np.isnan(mc_mean):
        overest = mc_mean - real_rate
        direction = 'завышает' if overest > 0.05 else ('занижает' if overest < -0.05 else 'точен')
        lines.append(
            f'<b>Систематический bias MC:</b> {overest*100:+.1f} пп '
            f'(MC {direction} реальную частоту)'
        )

    if not np.isnan(kat_mean) and not np.isnan(prai_mean):
        kat_prai_bias = kat_mean - prai_mean
        lines.append(
            f'<b>Систематический bias KAT vs PRAI:</b> {kat_prai_bias*100:+.1f} пп '
            f'({"KAT завышает" if kat_prai_bias > 0 else "PRAI выше KAT"})'
        )

    html.append('<br>'.join(lines))
    html.append('</div>')

    # ── 2. КЛАСТЕРНЫЙ СРЕЗ ───────────────────────────────────────────────────
    if 'dominant_cluster' in df_u.columns:
        cluster_names = {0: 'C0 Standard', 1: 'C1 Poor', 2: 'C2 High'}
        html.append('<div class="ivf-section"><h3> Кластерный профиль когорты</h3>')
        for c in [0, 1, 2]:
            sub = df_u[df_u['dominant_cluster'] == c]
            n_c = len(sub)
            if n_c == 0:
                continue
            n_out = sub['Preg'].notna().sum()
            rate_c = sub['Preg'].mean() if n_out > 0 else np.nan
            html.append(
                f'{cluster_names[c]}: <b>{n_c} пациентов ({n_c/n_total*100:.0f}%)</b> '
                f'— реальный исход: {"%.0f%%" % (rate_c*100) if not np.isnan(rate_c) else "нет данных"} '
                f'({n_out} с исходом) &nbsp; '
            )
        html.append('</div>')

    # ── 3. ОБЛАСТИ ВНИМАНИЯ ──────────────────────────────────────────────────
    html.append('<div class="ivf-section ivf-warn"><h3>⚠️ Области системного внимания</h3><ul>')

    # 3a. Переоценка бластоцист
    bl_flags = df_u[df_u['_flags'].apply(
        lambda fl: any(f['code'] in ('BL=0','BL↓↓') for f in fl)
    )]
    if len(bl_flags) > 0:
        html.append(
            f'<li><b>Переоценка выхода бластоцист</b> у {len(bl_flags)} пациентов — '
            f'модель систематически оптимистична на стадии бластуляции. '
            f'Возможная причина: не учитываются лабораторные факторы (качество спермы, '
            f'эмбриолог, питательные среды).</li>'
        )

    # 3b. Высокий разрыв KAT vs PRAI
    kat_warn = df_u[df_u['_flags'].apply(
        lambda fl: any(f['code'] == 'KAT↑↑PRAI' for f in fl)
    )]
    if len(kat_warn) > 0:
        html.append(
            f'<li><b>Сильное расхождение KAT vs PRAI</b> у {len(kat_warn)} пациентов — '
            f'симуляция MC генерирует завышенный входной вектор для KAT. '
            f'При консультации следует опираться на PRAI как более точный ориентир.</li>'
        )

    # 3c. Высокий spread между моделями
    spread_warn = df_u[df_u['_flags'].apply(
        lambda fl: any(f['code'] == 'SPREAD↑' for f in fl)
    )]
    if len(spread_warn) > 0:
        html.append(
            f'<li><b>Высокая неопределённость прогноза</b> у {len(spread_warn)} пациентов — '
            f'значительный разброс между слоями пайплайна. '
            f'Прогноз следует преподносить с широким доверительным интервалом.</li>'
        )

    # 3d. Ложноположительные (FP)
    fp = df_u[df_u['_flags'].apply(lambda fl: any(f['code'] == 'FP' for f in fl))]
    if len(fp) > 0:
        html.append(
            f'<li><b>Ложноположительные прогнозы</b>: {len(fp)} случай(ев) с высоким MC '
            f'и отсутствием беременности — обязательный разбор для калибровки модели.</li>'
        )

    # 3e. Общий % Priority 1
    p1 = (df_u['_priority'] == 1).sum()
    if p1 > 0:
        html.append(
            f'<li><b>Приоритет 1 (обязательный разбор):</b> {p1} пациентов '
            f'({p1/n_total*100:.0f}%) — см. таблицу ниже.</li>'
        )

    html.append('</ul></div>')

    # ── 4. ТАБЛИЦА ДЛЯ МЕДРАЗБОРА ────────────────────────────────────────────
    for priority_level, priority_label, div_class in [
        (1, '🔴 Приоритет 1 — Обязательный разбор',  'ivf-danger'),
        (2, '🟡 Приоритет 2 — Рекомендован разбор',  'ivf-warn'),
        (3, '🟢 Приоритет 3 — Плановый мониторинг',  'ivf-ok'),
    ]:
        subset = df_u[df_u['_priority'] == priority_level].copy()
        if subset.empty:
            continue

        html.append(f'<div class="ivf-section {div_class}">')
        html.append(f'<h3>{priority_label} ({len(subset)} пациентов)</h3>')
        html.append('<table class="review-table"><tr>'
                    '<th>Пациент</th><th>ID</th><th>Возраст</th>'
                    '<th>MC%</th><th>KAT%</th><th>PRAI%</th><th>DT%</th>'
                    '<th>Исход</th><th>Кластер</th>'
                    '<th>Флаги и суждения</th></tr>')

        cluster_names_s = {0: 'C0', 1: 'C1 Poor', 2: 'C2 High'}
        NO_FLAGS = '<i style="color:#aaa">без флагов</i>'

        for _, row in subset.sort_values('_priority').iterrows():
            flags  = row['_flags']
            reason = row['_p_reason']
            preg   = _safe(row, 'Preg')
            age    = _safe(row, 'Age', '?')
            cluster = _safe(row, 'dominant_cluster', -1)
            cname  = cluster_names_s.get(int(cluster), '—') if cluster != -1 else '—'
            name   = str(row.get('patient_name', row.get('_key', '—')))[:28]
            pid    = str(row.get('_key', '—'))

            # Исход
            if preg == 1:
                outcome_html = _badge('Беременность ✅', 'green')
            elif preg == 0:
                outcome_html = _badge('Нет ❌', 'red')
            else:
                outcome_html = _badge('нет данных', 'gray')

            # Флаги
            flag_html = ''
            for f in flags:
                col = {'danger': 'red', 'warn': 'orange', 'ok': 'green'}.get(f['level'], 'gray')
                flag_html += f'<b>{_badge(f["code"], col)}</b> {f["text"]}<br>'

            # Кластер-бейдж
            c_col = {'C0': 'blue', 'C1 Poor': 'orange', 'C2 High': 'green'}.get(cname, 'gray')
            c_badge = _badge(cname, c_col)

            html.append(
                f'<tr class="priority-{priority_level}">'
                f'<td>{name}</td>'
                f'<td><code style="font-size:10px">{pid}</code></td>'
                f'<td>{int(age) if age != "?" else "?"}</td>'
                f'<td>{_pct(_safe(row,"p_per_transfer"))}</td>'
                f'<td>{_pct(_safe(row,"p_kat_raw"))}</td>'
                f'<td>{_pct(_safe(row,"PRAI"))}</td>'
                f'<td>{_pct(_safe(row,"DIGITAL TWIN"))}</td>'
                f'<td>{outcome_html}</td>'
                f'<td>{c_badge}</td>'
                f'<td>{flag_html or NO_FLAGS}</td>'
                f'</tr>'
            )

        html.append('</table></div>')

    # ── 5. ВЫГРУЗКА СПИСКА ДЛЯ РАЗБОРА ──────────────────────────────────────
    html.append('<div class="ivf-section"><h3>📋 Список пациентов для передачи на разбор</h3>')
    review_list = (
        df_u[
            (df_u['_priority'].isin([1, 2])) &
            (df_u['_p_reason'] != 'Один или более критических флагов')
        ][
            ['patient_name', '_key', 'Age', '_priority', '_p_reason']
        ]
        .rename(columns={
            'patient_name': 'Пациент',
            '_key': 'ID',
            'Age': 'Возраст',
            '_priority': 'Приоритет',
            '_p_reason': 'Основание',
        })
        .sort_values('Приоритет')
    )

    html.append(review_list.to_html(index=False, border=0,
                                    classes='review-table', escape=False))
    html.append('</div>')

    html.append('</div>')  # ivf-block

    display(HTML(''.join(html)))

    # ── Возвращаем датафрейм для дальнейшей работы ───────────────────────────
    return review_list


# ══════════════════════════════════════════════════════════════════════════════
#  ЗАПУСК
#  df — объединённый датафрейм из предыдущих ячеек (merge DT + OPU)
# ══════════════════════════════════════════════════════════════════════════════

review_df = interpret_dt_summary(df)

Пациент,ID,Возраст,MC%,KAT%,PRAI%,DT%,Исход,Кластер,Флаги и суждения
TURGUNBOEVA NILUFAR,AD1118005,25,75.6%,72.6%,34.2%,48.4%,нет данных,C2 High,"KAT↑↑PRAI KAT (72.6%) сильно завышает: PRAI=34.2%, разрыв +38.4 пп — симуляция MC слишком оптимистичнаBL↓ Бластоцист меньше прогноза: 3 vs 7"
UMIRZAKOVA MALIKA,AD7420759,27,74.0%,62.7%,nan%,33.1%,нет данных,C2 High,"BL=0 Реально 0 бластоцист при прогнозе 9 — критический лабораторный исход, цикл без переноса"
KADIROVA AMIRA,AD6317635,26,72.2%,53.3%,21.2%,31.6%,нет данных,C0,"KAT↑↑PRAI KAT (53.3%) сильно завышает: PRAI=21.2%, разрыв +32.1 пп — симуляция MC слишком оптимистичнаBL↓↓ Бластоцист значительно меньше прогноза: 1 vs 4 (25% от ожидаемого)"
Davronova Bakhora,AB3537881,33,68.1%,56.7%,59.8%,41.9%,Нет ❌,C2 High,"FP Беременность НЕ наступила при MC=68.1% — ложноположительный прогноз, разбор показан"
Isakova Madina,AD0357260,29,72.3%,56.4%,50.0%,43.9%,Нет ❌,C2 High,"BL↓↓ Бластоцист значительно меньше прогноза: 2 vs 15 (13% от ожидаемого)FP Беременность НЕ наступила при MC=72.3% — ложноположительный прогноз, разбор показан"
Mamiyeva Dildora,AD0535057,25,75.4%,64.8%,65.6%,40.4%,Нет ❌,C0,"SPREAD↑ Высокий разброс между моделями пайплайна: 35.0 пп (MC=75.4%, KAT=64.8%, CSDI=40.4%) — повышенная неопределённость прогнозаFP Беременность НЕ наступила при MC=75.4% — ложноположительный прогноз, разбор показан"
Nurkobilova Muazzam,402890014,38,48.9%,49.1%,14.3%,31.0%,нет данных,C0,"KAT↑↑PRAI KAT (49.1%) сильно завышает: PRAI=14.3%, разрыв +34.8 пп — симуляция MC слишком оптимистичнаBL=0 Реально 0 бластоцист при прогнозе 3 — критический лабораторный исход, цикл без переноса"
Ergasheva Sarvinoz,AB7930079,24,75.5%,54.2%,31.4%,54.5%,Нет ❌,C0,"KAT↑PRAI KAT (54.2%) выше PRAI (31.4%) на 22.8 пп — умеренный оптимизм симуляцииBL↓ Бластоцист меньше прогноза: 2 vs 5FP Беременность НЕ наступила при MC=75.5% — ложноположительный прогноз, разбор показан"
Oozoqboyeva Zuxraxon,AE5515419,24,75.3%,60.7%,28.0%,47.0%,нет данных,C0,"KAT↑↑PRAI KAT (60.7%) сильно завышает: PRAI=28.0%, разрыв +32.7 пп — симуляция MC слишком оптимистичнаBL=0 Реально 0 бластоцист при прогнозе 4 — критический лабораторный исход, цикл без переноса"
Borotova Aziza,AC3214218,35,57.9%,50.2%,30.0%,29.8%,нет данных,C0,"KAT↑PRAI KAT (50.2%) выше PRAI (30.0%) на 20.2 пп — умеренный оптимизм симуляцииBL=0 Реально 0 бластоцист при прогнозе 3 — критический лабораторный исход, цикл без переноса"


In [24]:
# ── Надёжный экспорт в Excel ─────────────────────────────────
def _hex_to_rgb(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def _save_sheets(sheets, xlsx_path):
    """Пробует xlsxwriter → openpyxl → CSV-fallback."""
    for engine in ('xlsxwriter', 'openpyxl'):
        try:
            with pd.ExcelWriter(xlsx_path, engine=engine) as w:
                for name, frame in sheets.items():
                    frame.to_excel(w, sheet_name=name[:31], index=False)
            print(f'✅ Excel ({engine}): {xlsx_path}')
            return
        except Exception as e:
            print(f'  {engine}: {e}')
    # CSV fallback
    base = xlsx_path.replace('.xlsx', '')
    for name, frame in sheets.items():
        p = f'{base}_{name}.csv'
        frame.to_csv(p, index=False, encoding='utf-8-sig')
        print(f'  ✅ CSV: {p}')

# ── Итоговая сводная таблица ──────────────────────────────────
def _build_summary(df_in):
    cluster_names = {0: 'C0 Standard', 1: 'C1 Poor', 2: 'C2 High'}
    def _p(v):
        try: return f'{float(v)*100:.1f}%'
        except: return '—'
    def _flag(p, r, thr=0.3):
        if pd.isna(p) or pd.isna(r) or r == 0: return '—'
        return '⚠️' if abs(p - r) / r > thr else '✅'
    rows = []
    for _, row in df_in.iterrows():
        occ_r = int(row['OCC']) if pd.notna(row.get('OCC')) else '?'
        occ_p = int(row['med_okk']) if pd.notna(row.get('med_okk')) else '?'
        bl_r  = int(row['Bl']) if pd.notna(row.get('Bl')) else '?'
        bl_p  = int(row['med_blasts']) if pd.notna(row.get('med_blasts')) else '?'
        kat_prai = (
            f"{(row.get('p_kat_raw', np.nan) - row.get('PRAI', np.nan))*100:+.1f} пп"
            if pd.notna(row.get('p_kat_raw')) and pd.notna(row.get('PRAI')) else '—'
        )
        rows.append({
            'Пациент':     str(row.get('patient_name', ''))[:25],
            'ID':          row.get('_key', ''),
            'Возраст':     row.get('Age', ''),
            'OCC (р/п)':   f'{occ_r} / {occ_p}',
            'OCC':         _flag(row.get('med_okk'), row.get('OCC')),
            'Bl (р/п)':    f'{bl_r} / {bl_p}',
            'Bl':          _flag(row.get('med_blasts'), row.get('Bl')),
            'MC%':         _p(row.get('p_per_transfer')),
            'KAT%':        _p(row.get('p_kat_raw')),
            'PRAI%':       _p(row.get('PRAI')),
            'DT%':         _p(row.get('DIGITAL TWIN')),
            'KAT−PRAI':    kat_prai,
            'Кластер':     cluster_names.get(row.get('dominant_cluster'), '—'),
            'Исход':       ('✅' if row.get('Preg') == 1
                            else '❌' if row.get('Preg') == 0 else '—'),
        })
    return pd.DataFrame(rows)

summary_df = _build_summary(df_u)

# ── Сохранение ────────────────────────────────────────────────
_save_sheets({
    'Summary':      summary_df,
    'FullData':     df_u.drop(columns=[c for c in df_u.columns
                                        if c.startswith('_')], errors='ignore'),
    'WithOutcomes': df_preg,
}, 'DT_Analytics_Report.xlsx')

# ── Итоговая сводка в stdout ──────────────────────────────────
print(f'\nОтчёт: {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")}')
print(f'Уникальных пациентов: {len(df_u)} | С исходом: {len(df_preg)} | '
      f'Беременность: {int(df_preg["Preg"].sum())} / {len(df_preg)}')
print(f'\nСводная таблица ({len(summary_df)} строк):')
display(summary_df)

  xlsxwriter: No module named 'xlsxwriter'
✅ Excel (openpyxl): DT_Analytics_Report.xlsx

Отчёт: 2026-05-28 05:24
Уникальных пациентов: 51 | С исходом: 12 | Беременность: 6 / 12

Сводная таблица (51 строк):


,Пациент,ID,Возраст,OCC (р/п),OCC,Bl (р/п),Bl,MC%,KAT%,PRAI%,DT%,KAT−PRAI,Кластер,Исход
0,KARIMJANOVA SHOKHISTAKHON,AC2986596,30,8 / 8,✅,4 / 4,✅,68.2%,42.7%,23.5%,49.6%,+19.2 пп,C1 Poor,—
1,KUCHKOROVA MAFTUNAKHON,AB7871088,25,44 / 44,✅,20 / 16,✅,76.3%,73.4%,51.7%,52.4%,+21.7 пп,C2 High,—
2,TURGUNOVA MATLUBA,AD5571994,37,14 / 14,✅,9 / 7,✅,57.8%,36.5%,23.5%,48.2%,+13.0 пп,C0 Standard,—
3,HOMIDOVA KURMATOY,404378919,26,12 / 12,✅,5 / 5,✅,73.5%,59.3%,50.9%,54.4%,+8.4 пп,C0 Standard,—
4,ODILOVA MUXLISA,AD0497909,27,7 / 7,✅,2 / 3,⚠️,67.0%,47.6%,28.4%,35.8%,+19.2 пп,C1 Poor,—
5,TURGUNBOEVA NILUFAR,AD1118005,25,32 / 32,✅,3 / 7,⚠️,75.6%,72.6%,34.2%,48.4%,+38.4 пп,C2 High,—
6,UMAROVA ZARNIGOR,AD0012066,23,14 / 14,✅,7 / 6,✅,76.9%,54.8%,35.8%,36.3%,+19.0 пп,C0 Standard,—
7,TILAKOVA IBODAT,AB4648993,26,22 / 22,✅,9 / 10,✅,74.6%,68.4%,49.2%,43.3%,+19.2 пп,C2 High,—
8,ALIMOVA MOHIDIL,AB4449292,31,1 / 1,✅,0 / 1,—,51.1%,12.7%,12.7%,39.9%,+0.0 пп,C1 Poor,—
9,SAPAYEVA BARNO,AD4094629,38,13 / 13,✅,5 / 5,✅,55.8%,44.1%,49.6%,55.8%,-5.5 пп,C0 Standard,—


## 12 ГРАФ СХОДСТВА ПАЦИЕНТОВ — NetworkX + Plotly

In [25]:
# ── Зависимости ────────────────────────────────────────────────
import networkx as nx
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')
 
name_col = 'patient_name' if 'patient_name' in df_u.columns else '_key'
 
CLUSTER_NAMES = {0: 'C0 Standard', 1: 'C1 Poor',  2: 'C2 High'}
CLUSTER_HEX   = {0: '#1976D2',     1: '#C62828',   2: '#2E7D32'}

In [26]:
# ══════════════════════════════════════════════════════════════
# БЛОК A — ПОСТРОЕНИЕ ГРАФА
# ══════════════════════════════════════════════════════════════
 
# ── Признаки для расчёта сходства ────────────────────────────
# Три группы с разными весами:
#   Клинические входные  — то что знаем до цикла
#   Лабораторные итог    — реальный результат цикла
#   Предсказания DT      — что думала модель
GRAPH_FEATURES = {
    # (колонка, вес)
    'Age':            1.5,   # возраст — важный дифференциатор
    'amh':            2.0,   # AMH — ключевой маркер резерва
    'afc':            1.5,
    'OCC':            1.0,
    'MII':            1.0,
    'Bl':             1.2,
    'Good Bl':        1.2,
    'p_per_transfer': 1.0,
    'p_kat_raw':      1.0,
    'bayes_mean':     0.8,
}
GRAPH_FEATURES = {k: w for k, w in GRAPH_FEATURES.items()
                  if k in df_u.columns}
 
K_NEIGHBORS  = 5      # сколько соседей на пациента
SIM_THRESHOLD = 0.70  # минимальное сходство для ребра
 
# ── Матрица признаков с весами ────────────────────────────────
feat_cols = list(GRAPH_FEATURES.keys())
weights   = np.array(list(GRAPH_FEATURES.values()))
 
X_raw = df_u[feat_cols].copy()
X_raw = X_raw.fillna(X_raw.median())
 
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X_raw) * weights  # взвешенное масштабирование
 
sim_matrix = cosine_similarity(X_scaled)
np.fill_diagonal(sim_matrix, 0)
 
# ── Построение графа ──────────────────────────────────────────
G = nx.Graph()
 
for i, (idx, row) in enumerate(df_u.iterrows()):
    pname = str(row.get(name_col, row.get('_key', str(i))))
    G.add_node(
        pname,
        idx       = i,
        cluster   = int(row.get('dominant_cluster', -1)),
        age       = float(row.get('Age', 0)),
        amh       = float(row.get('amh', 0)),
        p_mc      = float(row.get('p_per_transfer', 0)),
        p_kat     = float(row.get('p_kat_raw', 0)),
        outcome   = row.get('Preg'),        # nan = нет данных
        c0_prob   = float(row.get('cluster_c0_prob', 0)),
        c1_prob   = float(row.get('cluster_c1_prob', 0)),
        c2_prob   = float(row.get('cluster_c2_prob', 0)),
    )
 
node_names = [str(row.get(name_col, row.get('_key', str(i))))
              for i, (_, row) in enumerate(df_u.iterrows())]
 
for i in range(len(df_u)):
    top_k = np.argsort(sim_matrix[i])[::-1][:K_NEIGHBORS]
    for j in top_k:
        sim_val = sim_matrix[i][j]
        if sim_val >= SIM_THRESHOLD and not G.has_edge(node_names[i], node_names[j]):
            G.add_edge(node_names[i], node_names[j],
                       weight=round(float(sim_val), 4),
                       sim_pct=round(float(sim_val) * 100, 1))
 
# ── Доп. атрибуты нод ─────────────────────────────────────────
degree_cent  = nx.degree_centrality(G)
between_cent = nx.betweenness_centrality(G, weight='weight')
 
for node in G.nodes():
    G.nodes[node]['degree_centrality']      = round(degree_cent[node], 4)
    G.nodes[node]['betweenness_centrality'] = round(between_cent[node], 4)
    G.nodes[node]['degree']                 = G.degree(node)
 
# ── Граф построен ─────────────────────────────────────────────
isolated = list(nx.isolates(G))
components = list(nx.connected_components(G))
 
print(f'✅ Граф построен')
print(f'   Признаков: {len(feat_cols)} | Пациентов: {G.number_of_nodes()}')
print(f'   Рёбер: {G.number_of_edges()} | Плотность: {nx.density(G):.3f}')
print(f'   Компонент связности: {len(components)}')
print(f'   Изолированных пациентов: {len(isolated)}')
if isolated:
    print(f'   → {isolated}  (нетипичный профиль)')
print(f'   Средняя степень нод: {np.mean([d for _, d in G.degree()]):.1f}')

✅ Граф построен
   Признаков: 10 | Пациентов: 50
   Рёбер: 128 | Плотность: 0.104
   Компонент связности: 3
   Изолированных пациентов: 1
   → ['KARIMJANOVA SHOKHISTAKHON']  (нетипичный профиль)
   Средняя степень нод: 5.1


In [27]:
# ══════════════════════════════════════════════════════════════
# БЛОК B — ВИЗУАЛИЗАЦИЯ ГРАФА
# ══════════════════════════════════════════════════════════════
 
# Layout: spring с seed для воспроизводимости
pos = nx.spring_layout(G, weight='weight', seed=42, k=2.5)
 
fig_graph = go.Figure()
 
# ── Рёбра ─────────────────────────────────────────────────────
# Группируем по «силе» связи: сильные / средние / слабые
edge_traces = {'strong': [], 'medium': [], 'weak': []}
 
for u, v, data in G.edges(data=True):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    sim = data.get('weight', 0)
    category = 'strong' if sim >= 0.88 else 'medium' if sim >= 0.78 else 'weak'
    edge_traces[category].append((x0, x1, y0, y1, sim, u, v))
 
EDGE_CFG = {
    'strong': dict(color='rgba(100,100,100,0.55)', width=2.5, dash='solid',  name='Высокое сходство (>88%)'),
    'medium': dict(color='rgba(150,150,150,0.35)', width=1.5, dash='solid',  name='Среднее (78–88%)'),
    'weak':   dict(color='rgba(180,180,180,0.18)', width=0.8, dash='dot',    name='Слабое (<78%)'),
}
 
for cat, edges in edge_traces.items():
    if not edges:
        continue
    cfg = EDGE_CFG[cat]
    xs, ys, hover_x, hover_y, hover_txt = [], [], [], [], []
    for x0, x1, y0, y1, sim, u, v in edges:
        xs  += [x0, x1, None]
        ys  += [y0, y1, None]
        hover_x.append((x0 + x1) / 2)
        hover_y.append((y0 + y1) / 2)
        hover_txt.append(f'{u[:14]} ↔ {v[:14]}<br>Сходство: {sim*100:.1f}%')
 
    fig_graph.add_trace(go.Scatter(
        x=xs, y=ys, mode='lines',
        line=dict(color=cfg['color'], width=cfg['width'], dash=cfg['dash']),
        name=cfg['name'],
        legendgroup='edges',
        hoverinfo='skip',
        showlegend=True,
    ))
    # Hover-маркеры на серединах рёбер
    fig_graph.add_trace(go.Scatter(
        x=hover_x, y=hover_y, mode='markers',
        marker=dict(size=6, color='rgba(0,0,0,0)'),
        text=hover_txt,
        hovertemplate='%{text}<extra></extra>',
        showlegend=False,
    ))
 
# ── Ноды по кластерам ─────────────────────────────────────────
OUTCOME_BORDER = {1.0:  ('rgba(0,200,0,1.0)',  3.5),
                  0.0:  ('rgba(220,0,0,1.0)',   3.5),
                  None: ('rgba(255,255,255,0.7)', 1.2)}
CLUSTER_SYMBOL = {0: 'circle', 1: 'diamond', 2: 'square'}
 
for c in [0, 1, 2]:
    nodes_c = [n for n, d in G.nodes(data=True) if d.get('cluster') == c]
    if not nodes_c:
        continue
 
    chex  = CLUSTER_HEX[c]
    cname = CLUSTER_NAMES[c]
 
    nx_vals = [pos[n][0] for n in nodes_c]
    ny_vals = [pos[n][1] for n in nodes_c]
 
    # Размер: degree centrality → больше соединений = больший маркер
    sizes  = [12 + G.nodes[n]['degree'] * 4 for n in nodes_c]
    border_colors = []
    border_widths = []
    for n in nodes_c:
        out = G.nodes[n].get('outcome')
        bc, bw = OUTCOME_BORDER.get(out, OUTCOME_BORDER[None])
        border_colors.append(bc)
        border_widths.append(bw)
 
    # Hover с полным профилем
    hover_texts = []
    for n in nodes_c:
        nd = G.nodes[n]
        lines = [
            f'<b>{n}</b>',
            f'Кластер: {cname}',
            f'Age: {nd["age"]:.0f}  AMH: {nd["amh"]:.1f}',
            f'MC: {nd["p_mc"]*100:.1f}%  KAT: {nd["p_kat"]*100:.1f}%',
            f'Связей: {nd["degree"]}',
            f'Центральность: {nd["degree_centrality"]:.3f}',
        ]
        out = nd.get('outcome')
        if out == 1.0:   lines.append('Исход: Беременность ✅')
        elif out == 0.0: lines.append('Исход: Нет ❌')
        else:            lines.append('Исход: нет данных')
        hover_texts.append('<br>'.join(lines))
 
    fig_graph.add_trace(go.Scatter(
        x=nx_vals, y=ny_vals,
        mode='markers+text',
        name=cname,
        text=[n[:12] for n in nodes_c],
        textposition='top center',
        textfont=dict(size=7, color='rgba(200,200,200,0.75)'),
        hovertext=hover_texts,
        hovertemplate='%{hovertext}<extra></extra>',
        marker=dict(
            symbol=CLUSTER_SYMBOL[c],
            size=sizes,
            color=hex_rgba(chex, 0.62),
            line=dict(color=border_colors, width=border_widths),
        ),
        legendgroup='nodes',
    ))
 
# Выделяем изолированных (серым ромбом)
if isolated:
    fig_graph.add_trace(go.Scatter(
        x=[pos[n][0] for n in isolated],
        y=[pos[n][1] for n in isolated],
        mode='markers',
        name='Изолированные',
        marker=dict(symbol='x', size=14, color='rgba(150,150,150,0.7)',
                    line=dict(color='rgba(100,100,100,0.9)', width=2)),
        text=isolated,
        hovertemplate='<b>%{text}</b><br>Нетипичный профиль<extra></extra>',
    ))
 
fig_graph.update_layout(
    **{k: v for k, v in LAYOUT.items() if k != 'margin'},
    height=650,
    title=dict(
        text=(f'Граф сходства пациентов  ·  '
              f'k={K_NEIGHBORS}  порог={SIM_THRESHOLD:.0%}  '
              f'признаков={len(feat_cols)}'),
        font=dict(size=14),
    ),
    showlegend=True,
    legend=dict(
        x=1.01, y=1,
        bgcolor='rgba(0,0,0,0)',
        bordercolor='rgba(150,150,150,0.35)', borderwidth=1,
        font=dict(size=10),
    ),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    margin=dict(t=70, b=30, l=20, r=160),
)
fig_graph.show()
 
display(HTML(
    '<div style="font-family:Inter,sans-serif;font-size:11px;'
    'padding:9px 14px;border-radius:6px;'
    'border-left:3px solid #888;line-height:2;display:inline-block">'
    '<b>Форма</b>: ● C0 Standard · ◆ C1 Poor · ■ C2 High &nbsp;|&nbsp; '
    '<b>Размер</b>: число связей (degree) &nbsp;|&nbsp; '
    '<b>Обводка</b>: '
    '<span style="color:#00C800">зел.</span>=беременность · '
    '<span style="color:#E60000">красн.</span>=нет · белая=нет данных<br>'
    '<b>Рёбра</b>: толщина = сила сходства · '
    '× = изолированный (нетипичный профиль)'
    '</div>'
))

Форма : ● C0 Standard · ◆ C1 Poor · ■ C2 High  |  Размер : число связей (degree)  |  Обводка : зел. =беременность · красн. =нет · белая=нет данных Рёбра : толщина = сила сходства · × = изолированный (нетипичный профиль)

In [28]:
# ══════════════════════════════════════════════════════════════
# БЛОК C — АНАЛИЗ ПОХОЖИХ СЛУЧАЕВ (SIMILAR CASES)
# ══════════════════════════════════════════════════════════════
 
def find_similar_cases(patient_name: str,
                       G: nx.Graph,
                       df_ref: pd.DataFrame,
                       name_col_: str,
                       n_top: int = 5,
                       with_outcome_only: bool = False) -> pd.DataFrame:
    """
    Возвращает топ-N клинически похожих пациентов
    с их ключевыми параметрами и исходами.
    """
    # Поиск ноды по частичному совпадению имени
    candidates = [n for n in G.nodes() if patient_name.upper() in n.upper()]
    if not candidates:
        return pd.DataFrame(columns=['Найдено'] + ['Причина'])
    node = candidates[0]
 
    nbrs = [(nbr, G[node][nbr]['weight'])
            for nbr in G.neighbors(node)]
    nbrs.sort(key=lambda x: x[1], reverse=True)
 
    if with_outcome_only:
        nbrs = [(n, w) for n, w in nbrs
                if G.nodes[n].get('outcome') is not None]
 
    nbrs = nbrs[:n_top]
 
    rows = []
    for nbr_name, sim in nbrs:
        nd = G.nodes[nbr_name]
        # Находим строку в df
        mask = df_ref[name_col_].str.upper().str.strip() == nbr_name.upper()
        df_row = df_ref[mask]
        if df_row.empty:
            continue
        r = df_row.iloc[0]
 
        outcome = nd.get('outcome')
        rows.append({
            'Пациент':       nbr_name[:25],
            'Сходство':      f'{sim*100:.1f}%',
            'Возраст':       int(r.get('Age', 0)),
            'AMH':           round(r.get('amh', 0), 1),
            'AFC':           int(r.get('afc', 0)),
            'OCC':           int(r.get('OCC', 0)) if pd.notna(r.get('OCC')) else '—',
            'Бластоцисты':   int(r.get('Bl', 0))  if pd.notna(r.get('Bl'))  else '—',
            'MC%':           f'{r.get("p_per_transfer",0)*100:.1f}%',
            'KAT%':          f'{r.get("p_kat_raw",0)*100:.1f}%',
            'PRAI%':         f'{r.get("PRAI",0)*100:.1f}%' if pd.notna(r.get('PRAI')) else '—',
            'Кластер':       CLUSTER_NAMES.get(int(nd.get('cluster',-1)), '—'),
            'Исход':         ('✅' if outcome == 1.0 else
                              '❌' if outcome == 0.0 else '—'),
        })
    return pd.DataFrame(rows)
 
 
# ── Разбор: пациенты с наибольшей неопределённостью ──────────
pred_list_g = [c for c in
    ['p_per_transfer','p_kat_raw','p_nvsa','p_csdi','DIGITAL TWIN','PRAI']
    if c in df_u.columns]
 
if 'pred_std' not in df_u.columns:
    df_u['pred_std'] = df_u[pred_list_g].std(axis=1)
 
# Топ-5 пациентов для разбора похожих случаев
# (высокая неопределённость или известный ложноположительный исход)
priority_patients = df_u.nlargest(5, 'pred_std')[name_col].tolist()
 
# Добавляем ложноположительных если есть
fp_mask = (
    df_u['Preg'].notna() &
    (df_u['Preg'] == 0) &
    (df_u['p_per_transfer'] > 0.60)
)
fp_patients = df_u[fp_mask][name_col].tolist()
for p in fp_patients:
    if p not in priority_patients:
        priority_patients.insert(0, p)
priority_patients = priority_patients[:6]
 
display(HTML(
    '<hr style="opacity:0.3">'
    '<h3 style="font-family:Inter,sans-serif;font-size:13px;margin-bottom:4px">'
    '🔍 Similar Cases — клинически похожие пациенты</h3>'
    '<div style="font-family:Inter,sans-serif;font-size:11px;'
    'color:rgba(180,180,180,0.85)">'
    'Приоритет: ложноположительные прогнозы + высокая неопределённость моделей'
    '</div>'
))
 
for pname in priority_patients:
    sim_df = find_similar_cases(
        patient_name=pname,
        G=G,
        df_ref=df_u,
        name_col_=name_col,
        n_top=5,
    )
    if sim_df.empty:
        continue
 
    # Определяем контекст пациента
    mask = df_u[name_col].str.upper().str.strip() == pname.upper()
    if not mask.any():
        continue
    ref = df_u[mask].iloc[0]
    preg = ref.get('Preg')
    mc   = ref.get('p_per_transfer', 0) * 100
    std  = ref.get('pred_std', 0) * 100
 
    preg_badge = ('✅' if preg == 1.0 else '❌' if preg == 0.0 else '— нет данных')
    header_color = ('#c62828' if (preg == 0.0 and mc > 60)
                    else '#1976D2' if preg is None or np.isnan(float(preg) if preg is not None else float('nan'))
                    else '#2E7D32')
 
    display(HTML(
        f'<div style="font-family:Inter,sans-serif;'
        f'padding:8px 12px;margin-top:14px;border-radius:5px;'
        f'border-left:3px solid {header_color}">'
        f'<b>{pname[:30]}</b> &nbsp; '
        f'MC={mc:.1f}% · SD={std:.1f}пп · Исход: {preg_badge}'
        f'</div>'
    ))
 
    # Соседи с известными исходами отдельно
    sim_with_out = sim_df[sim_df['Исход'] != '—']
    if not sim_with_out.empty:
        n_preg = (sim_with_out['Исход'] == '✅').sum()
        n_no   = (sim_with_out['Исход'] == '❌').sum()
        display(HTML(
            f'<div style="font-family:Inter,sans-serif;font-size:11px;'
            f'padding:4px 12px;color:rgba(180,180,180,0.85)">'
            f'Среди {len(sim_with_out)} похожих с известным исходом: '
            f'<span style="color:#00C800"><b>{n_preg} беременность</b></span> · '
            f'<span style="color:#E60000"><b>{n_no} нет</b></span>'
            f'</div>'
        ))
 
    display(sim_df.style
            .applymap(lambda v: 'color:#00C800;font-weight:bold' if v == '✅' else
                                'color:#E60000;font-weight:bold' if v == '❌' else '',
                      subset=['Исход'])
            .set_properties(**{'font-size': '11px', 'font-family': 'Inter,sans-serif'})
            .hide(axis='index')
    )
 
 

🔍 Similar Cases — клинически похожие пациенты Приоритет: ложноположительные прогнозы + высокая неопределённость моделей

Ergasheva Sarvinoz   MC=75.5% · SD=14.5пп · Исход: ❌

Среди 1 похожих с известным исходом: 1 беременность · 0 нет

Пациент,Сходство,Возраст,AMH,AFC,OCC,Бластоцисты,MC%,KAT%,PRAI%,Кластер,Исход
Joraeva Shaxzodaxon,96.2%,21,2.500000,11,10,2,77.1%,55.1%,51.9%,C1 Poor,—
Oozoqboyeva Zuxraxon,92.7%,24,2.500000,10,9,0,75.3%,60.7%,28.0%,C0 Standard,—
Ganiyeva Nodirabegim,92.4%,28,2.500000,13,13,2,70.5%,55.7%,57.0%,C0 Standard,✅
KADIROVA AMIRA,89.2%,26,2.500000,22,16,1,72.2%,53.3%,21.2%,C0 Standard,—
KARIMOVA AZIMA,89.0%,27,2.500000,18,14,3,73.2%,58.8%,41.5%,C0 Standard,—


Davronova Bakhora   MC=68.1% · SD=11.7пп · Исход: ❌

...

Среди 2 похожих с известным исходом: 1 беременность · 1 нет

Пациент,Сходство,Возраст,AMH,AFC,OCC,Бластоцисты,MC%,KAT%,PRAI%,Кластер,Исход
Ergasheva Sarvinoz,89.2%,24,2.500000,15,14,2,75.5%,54.2%,31.4%,C0 Standard,❌
Ganiyeva Nodirabegim,82.1%,28,2.500000,13,13,2,70.5%,55.7%,57.0%,C0 Standard,✅
Oozoqboyeva Zuxraxon,79.7%,24,2.500000,10,9,0,75.3%,60.7%,28.0%,C0 Standard,—
KARIMOVA AZIMA,79.6%,27,2.500000,18,14,3,73.2%,58.8%,41.5%,C0 Standard,—
SULTANOVA GULNORA,79.5%,29,2.500000,15,11,—,70.7%,55.0%,55.0%,C0 Standard,—


Mamataliyeva Shaxnoza   MC=65.0% · SD=18.5пп · Исход: ✅

Среди 1 похожих с известным исходом: 1 беременность · 0 нет

Пациент,Сходство,Возраст,AMH,AFC,OCC,Бластоцисты,MC%,KAT%,PRAI%,Кластер,Исход
Kodirova Dilfusa,74.6%,34,2.500000,15,15,2,65.3%,57.6%,47.0%,C0 Standard,✅


In [29]:
# ══════════════════════════════════════════════════════════════
# БЛОК D — СТРУКТУРНЫЙ ОТЧЁТ ГРАФА
# ══════════════════════════════════════════════════════════════
 
# ── Распределение степеней ───────────────────────────────────
degrees = [d for _, d in G.degree()]
 
fig_graph_stats = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        'Распределение связей (degree)',
        'Betweenness centrality',
        'Сходство внутри/между кластерами',
    ],
    horizontal_spacing=0.12,
)
 
# Histogram degrees
fig_graph_stats.add_trace(go.Histogram(
    x=degrees, nbinsx=10,
    marker=dict(color=hex_rgba(C['blue'], 0.55),
                line=dict(color='white', width=0.8)),
    hovertemplate='Degree=%{x}: %{y} пациентов<extra></extra>',
    name='Degree',
    showlegend=False,
), row=1, col=1)
fig_graph_stats.add_vline(
    x=np.mean(degrees), line_dash='dot',
    line_color='rgba(200,100,0,0.7)',
    annotation_text=f'μ={np.mean(degrees):.1f}',
    annotation_position='top right',
    annotation_font_size=9,
    col=1,
)
 
# Betweenness — топ-10 «мостовых» пациентов
top_btw = sorted(between_cent.items(), key=lambda x: x[1], reverse=True)[:10]
btw_names  = [n[:16] for n, _ in top_btw][::-1]
btw_values = [v for _, v in top_btw][::-1]
btw_colors = [hex_rgba(CLUSTER_HEX.get(G.nodes[n].get('cluster',-1),'#888'), 0.55)
              for n, _ in reversed(top_btw)]
 
fig_graph_stats.add_trace(go.Bar(
    y=btw_names, x=btw_values,
    orientation='h',
    marker=dict(color=btw_colors,
                line=dict(color='rgba(255,255,255,0.5)', width=0.5)),
    hovertemplate='<b>%{y}</b><br>Betweenness: %{x:.4f}<extra></extra>',
    name='Betweenness',
    showlegend=False,
), row=1, col=2)
 
# Внутри/между кластерами — средняя схожесть
cluster_sim_data = {'label': [], 'sim': [], 'type': []}
for u, v, data in G.edges(data=True):
    c_u = G.nodes[u].get('cluster', -1)
    c_v = G.nodes[v].get('cluster', -1)
    sim = data.get('weight', 0)
    etype = 'Внутри кластера' if c_u == c_v else 'Между кластерами'
    cluster_sim_data['sim'].append(sim)
    cluster_sim_data['type'].append(etype)
 
for etype, color in [('Внутри кластера', C['green']),
                     ('Между кластерами', C['orange'])]:
    vals = [s for s, t in zip(cluster_sim_data['sim'],
                              cluster_sim_data['type']) if t == etype]
    if not vals:
        continue
    fig_graph_stats.add_trace(go.Box(
        y=vals, name=etype,
        boxmean=True,
        fillcolor=hex_rgba(color, 0.18),
        line=dict(color=hex_rgba(color, 0.60), width=1.8),
        marker=dict(size=5, opacity=0.5,
                    color=hex_rgba(color, 0.55)),
        hovertemplate=f'{etype}: %{{y:.3f}}<extra></extra>',
    ), row=1, col=3)
 
fig_graph_stats.update_xaxes(title_text='Степень', row=1, col=1, zeroline=False)
fig_graph_stats.update_yaxes(title_text='N пациентов', row=1, col=1, zeroline=False)
fig_graph_stats.update_xaxes(zeroline=False, row=1, col=2)
fig_graph_stats.update_yaxes(title_text='Cosine similarity', range=[0.65, 1.02],
                              zeroline=False, row=1, col=3)
 
fig_graph_stats.update_layout(
    **{k: v for k, v in LAYOUT.items() if k != 'margin'},
    height=380,
    title=dict(text='Структурный анализ графа', font=dict(size=14)),
    showlegend=True,
    legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.12,
                bgcolor='rgba(0,0,0,0)', font=dict(size=10)),
    margin=dict(t=60, b=80, l=120, r=30),
)
fig_graph_stats.show()
 
# ── Финальный текстовый отчёт ─────────────────────────────────
report_lines = [
    f'<b>Граф:</b> {G.number_of_nodes()} пациентов · '
    f'{G.number_of_edges()} рёбер · '
    f'плотность {nx.density(G):.3f}',
    f'<b>Изолированных (нетипичных):</b> {len(isolated)}'
    + (f' → {", ".join(s[:15] for s in isolated)}' if isolated else ' — все пациенты имеют соседей'),
    f'<b>Компонент связности:</b> {len(components)} '
    + ('(граф связен)' if len(components) == 1 else f'(фрагментирован)'),
]
 
# Топ-3 центральных пациента
top_central = sorted(degree_cent.items(), key=lambda x: x[1], reverse=True)[:3]
top_str = ' · '.join(f'{n[:15]} ({v:.2f})' for n, v in top_central)
report_lines.append(f'<b>Самые "типичные" пациенты (degree centrality):</b> {top_str}')
 
# Мост-пациенты (высокий betweenness)
top_bridge = sorted(between_cent.items(), key=lambda x: x[1], reverse=True)[:2]
bridge_str = ' · '.join(f'{n[:15]} ({v:.3f})' for n, v in top_bridge)
report_lines.append(f'<b>Мостовые пациенты (betweenness):</b> {bridge_str}')
 
# Среднее сходство внутри vs между
intra = [s for s,t in zip(cluster_sim_data['sim'], cluster_sim_data['type'])
         if t == 'Внутри кластера']
inter = [s for s,t in zip(cluster_sim_data['sim'], cluster_sim_data['type'])
         if t == 'Между кластерами']
if intra and inter:
    report_lines.append(
        f'<b>Сходство внутри кластера:</b> {np.mean(intra):.3f} vs '
        f'между кластерами: {np.mean(inter):.3f} '
        + ('→ кластеры хорошо разделены' if np.mean(intra) > np.mean(inter) + 0.03
           else '→ кластеры перекрываются')
    )
 
display(HTML(
    '<div style="font-family:Inter,sans-serif;font-size:11px;'
    'padding:10px 14px;border-radius:6px;'
    'border-left:3px solid #888;line-height:2.2;display:inline-block">'
    + '<br>'.join(report_lines) +
    '</div>'
))

Граф: 50 пациентов · 128 рёбер · плотность 0.104 Изолированных (нетипичных): 1 → KARIMJANOVA SHO Компонент связности: 3 (фрагментирован) Самые "типичные" пациенты (degree centrality): NURULOVA ASILYA (0.18) · Nazaraliyeva Ra (0.16) · Rambergenova El (0.16) Мостовые пациенты (betweenness): ODILOVA MUXLISA (0.429) · KADIROVA AMIRA (0.413) Сходство внутри кластера: 0.882 vs между кластерами: 0.812 → кластеры хорошо разделены

In [30]:
# ══════════════════════════════════════════════════════════════
# СТАТИСТИЧЕСКИЙ ТЕСТ: Intra- vs Inter-cluster Similarity
# Mann-Whitney U + Effect Size (Cohen's d, rank-biserial r)
# + Bootstrap 95% CI для медиан
# ══════════════════════════════════════════════════════════════

from scipy.stats import mannwhitneyu, shapiro, kstest, norm
import scipy.stats as stats
from IPython.display import display, HTML

# ── Сбор данных рёбер по типу (используем граф G из предыдущей ячейки) ───────
intra_sims = []   # рёбра внутри кластера
inter_sims = []   # рёбра между кластерами
edge_records = []

for u, v, data in G.edges(data=True):
    c_u = G.nodes[u].get('cluster', -1)
    c_v = G.nodes[v].get('cluster', -1)
    sim = data.get('weight', 0)
    etype = 'intra' if c_u == c_v else 'inter'
    if etype == 'intra':
        intra_sims.append(sim)
    else:
        inter_sims.append(sim)
    edge_records.append({
        'patient_u': u[:20], 'patient_v': v[:20],
        'cluster_u': CLUSTER_NAMES.get(c_u, str(c_u)),
        'cluster_v': CLUSTER_NAMES.get(c_v, str(c_v)),
        'similarity': sim,
        'edge_type': etype,
    })

intra_arr = np.array(intra_sims)
inter_arr = np.array(inter_sims)

print(f'Рёбер intra-cluster: {len(intra_arr)}')
print(f'Рёбер inter-cluster: {len(inter_arr)}')
print()

# ── 1. Нормальность — Shapiro-Wilk ────────────────────────────────────────────
# (если n < 50 используем Shapiro, иначе KS)
def normality_test(arr, label):
    if len(arr) < 50:
        stat, p = shapiro(arr)
        test_name = "Shapiro-Wilk"
    else:
        stat, p = kstest(arr, 'norm', args=(arr.mean(), arr.std()))
        test_name = "Kolmogorov-Smirnov"
    normal = p > 0.05
    print(f'  {label}: {test_name} W={stat:.4f}, p={p:.4f} '
          f'→ {"нормальное ✅" if normal else "не нормальное → Mann-Whitney ✅"}')
    return normal

print('── Тест нормальности распределений ──')
intra_normal = normality_test(intra_arr, 'Intra-cluster')
inter_normal = normality_test(inter_arr, 'Inter-cluster')
print()

# ── 2. Mann-Whitney U (непараметрический, не требует нормальности) ────────────
U_stat, p_value = mannwhitneyu(intra_arr, inter_arr, alternative='greater')
# alternative='greater': H1: intra > inter (односторонний, направленная гипотеза)

n1, n2 = len(intra_arr), len(inter_arr)

# ── 3. Effect size: rank-biserial correlation r ───────────────────────────────
# r = 1 - (2*U)/(n1*n2) — стандартная мера для Mann-Whitney
r_rb = 1 - (2 * U_stat) / (n1 * n2)

# Cohen's d (параметрический, для справки)
pooled_sd = np.sqrt(((n1-1)*intra_arr.std()**2 + (n2-1)*inter_arr.std()**2) / (n1+n2-2))
cohens_d  = (intra_arr.mean() - inter_arr.mean()) / pooled_sd if pooled_sd > 0 else 0

# Интерпретация effect size
def interpret_r(r):
    ar = abs(r)
    if ar >= 0.50: return 'Large'
    if ar >= 0.30: return 'Medium'
    if ar >= 0.10: return 'Small'
    return 'Negligible'

def interpret_d(d):
    ad = abs(d)
    if ad >= 0.80: return 'Large'
    if ad >= 0.50: return 'Medium'
    if ad >= 0.20: return 'Small'
    return 'Negligible'

# ── 4. Bootstrap 95% CI для медиан ───────────────────────────────────────────
np.random.seed(42)
N_BOOT = 10_000

def bootstrap_median_ci(arr, n_boot=N_BOOT, alpha=0.05):
    boot_medians = np.array([
        np.median(np.random.choice(arr, size=len(arr), replace=True))
        for _ in range(n_boot)
    ])
    ci_lo = np.percentile(boot_medians, alpha/2 * 100)
    ci_hi = np.percentile(boot_medians, (1 - alpha/2) * 100)
    return ci_lo, ci_hi

intra_ci_lo, intra_ci_hi = bootstrap_median_ci(intra_arr)
inter_ci_lo, inter_ci_hi = bootstrap_median_ci(inter_arr)

# ── 5. Вывод результатов ──────────────────────────────────────────────────────
print('══ РЕЗУЛЬТАТЫ ТЕСТА ══════════════════════════════════════')
print(f'H₀: Intra-cluster similarity = Inter-cluster similarity')
print(f'H₁: Intra-cluster similarity > Inter-cluster similarity')
print()
print(f'Mann-Whitney U = {U_stat:.1f}')
print(f'p-value (one-tailed) = {p_value:.6f}',
      '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns')
print()
print(f'── Описательные статистики ──')
print(f'  Intra-cluster: median = {np.median(intra_arr):.4f}  '
      f'[{intra_ci_lo:.4f}, {intra_ci_hi:.4f}] 95% CI  '
      f'mean = {intra_arr.mean():.4f}  SD = {intra_arr.std():.4f}  n = {n1}')
print(f'  Inter-cluster: median = {np.median(inter_arr):.4f}  '
      f'[{inter_ci_lo:.4f}, {inter_ci_hi:.4f}] 95% CI  '
      f'mean = {inter_arr.mean():.4f}  SD = {inter_arr.std():.4f}  n = {n2}')
print()
print(f'── Effect size ──')
print(f'  Rank-biserial r = {r_rb:.4f}  ({interpret_r(r_rb)})')
print(f'  Cohen\'s d       = {cohens_d:.4f}  ({interpret_d(cohens_d)})')
print()
if p_value < 0.05:
    print(f'✅ H₀ отвергается: Intra-cluster similarity значимо выше '
          f'inter-cluster (p={p_value:.4f})')
else:
    print(f'⚠️  H₀ не отвергается: различие не достигает уровня значимости')

# ── 6. Визуализация ───────────────────────────────────────────────────────────
fig_mw = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        'Распределения сходства',
        'Медианы с 95% Bootstrap CI',
        'Матрица сходства по кластерам',
    ],
    horizontal_spacing=0.11,
)

# ── (1,1) Overlapping histograms ──────────────────────────────
for arr, label, color in [
    (intra_arr, f'Intra-cluster (n={n1})', C['blue']),
    (inter_arr, f'Inter-cluster (n={n2})', C['red']),
]:
    fig_mw.add_trace(go.Histogram(
        x=arr, nbinsx=14,
        name=label,
        marker=dict(color=hex_rgba(color, 0.52),
                    line=dict(color=hex_rgba(color, 0.80), width=0.8)),
        opacity=0.80,
        hovertemplate=f'{label}: %{{x:.3f}}<extra></extra>',
    ), row=1, col=1)

# Медианы как vline
for val, color, lbl in [
    (np.median(intra_arr), C['blue'], f'Intra med={np.median(intra_arr):.3f}'),
    (np.median(inter_arr), C['red'],  f'Inter med={np.median(inter_arr):.3f}'),
]:
    fig_mw.add_vline(
        x=val, line_dash='dash', line_color=color, line_width=2,
        annotation_text=lbl, annotation_position='top',
        annotation_font_size=9,
        col=1,
    )

# ── (1,2) Median + Bootstrap CI ──────────────────────────────
for xi, (arr, label, color, ci_lo, ci_hi) in enumerate([
    (intra_arr, 'Intra-cluster', C['blue'], intra_ci_lo, intra_ci_hi),
    (inter_arr, 'Inter-cluster', C['red'],  inter_ci_lo, inter_ci_hi),
]):
    med = np.median(arr)
    # Violin
    fig_mw.add_trace(go.Violin(
        x=[label] * len(arr),
        y=arr,
        name=label,
        showlegend=False,
        fillcolor=hex_rgba(color, 0.18),
        line_color=hex_rgba(color, 0.55),
        box_visible=True,
        meanline_visible=True,
        width=0.5,
        hoverinfo='skip',
    ), row=1, col=2)
    # Bootstrap CI как отдельная точка со шкалой ошибок
    fig_mw.add_trace(go.Scatter(
        x=[label], y=[med],
        mode='markers',
        showlegend=False,
        error_y=dict(
            type='data',
            array=[ci_hi - med],
            arrayminus=[med - ci_lo],
            color=hex_rgba(color, 0.80),
            thickness=2.5, width=10,
        ),
        marker=dict(size=12, color=hex_rgba(color, 0.85),
                    line=dict(color='white', width=1.5)),
        hovertemplate=(
            f'<b>{label}</b><br>'
            f'Median: {med:.4f}<br>'
            f'95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]<extra></extra>'
        ),
    ), row=1, col=2)

# Аннотация p-value
sig_label = ('***' if p_value < 0.001 else '**' if p_value < 0.01
             else '*'   if p_value < 0.05 else 'ns')
fig_mw.add_annotation(
    x=0.5, y=1.03,
    xref='x2 domain', yref='y2 domain',
    text=f'Mann-Whitney U={U_stat:.0f}, p={p_value:.4f} {sig_label}<br>'
         f'r={r_rb:.3f} ({interpret_r(r_rb)})',
    showarrow=False,
    font=dict(size=10),
    align='center',
    bgcolor='rgba(255,255,255,0.0)',
)

fig_mw.update_yaxes(
    title_text='Cosine similarity',
    range=[0.65, 1.02], zeroline=False, row=1, col=2,
)

# ── (1,3) Heatmap средних сходств между кластерами ───────────
cluster_ids = [0, 1, 2]
heat_z = np.zeros((3, 3))
heat_n = np.zeros((3, 3), dtype=int)

for u, v, data in G.edges(data=True):
    c_u = int(G.nodes[u].get('cluster', -1))
    c_v = int(G.nodes[v].get('cluster', -1))
    if c_u in [0,1,2] and c_v in [0,1,2]:
        heat_z[c_u][c_v] += data.get('weight', 0)
        heat_z[c_v][c_u] += data.get('weight', 0)
        heat_n[c_u][c_v] += 1
        if c_u != c_v:
            heat_n[c_v][c_u] += 1

# Среднее (защита от деления на 0)
heat_mean = np.where(heat_n > 0, heat_z / np.maximum(heat_n, 1), 0)
# Диагональ — intra
for ci in range(3):
    intra_c = [G[u][v]['weight']
               for u, v, _ in G.edges(data=True)
               if G.nodes[u].get('cluster') == ci
               and G.nodes[v].get('cluster') == ci]
    heat_mean[ci][ci] = np.mean(intra_c) if intra_c else 0

cluster_labels = ['C0 Standard', 'C1 Poor', 'C2 High']
text_annot = [[f'{heat_mean[i][j]:.3f}\nn={heat_n[i][j]}'
               for j in range(3)] for i in range(3)]

fig_mw.add_trace(go.Heatmap(
    z=heat_mean,
    x=cluster_labels,
    y=cluster_labels,
    colorscale='Blues',
    zmin=0.75, zmax=0.95,
    text=text_annot,
    texttemplate='%{text}',
    textfont=dict(size=10),
    colorbar=dict(title='Mean sim', len=0.55, x=1.02,
                  thickness=12, tickfont=dict(size=9)),
    hovertemplate='%{y} ↔ %{x}<br>Mean sim: %{z:.3f}<extra></extra>',
), row=1, col=3)

# ── Layout ────────────────────────────────────────────────────
fig_mw.update_layout(
    **{k: v for k, v in LAYOUT.items() if k != 'margin'},
    height=450,
    barmode='overlay',
    title=dict(
        text=(f'Mann-Whitney U test: Intra- vs Inter-cluster Cosine Similarity  '
              f'|  p={p_value:.4f} {sig_label}  |  r={r_rb:.3f} ({interpret_r(r_rb)})'),
        font=dict(size=13),
    ),
    legend=dict(
        orientation='h', x=0.5, xanchor='center',
        y=-0.12, yanchor='top',
        bgcolor='rgba(0,0,0,0)',
        bordercolor='rgba(150,150,150,0.35)', borderwidth=1,
        font=dict(size=10),
    ),
    margin=dict(t=70, b=80, l=60, r=80),
)
fig_mw.update_xaxes(zeroline=False, row=1, col=1)
fig_mw.update_yaxes(title_text='N рёбер', zeroline=False, row=1, col=1)
fig_mw.show()

# ── 7. Форматированный блок для статьи ───────────────────────
display(HTML(
    '<hr style="opacity:0.3">'
    '<div style="font-family:Georgia,serif;font-size:12px;'
    'padding:14px 18px;border-radius:6px;'
    'border-left:3px solid #888;line-height:2.0">'
    '<b>Statistical comparison of intra- vs inter-cluster edge similarity.</b> '
    'To assess whether the Digital Twin cluster assignments corresponded to '
    'statistically distinct patient subpopulations, cosine similarity values were '
    f'compared between intra-cluster graph edges (n={n1}) and inter-cluster edges '
    f'(n={n2}) using a one-tailed Mann-Whitney U test (H₁: intra &gt; inter). '
    f'Intra-cluster edges demonstrated significantly higher similarity '
    f'(median = {np.median(intra_arr):.3f}, '
    f'95% bootstrap CI [{intra_ci_lo:.3f}, {intra_ci_hi:.3f}]) '
    f'compared to inter-cluster edges '
    f'(median = {np.median(inter_arr):.3f}, '
    f'95% bootstrap CI [{inter_ci_lo:.3f}, {inter_ci_hi:.3f}]; '
    f'U = {U_stat:.1f}, p = {p_value:.4f}). '
    f'The rank-biserial correlation indicated a {interpret_r(r_rb).lower()} effect size '
    f'(r = {r_rb:.3f}), and Cohen\'s d = {cohens_d:.3f} ({interpret_d(cohens_d).lower()}). '
    'These findings confirm that patients sharing a DT cluster assignment are '
    'substantially more similar to one another across independent clinical and '
    'predictive features than to patients in alternative clusters, providing '
    'convergent validation of the Digital Twin cluster structure.</div>'
))

Рёбер intra-cluster: 97
Рёбер inter-cluster: 31

── Тест нормальности распределений ──
  Intra-cluster: Kolmogorov-Smirnov W=0.1002, p=0.2665 → нормальное ✅
  Inter-cluster: Shapiro-Wilk W=0.9683, p=0.4723 → нормальное ✅

══ РЕЗУЛЬТАТЫ ТЕСТА ══════════════════════════════════════
H₀: Intra-cluster similarity = Inter-cluster similarity
H₁: Intra-cluster similarity > Inter-cluster similarity

Mann-Whitney U = 2259.5
p-value (one-tailed) = 0.000013 ***

── Описательные статистики ──
  Intra-cluster: median = 0.8966  [0.8803, 0.9214] 95% CI  mean = 0.8824  SD = 0.0787  n = 97
  Inter-cluster: median = 0.8070  [0.7723, 0.8438] 95% CI  mean = 0.8121  SD = 0.0663  n = 31

── Effect size ──
  Rank-biserial r = -0.5028  (Large)
  Cohen's d       = 0.9253  (Large)

✅ H₀ отвергается: Intra-cluster similarity значимо выше inter-cluster (p=0.0000)


Statistical comparison of intra- vs inter-cluster edge similarity. To assess whether the Digital Twin cluster assignments corresponded to statistically distinct patient subpopulations, cosine similarity values were compared between intra-cluster graph edges (n=97) and inter-cluster edges (n=31) using a one-tailed Mann-Whitney U test (H₁: intra > inter). Intra-cluster edges demonstrated significantly higher similarity (median = 0.897, 95% bootstrap CI [0.880, 0.921]) compared to inter-cluster edges (median = 0.807, 95% bootstrap CI [0.772, 0.844]; U = 2259.5, p = 0.0000). The rank-biserial correlation indicated a large effect size (r = -0.503), and Cohen's d = 0.925 (large). These findings confirm that patients sharing a DT cluster assignment are substantially more similar to one another across independent clinical and predictive features than to patients in alternative clusters, providing convergent validation of the Digital Twin cluster structure.